In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans

## ORDEN DE EJECUCIÓN OBLIGATORIO TRAS REINICIAR KERNEL
1. Celda de imports y PALETA
2. Celda de carga de datos (df_unificado_clustering)
3. Celda de construcción de df_rfm
4. Celda 5 — log1p + escalado
5. Celda DBSCAN
6. Celda KMeans final
7. Celda merge df_plot
- A partir de aquí todas las figuras funcionan

## OBJETIVO 1 y 2

##### Importación de los datos originales de los datos sobre préstamos físicos

In [ ]:
prestamos_original2022 = pd.read_excel('datos_originales/datos_revistas_consultas/2022.xlsx',header=3)
prestamos_original2023 = pd.read_excel('datos_originales/datos_revistas_consultas/2023.xlsx',header=3)
prestamos_original2024 = pd.read_excel('datos_originales/datos_revistas_consultas/2024.xlsx',header=3)
prestamos_original2025 = pd.read_excel('datos_originales/datos_revistas_consultas/2025.xlsx',header=3)

##### Importación de los datos originales de los datos sobre consulta en bases de datos

In [ ]:
consultas_original2022 = pd.read_excel('datos_originales/Estadísticas_basesdedatos/Estadisticas 2022.xlsx',header=3)
consultas_original2023 = pd.read_excel('datos_originales/Estadísticas_basesdedatos/Estadisticas 2023.xlsx',header=3)
consultas_original2024 = pd.read_excel('datos_originales/Estadísticas_basesdedatos/Estadisticas 2024.xlsx',header=3)
consultas_original2025 = pd.read_excel('datos_originales/Estadísticas_basesdedatos/Estadisticas 2025.xlsx',header=5)

In [ ]:
display(consultas_original2025.head(200))

In [ ]:
consultas_original2025.shape

In [ ]:
prestamos_original2022.shape

#### Limpieza de los datos de consultas virtuales.

In [ ]:
print("Datos 2022")
display(consultas_original2022.isnull().sum())
print(consultas_original2022.shape)

print("")
print("Datos 2023")
display(consultas_original2023.isnull().sum())
print(consultas_original2023.shape)

print("")
print("Datos 2024")
display(consultas_original2024.isnull().sum())
print(consultas_original2024.shape)

print("")
print("Datos 2025")
display(consultas_original2025.isnull().sum())
print(consultas_original2025.shape)

1. Vamos a organizar el nombre de las variables, quitar posibles espacios, etc

In [ ]:
consultas_original2022.columns = consultas_original2022.columns.str.strip()
consultas_original2022.columns = consultas_original2022.columns.str.replace(' ', '_')

consultas_original2023.columns = consultas_original2023.columns.str.strip()
consultas_original2023.columns = consultas_original2023.columns.str.replace(' ', '_')

consultas_original2024.columns = consultas_original2024.columns.str.strip()
consultas_original2024.columns = consultas_original2024.columns.str.replace(' ', '_')

consultas_original2025.columns = consultas_original2025.columns.str.strip()
consultas_original2025.columns = consultas_original2025.columns.str.replace(' ', '_')

In [ ]:
print("Nuevos nombres de columnas:")
print(consultas_original2022.columns.tolist())
print(consultas_original2023.columns.tolist())
print(consultas_original2024.columns.tolist())
print(consultas_original2025.columns.tolist())

Se hace una tabla con el tamaño de cada una de las tablas por año para entender con cuánta información contamos por año y que si coincida al hacer la unión.

In [ ]:
# 1. Contamos las filas de cada DataFrame individual
conteo_2022 = len(consultas_original2022)
conteo_2023 = len(consultas_original2023)
conteo_2024 = len(consultas_original2024)
conteo_2025 = len(consultas_original2025)

# 2. Creamos la tabla de auditoría
consultas_cantidad = {
    'Periodo': ['2022', '2023', '2024', '2025', 'TOTAL DE RESGISTROS'],
    'Cantidad_Registros': [
        conteo_2022, 
        conteo_2023, 
        conteo_2024, 
        conteo_2025, 
        (conteo_2022 + conteo_2023 + conteo_2024 + conteo_2025),
    ]
}

consultas_cantidad = pd.DataFrame(consultas_cantidad)


print("--- TABLA DE CANTIDAD DE DATOS POR TABLA ---")
print(consultas_cantidad)


2. Se realiza una matriz de valores nulos por cada año
- Ver columnas que podrían eliminarse o dónde es necesario posiblemente imputar, y ver la estructura de las tablas que tan igual es

In [ ]:
datasets = {
    '2022': consultas_original2022,
    '2023': consultas_original2023,
    '2024': consultas_original2024,
    '2025': consultas_original2025
}

dict_resumen = {}

for anio, df in datasets.items():
    dict_resumen[anio] = (df.isnull().sum() / len(df)) * 100


df_calidad = pd.DataFrame(dict_resumen)
df_calidad = df_calidad.round(1)

df_calidad.style.background_gradient(cmap='Reds').format("{:.2f}")

Vamos eliminar las columnas 100% vacías.

In [ ]:
columnas_vacias = df_calidad[df_calidad.apply(lambda x: x.fillna(100).eq(100).all(), axis=1)].index.tolist()

print(f"Columnas que se eliminarán por estar totalmente vacías: {columnas_vacias}")

In [ ]:
# Lista de tablas
tablas = [consultas_original2022, consultas_original2023, consultas_original2024, consultas_original2025]

# Eliminación en un bucle
for i in range(len(tablas)):
    tablas[i].drop(columns=columnas_vacias, inplace=True, errors='ignore')

In [ ]:
# 1. Recalculamos los porcentajes de nulos con las tablas ya filtradas
datasets_limpios = {
    '2022': consultas_original2022,
    '2023': consultas_original2023,
    '2024': consultas_original2024,
    '2025': consultas_original2025
}

dict_verificacion = {}
for anio, df in datasets_limpios.items():
    dict_verificacion[anio] = (df.isnull().sum() / len(df)) * 100

# 2. Creamos el nuevo DataFrame de verificación
df_verificacion = pd.DataFrame(dict_verificacion)

# 3. Mostramos con el formato elegante de 2 decimales
df_verificacion.style.background_gradient(cmap='Reds').format("{:.2f}")

Hay una columna con nombre identificación que se encuentra en un solo año y que no tiene la variable "Código" y que justo tiene una varriable "Identificación" que no tiene nulos, esta podría ser la columna de códigos.

In [ ]:
# Ver contenido de la columna, se evidencia que corresponde a la cédula del usuario, es decir, corresponde a la variable "Código"
print(consultas_original2024['Identificación'].sample(10))

In [ ]:
consultas_original2024 = consultas_original2024.rename(columns={'Identificación': 'Código'})

In [ ]:
# Ver contenido de la columna, se evidencia que corresponde a la cédula del usuario, es decir, corresponde a la variable "Código"
print(consultas_original2025['Identificación'].sample(10))

In [ ]:
consultas_original2025 = consultas_original2025.rename(columns={'Identificación': 'Código'})

Ya deben quedar todas las variables con los mismos nombres

In [ ]:
print("Nuevos nombres de columnas:")
print(consultas_original2022.columns.tolist())
print(consultas_original2023.columns.tolist())
print(consultas_original2024.columns.tolist())
print(consultas_original2025.columns.tolist())

In [ ]:
# 1. Recalculamos los porcentajes de nulos con las tablas ya filtradas
datasets_limpios = {
    '2022': consultas_original2022,
    '2023': consultas_original2023,
    '2024': consultas_original2024,
    '2025': consultas_original2025
}

dict_verificacion = {}
for anio, df in datasets_limpios.items():
    dict_verificacion[anio] = (df.isnull().sum() / len(df)) * 100

# 2. Creamos el nuevo DataFrame de verificación
df_verificacion = pd.DataFrame(dict_verificacion)

# 3. Mostramos con el formato elegante de 2 decimales
df_verificacion.style.background_gradient(cmap='Reds').format("{:.2f}")

In [ ]:
display(consultas_original2022.sample(10))

Hacemos el cambio del nombre de la columna para mayor claridad a "Nivel_Académico"

In [ ]:
consultas_original2022 = consultas_original2022.rename(columns={'Programa': 'Nivel_academico'})
consultas_original2023 = consultas_original2023.rename(columns={'Programa': 'Nivel_academico'})
consultas_original2024 = consultas_original2024.rename(columns={'Programa': 'Nivel_academico'})
consultas_original2025 = consultas_original2025.rename(columns={'Programa': 'Nivel_academico'})

In [ ]:
display(consultas_original2022.sample(100))

In [ ]:
consultas_original2025['Nivel_academico'].unique

Realizamos la unión de las tablas por los 4 años antes de hacer la gestión de los datos nulos o faltantes, ya teniendo los nombres de las columnas iguales por las siguientes razones:

- Al unificar las tablas en un único DataFrame Maestro, se garantiza que las reglas de limpieza, normalización de nombres y manejo de valores nulos se apliquen de forma homogénea.
- La consolidación aumenta la muestra estadística, permitiendo una imputación histórica. Esto significa que los valores faltantes en periodos recientes (ej. Facultad o Programa en 2025) pueden ser deducidos o "completados" mediante la persistencia de datos de periodos anteriores (2022-2024), reduciendo así la pérdida de información por eliminación de filas.
- Evaluar la completitud sobre el volumen total permite identificar si los datos faltantes son errores aleatorios o fallos sistemáticos de un año específico, permitiendo una toma de decisiones más robusta para el modelo de Clustering.


In [ ]:
# Etiquetamos cada año antes de unir. Se usa este para evitar errores al momento de ejecutar el modelo.
# Con esto también aseguramos que al momento de la unión cada dato quede con el año que le corresponde realmente.
consultas_original2022['Anio'] = 2022
consultas_original2023['Anio'] = 2023
consultas_original2024['Anio'] = 2024
consultas_original2025['Anio'] = 2025

# Concatenación Vertical
# Usamos ignore_index=True para que el índice sea continuo del 0 al total
consultas_maestro = pd.concat([consultas_original2022, consultas_original2023, consultas_original2024, consultas_original2025], ignore_index=True)

# 4. Verificación de Consistencia
print(f"--- Unión Exitosa ---")
print(f"Registros totales: {consultas_maestro.shape[0]}")
print(f"Columnas finales: {consultas_maestro.shape[1]}")

# Verificamos que no haya creado "columnas fantasma" por diferencias de nombres
print("\nLista de columnas unificadas:")
print(consultas_maestro.columns.tolist())

Se evidencia que la cantidad de datos en la unión es la misma que cuando se hizo el conteo por tabla. 
- **1'432.430** de datos para las consultas virtuales desde 2022 a 2025

In [ ]:
consultas_maestro.sample(20)

Ahora vamos a organizar los nombres de todas las variables para que estén en un formato para que el modelo se pueda realizar sin problema. Es decir, se deben quitar tildes, mayúsculas y elementos raros como Ñ.

In [ ]:
#----------Quitamos la columna Año (ya tenemos Anio)------------

consultas_maestro = consultas_maestro.drop(columns=['Año'])

#------------------Quitamos tildes y mayúsculas----------------
import unicodedata

# 1. Función para quitar tildes
def limpiar_texto(texto):
    texto = unicodedata.normalize('NFD', texto)
    texto = texto.encode('ascii', 'ignore').decode("utf-8")
    return texto.lower().strip()

# 2. Aplicamos la limpieza a todas las columnas del maestro
consultas_maestro.columns = [limpiar_texto(col) for col in consultas_maestro.columns]

#------------------Ponemos anio al principio-----------------

# Obtenemos la lista de todas las columnas actuales
columnas = consultas_maestro.columns.tolist()

# Movemos 'Anio' al inicio de la lista
# Sacamos 'Anio' de su posición actual y lo insertamos en la posición 0
columnas.insert(0, columnas.pop(columnas.index('anio')))

# eorganizamos el DataFrame con el nuevo orden
consultas_maestro = consultas_maestro[columnas]

#------------------Verificamos los cambios-------------------
consultas_maestro.head()

In [ ]:
consultas_maestro.sample(10)

Ya con todas las uniones vamos a **tipificar** el dataset (Preprocesamiento o Limpieza Técnica):
- Se realiza la conversión de la variable transaccional fechas de formato object a datetime64, habilitando el uso de descriptores temporales. Asimismo, se transformaron las variables nominales (Facultad, Programa, etc.) al tipo category y las métricas de uso a numeric. Esta estandarización no solo optimiza el consumo de memoria RAM, sino que garantiza la precisión aritmética en el cálculo de indicadores y la integridad del modelo de clustering.

In [ ]:
# 1. Convertir la columna de fechas de Texto a Datetime real
# Pandas detectará automáticamente el formato YYYY-MM-DD HH:MM:SS
consultas_maestro['fechas'] = pd.to_datetime(consultas_maestro['fechas'], errors='coerce')

# 2. Convertir variables de texto repetitivas a tipo 'category'
# Esto reduce el uso de memoria hasta en un 80%
columnas_categoricas = ['sede', 'tipo_de_usuario', 'carrera','nivel_academico', 'facultad', 'codigo','recurso_electronico', 'anio']
for col in columnas_categoricas:
    if col in consultas_maestro.columns:
        consultas_maestro[col] = consultas_maestro[col].astype('category')

# 3. Asegurar que las métricas sean numéricas
# 'coerce' convierte cualquier error (como un texto perdido) en NaN para limpiarlo luego
columnas_numericas = ['sesiones', 'busquedas', 'descargas']
for col in columnas_numericas:
    if col in consultas_maestro.columns:
        consultas_maestro[col] = pd.to_numeric(consultas_maestro[col], errors='coerce')

# 4. Verificación final de los "Dtypes"
print("--- Tipos de Datos Actualizados ---")
print(consultas_maestro.dtypes)

**Ingeniería de características**

Aprovechando la tipificación de la variable fechas a datetime64, se derivaron nuevas dimensiones de análisis: mes, nombre del mes, día de la semana y hora de la transacción.

Esta granularidad facilita la identificación de patrones de estudio según el calendario institucional (ej. periodos de exámenes finales vs. vacaciones) y la determinación de las franjas horarias de mayor demanda tecnológica, lo cual es un insumo crítico para la segmentación de usuarios en el modelo de clustering.

In [ ]:
# 1. Extraer el Mes (Numérico para ordenar y Nombre para gráficas)
consultas_maestro['mes'] = consultas_maestro['fechas'].dt.month
consultas_maestro['mes_nombre'] = consultas_maestro['fechas'].dt.month_name()

# 2. Extraer el Día (0=Lunes, 6=Domingo) y su Nombre
consultas_maestro['dia_semana'] = consultas_maestro['fechas'].dt.dayofweek
consultas_maestro['dia_nombre'] = consultas_maestro['fechas'].dt.day_name()

# 3. Extraer la Hora (0-23) para ver picos de tráfico
consultas_maestro['hora'] = consultas_maestro['fechas'].dt.hour

# 4.
# (Opcional, si te salen en inglés como 'January' o 'Monday')
meses_es = {1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril', 5:'Mayo', 6:'Junio',
            7:'Julio', 8:'Agosto', 9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'}
dias_es = {0:'Lunes', 1:'Martes', 2:'Miercoles', 3:'Jueves', 4:'Viernes', 5:'Sabado', 6:'Domingo'}

consultas_maestro['mes_nombre'] = consultas_maestro['mes'].map(meses_es)
consultas_maestro['dia_nombre'] = consultas_maestro['dia_semana'].map(dias_es)

# Verificamos cómo se ve nuestra nueva "línea de tiempo"
consultas_maestro[['fechas', 'mes_nombre', 'dia_nombre', 'hora']].head()

Vamos a crear una nueva columna que me especifique que estos datos corresponden a las consultas digitales a bases de datos para diferenciar los otros datos que se van a analizar

In [ ]:
# Añadimos la columna identificadora para el entorno digital
consultas_maestro['tipo_recurso'] = 'digital'

# Verificamos que se haya creado correctamente al final
consultas_maestro[['anio', 'facultad', 'tipo_recurso']].head()

In [ ]:
display(consultas_maestro.sample(10))

In [ ]:
consultas_maestro['recurso_electronico'] = consultas_maestro['recurso_electronico'].astype('category')

In [ ]:
# Resumen estructural
print(f"Dimensiones finales: {consultas_maestro.shape}")
print("-" * 30)
print(consultas_maestro.info())

In [ ]:
consultas_maestro.describe()

#### Limpieza de los datos de préstamos físicos

Primero vamos a unir los datos de los programas ya que no los trae, de la tabla programas

In [ ]:
columnas_carreras = ['Año', 'Tipo Trans.', 'Cod. Usuario', 'Departamento', 'Programa']

carreras_2022 = pd.read_excel('informacion_programas/Listado2022 programa.xlsx', names=columnas_carreras, header=0)
carreras_2023 = pd.read_excel('informacion_programas/Listado2023 programa.xlsx', names=columnas_carreras, header=0)
carreras_2024 = pd.read_excel('informacion_programas/Listado2024 programa.xlsx', names=columnas_carreras, header=0)
carreras_2025 = pd.read_excel('informacion_programas/Listado2025 programa.xlsx', names=columnas_carreras, header=0)

In [ ]:
carreras_2022.sample(5)

In [ ]:
prestamos_original2022.sample(5)

Los datos de préstamos físicos del 2022 vienen con una columna llamada Programa con todos los valores vacíos, la vamos a eliminar para evitar errores al momento de hacer el merge con los datos reales de los programas por año.

In [ ]:
prestamos_original2022 = prestamos_original2022.drop(columns=['Programa'])

In [ ]:
#prestamos_original2022.columns.tolist
prestamos_original2023.columns.tolist

Hacemos la unión de las tablas de programa con para los datos de préstamos, con la llave del Cod. Usuario, transformamos la variables Programa por carrera para que coincida con la estructura de los datos de consultas.

In [ ]:
# 1. Unimos cada año con su respectivo listado para ver qué tanto falta

# --- Para 2025 ---
prestamos_2025 = pd.merge(prestamos_original2025, carreras_2025[['Cod. Usuario', 'Programa']], on='Cod. Usuario', how='left', validate="many_to_many")
prestamos_2025.rename(columns={'Programa': 'carrera'}, inplace=True)

# --- Para 2024 ---
prestamos_2024 = pd.merge(prestamos_original2024, carreras_2024[['Cod. Usuario', 'Programa']], on='Cod. Usuario', how='left', validate="many_to_many")
prestamos_2024.rename(columns={'Programa': 'carrera'}, inplace=True)

# --- Para 2023 ---
prestamos_2023 = pd.merge(prestamos_original2023, carreras_2023[['Cod. Usuario', 'Programa']], on='Cod. Usuario', how='left', validate="many_to_many")
prestamos_2023.rename(columns={'Programa': 'carrera'}, inplace=True)

# --- Para 2022 ---
prestamos_2022 = pd.merge(prestamos_original2022, carreras_2022[['Cod. Usuario', 'Programa']], on='Cod. Usuario', how='left', validate="many_to_many")
prestamos_2022.rename(columns={'Programa': 'carrera'}, inplace=True)

print("Cruce anual completado.")

In [ ]:
prestamos_2024.sample(10)

##### Paso 1: Limpieza de Columnas y Tipificación

In [ ]:
prestamos_original2025.columns.tolist

In [ ]:
print("Datos 2022")
display(prestamos_original2022.isnull().sum())
print(prestamos_original2022.shape)

1. Vamos a organizar el nombre de las variables, quitar posibles espacios, etc

In [ ]:
prestamos_2022.columns = prestamos_2022.columns.str.strip()
prestamos_2022.columns = prestamos_2022.columns.str.replace(' ', '_')

prestamos_2023.columns = prestamos_2023.columns.str.strip()
prestamos_2023.columns = prestamos_2023.columns.str.replace(' ', '_')

prestamos_2024.columns = prestamos_2024.columns.str.strip()
prestamos_2024.columns = prestamos_2024.columns.str.replace(' ', '_')

prestamos_2025.columns = prestamos_2025.columns.str.strip()
prestamos_2025.columns = prestamos_2025.columns.str.replace(' ', '_')

In [ ]:
print("Nuevos nombres de columnas:")
print(prestamos_2022.columns.tolist())
print(prestamos_2023.columns.tolist())
print(prestamos_2024.columns.tolist())
print(prestamos_2025.columns.tolist())

Se hace una tabla con el tamaño de cada una de las tablas por año para entender con cuánta información contamos por año y que si coincida al hacer la unión.

In [ ]:
# 1. Contamos las filas de cada DataFrame individual
pconteo_2022 = len(prestamos_2022)
pconteo_2023 = len(prestamos_2023)
pconteo_2024 = len(prestamos_2024)
pconteo_2025 = len(prestamos_2025)

# 2. Creamos la tabla de auditoría
pconsultas_cantidad = {
    'Periodo': ['2022', '2023', '2024', '2025', 'TOTAL DE REGISTROS'],
    'Cantidad_Registros': [
        pconteo_2022, 
        pconteo_2023, 
        pconteo_2024, 
        pconteo_2025, 
        (pconteo_2022 + pconteo_2023 + pconteo_2024 + pconteo_2025),
    ]
}

pconsultas_cantidad = pd.DataFrame(pconsultas_cantidad)


print("--- TABLA DE CANTIDAD DE DATOS POR TABLA ---")
print(pconsultas_cantidad)

2. Se realiza una matriz de valores nulos por cada año
- Ver columnas que podrían eliminarse o dónde es necesario posiblemente imputar, y ver la estructura de las tablas que tan igual es

In [ ]:
pdatasets = {
    '2022': prestamos_2022,
    '2023': prestamos_2023,
    '2024': prestamos_2024,
    '2025': prestamos_2025
}

pdict_resumen = {}

for anio, df in pdatasets.items():
    pdict_resumen[anio] = (df.isnull().sum() / len(df)) * 100


pdf_calidad = pd.DataFrame(pdict_resumen)
pdf_calidad = pdf_calidad.round(1)

pdf_calidad.style.background_gradient(cmap='Reds').format("{:.2f}")

No hay columnas 100% vacías para eliminar

Ahora revisemos la cantidad de nulos.

In [ ]:
# 1. Calculamos la cantidad de nulos por cada año individual
# Asegúrate de usar los nombres de tus variables de préstamos ya procesadas
conteo_2022 = prestamos_2022['carrera'].isna().sum()
conteo_2023 = prestamos_2023['carrera'].isna().sum()
conteo_2024 = prestamos_2024['carrera'].isna().sum()
conteo_2025 = prestamos_2025['carrera'].isna().sum()
total_nulos = conteo_2022 + conteo_2023 + conteo_2024 + conteo_2025

# 2. Creamos el DataFrame de auditoría
df_auditoria_nulos = pd.DataFrame({
    'Periodo': ['2022', '2023', '2024', '2025', 'TOTAL DE NULOS'],
    'Cantidad_Nulos': [conteo_2022, conteo_2023, conteo_2024, conteo_2025, total_nulos]
})

# 3. Visualización limpia
print("REGISTROS SIN CARRERA (NULOS):")
display(df_auditoria_nulos)

Se identificó una brecha de información significativa en la variable carrera, con un promedio de nulos del 43%. Este fenómeno sugiere una alta presencia de usuarios 'longitudinales' (estudiantes de cohortes anteriores no presentes en los listados anuales vigentes). Se determina como acción necesaria realizar una Imputación Cruzada basados en años anteriores para hacer la imputación.

Primero hay que transformar las variables de los datos de carreras para que coincidan

In [ ]:
carreras_2022.columns = carreras_2022.columns.str.strip()
carreras_2022.columns = carreras_2022.columns.str.replace(' ', '_')

carreras_2023.columns = carreras_2023.columns.str.strip()
carreras_2023.columns = carreras_2023.columns.str.replace(' ', '_')

carreras_2024.columns = carreras_2024.columns.str.strip()
carreras_2024.columns = carreras_2024.columns.str.replace(' ', '_')

carreras_2025.columns = carreras_2025.columns.str.strip()
carreras_2025.columns = carreras_2025.columns.str.replace(' ', '_')

In [ ]:
print("Nuevos nombres de columnas:")
print(carreras_2022.columns.tolist())
print(carreras_2023.columns.tolist())
print(carreras_2024.columns.tolist())
print(carreras_2025.columns.tolist())

In [ ]:
carreras_2025.sample(10)

Vamos a crear el diccionario para imputar datos faltantes de la carrera con los de otros años

In [ ]:
# 1. Unimos todos los listados de alumnos de los 4 años
# Usamos las variables que ya tienes en tu entorno
listados_estudiantes = [carreras_2022, carreras_2023, carreras_2024, carreras_2025]
estudiantes_total = pd.concat(listados_estudiantes, ignore_index=True)

# 2. Creamos el Diccionario Maestro (Cod. Usuario -> Carrera)
# Eliminamos duplicados: si un alumno aparece en varios años, 
# nos quedamos con el registro más reciente (.drop_duplicates con keep='last')
diccionario_maestro = (
    estudiantes_total[['Cod._Usuario', 'Programa']]
    .drop_duplicates(subset=['Cod._Usuario'], keep='last')
    .rename(columns={'Programa': 'carrera_maestra'})
)

print(f"Diccionario Maestro reconstruido con {len(diccionario_maestro):,} alumnos únicos.")

In [ ]:
diccionario_maestro.sample(5)

In [ ]:
# 1. ASIGNAR ETIQUETA DE AÑO A CADA TABLA INDIVIDUAL
# Esto garantiza que al unir todo, sepamos a qué periodo pertenece cada fila
prestamos_2022['Año'] = 2022
prestamos_2023['Año'] = 2023
prestamos_2024['Año'] = 2024
prestamos_2025['Año'] = 2025

# 2. CREAR EL MAESTRO INICIAL (CONCATENACIÓN)
# Unimos los 308,165 registros en un solo dataframe
prestamos_maestro = pd.concat([prestamos_2022, prestamos_2023, prestamos_2024, prestamos_2025], ignore_index=True)

In [ ]:
prestamos_maestro.sample(5)

In [ ]:
# 3. PROCESO DE IMPUTACIÓN (RESCATE DE CARRERAS)
# Cruzamos con el 'diccionario_maestro' que contiene la historia académica de los alumnos
prestamos_maestro = pd.merge(prestamos_maestro, diccionario_maestro, on='Cod._Usuario', how='left')

# Llenamos los huecos de 'carrera' usando la información rescatada ('carrera_maestra')
prestamos_maestro['carrera'] = prestamos_maestro['carrera'].fillna(prestamos_maestro['carrera_maestra'])

# 4. LIMPIEZA FINAL Y NORMALIZACIÓN
# Borramos la columna de apoyo y formateamos el texto para que sea uniforme
prestamos_maestro.drop(columns=['carrera_maestra'], inplace=True, errors='ignore')

# Convertimos a string, quitamos tildes, pasamos a mayúsculas y eliminamos espacios extra
prestamos_maestro['carrera'] = (
    prestamos_maestro['carrera']
    .astype(str)
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
    .str.upper()
    .str.strip()
)

# 5. RESTAURACIÓN DE NULOS REALES (NaN)
# Reemplazamos las representaciones de texto de nulos por valores NaN técnicos
prestamos_maestro['carrera'] = prestamos_maestro['carrera'].replace(['NAN', 'nan', 'NONE', 'NULL'], np.nan)

print(f"Maestro Creado con éxito.")
print(f"Total de registros: {len(prestamos_maestro):,}")
print(f"Columnas disponibles: {prestamos_maestro.columns.tolist()}")

# Visualizar una muestra para confirmar la columna 'Año'
display(prestamos_maestro[['Año', 'Fecha_Trans.', 'Cod._Usuario', 'carrera']].sample(5))
prestamos_maestro.sample(10)

In [ ]:
nulos_finales = prestamos_maestro['carrera'].isna().sum()
print(nulos_finales)

Se disminuyó la cantidad de nulos en la variable carrera pasando de 129.052 a 128.807

In [ ]:
prestamos_maestro.columns

Creamos nuevamente la tabla de cantidad de nulos para ver el estado.

In [ ]:
# 1. Definimos los años que queremos analizar
anios_interes = [2022, 2023, 2024, 2025]
p_resumen_final = {}

# 2. Iteramos sobre la columna 'Año' (o 'anio' si lo cambiaste)
for anio in anios_interes:
    # Filtramos el maestro por el año correspondiente
    # Nota: Si tu columna se llama 'Año', asegúrate de escribirlo tal cual
    df_anio = prestamos_maestro[prestamos_maestro['Año'] == anio]
    
    if not df_anio.empty:
        # Calculamos el % de nulos por columna para ese año
        p_resumen_final[str(anio)] = (df_anio.isnull().sum() / len(df_anio)) * 100

# 3. Creamos el DataFrame de calidad final
pdf_calidad_final = pd.DataFrame(p_resumen_final)

# 4. Visualización con Gradiente de Color
print("ESTADO FINAL NULOS EN DATOS (POST-IMPUTACIÓN)")
pdf_calidad_final.style.background_gradient(cmap='YlOrRd').format("{:.2f}%")

Así queda la estructura del maestro de datos de préstamos físicos

In [ ]:
prestamos_maestro.dtypes

In [ ]:
prestamos_maestro['Cod._Ejemplar']

Ahora vamos a renombrar las variables de los préstamos de tal manera que coincidan con las de consultas que ya creamos anteriormente

In [ ]:
prestamos_maestro = prestamos_maestro.rename(columns={
    'Cod._Usuario': 'codigo',
    'Año': 'anio',
    'Departamento' : 'facultad',
    'carrera' : 'carrera',
    'Fecha_Trans.' : 'fechas',
    'Título' : 'titulo'
})

In [ ]:
prestamos_maestro.dtypes

In [ ]:
consultas_maestro.sample()

In [ ]:
prestamos_maestro.sample()

Vamos a convertir primero la columna de fecha de préstamo en las mismas que convertimos en las consultas

Vamos a organizar el formato de cada variable

In [ ]:
prestamos_maestro['fechas'] = pd.to_datetime(prestamos_maestro['fechas'], errors='coerce')
prestamos_maestro['anio'] = prestamos_maestro['anio'].astype('category')
prestamos_maestro['Tipo_Trans'] = prestamos_maestro['Tipo_Trans.'].astype('category')
prestamos_maestro['facultad'] = prestamos_maestro['facultad'].astype('category')
prestamos_maestro['codigo'] = prestamos_maestro['codigo'].astype('category')
prestamos_maestro['titulo'] = prestamos_maestro['titulo'].astype('category')
prestamos_maestro['Autor(es)'] = prestamos_maestro['Autor(es)'].astype('category')
prestamos_maestro['Sig._Topográfica'] = prestamos_maestro['Sig._Topográfica'].astype('category')
prestamos_maestro['Cat._Ejemplar'] = prestamos_maestro['Cat._Ejemplar'].astype('category')
prestamos_maestro['carrera'] = prestamos_maestro['carrera'].astype('category')
prestamos_maestro['Cod._Ejemplar'] = prestamos_maestro['Cod._Ejemplar'].astype('category')

prestamos_maestro.dtypes

Ahora vamos a organizar los nombres de todas las variables faltantes por organizar para que no presente problema en el modelo

In [ ]:
# 1. Función para quitar tildes creada anteriormente
prestamos_maestro.columns = [limpiar_texto(col) for col in prestamos_maestro.columns]

#------------------Ponemos anio al principio-----------------

# Obtenemos la lista de todas las columnas actuales
pcolumnas = prestamos_maestro.columns.tolist()

# eorganizamos el DataFrame con el nuevo orden
prestamos_maestro = prestamos_maestro[pcolumnas]

# Quitamos puntos
prestamos_maestro.columns = [
    col.lower().replace('.', '').replace(' ', '_').strip() 
    for col in prestamos_maestro.columns]

#------------------Verificamos los cambios-------------------
prestamos_maestro.head()

In [ ]:
prestamos_maestro.sample(10)

In [ ]:
prestamos_maestro.columns.tolist

In [ ]:
# 1. Extraer el Mes (Numérico para ordenar y Nombre para gráficas)
prestamos_maestro['mes'] = prestamos_maestro['fechas'].dt.month
prestamos_maestro['mes_nombre'] = prestamos_maestro['fechas'].dt.month_name()

# 2. Extraer el Día (0=Lunes, 6=Domingo) y su Nombre
prestamos_maestro['dia_semana'] = prestamos_maestro['fechas'].dt.dayofweek
prestamos_maestro['dia_nombre'] = prestamos_maestro['fechas'].dt.day_name()

# 3. Extraer la Hora (0-23) para ver picos de tráfico
prestamos_maestro['hora'] = prestamos_maestro['fechas'].dt.hour

# 4.
# (Opcional, si te salen en inglés como 'January' o 'Monday')
pmeses_es = {1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril', 5:'Mayo', 6:'Junio',
            7:'Julio', 8:'Agosto', 9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'}
pdias_es = {0:'Lunes', 1:'Martes', 2:'Miercoles', 3:'Jueves', 4:'Viernes', 5:'Sabado', 6:'Domingo'}

prestamos_maestro['mes_nombre'] = prestamos_maestro['mes'].map(pmeses_es)
prestamos_maestro['dia_nombre'] = prestamos_maestro['dia_semana'].map(pdias_es)

# Verificamos cómo se ve nuestra nueva "línea de tiempo"
prestamos_maestro[['fechas', 'mes_nombre', 'dia_nombre', 'hora']].sample(10)

Vamos a crear una nueva columna que me especifique que estos datos corresponden a los préstamos físicos para diferenciar los otros datos que se van a analizar

In [ ]:
# Añadimos la columna identificadora para el entorno físico
prestamos_maestro['tipo_recurso'] = 'fisico'

# Verificamos que se haya creado correctamente al final
prestamos_maestro[['anio', 'facultad', 'tipo_recurso']].head()

##### Vamos a mirar cómo relacionamos las bases de datos y los libros físicos en una sola categoría

In [ ]:
print(list(consultas_maestro['recurso_electronico'].unique()))

In [ ]:
# Crear un resumen de plataformas con su conteo de uso
resumen_plataformas = consultas_maestro['recurso_electronico'].value_counts().reset_index()
resumen_plataformas.columns = ['Plataforma', 'Cantidad_Consultas']

# Mostrar las 74 filas completas (sin recortes)
with pd.option_context('display.max_rows', None):
    display(resumen_plataformas)

##### Vamos a eliminar columnas de ambos maestros que ya sabemos no nos sirven

In [ ]:
prestamos_maestro.sample(10)

In [ ]:
prestamos_maestro = prestamos_maestro.drop(columns='fecha_devol', errors='ignore')
prestamos_maestro = prestamos_maestro.drop(columns='cod._ejemplar', errors='ignore')
prestamos_maestro = prestamos_maestro.drop(columns='tipo_trans', errors='ignore')

consultas_maestro = consultas_maestro.drop(columns='sede', errors='ignore')


In [ ]:
# 1. Convertir variables numéricas a int32 (como en consultas)
# Usamos 'Int32' con mayúscula para manejar posibles nulos sin error
cols_to_int = ['mes', 'dia_semana', 'hora']
for col in cols_to_int:
    prestamos_maestro[col] = prestamos_maestro[col].astype('Int32')

# 2. Asegurar que las variables de texto sean tipo 'object' (como en consultas)
cols_to_obj = ['mes_nombre', 'dia_nombre', 'tipo_recurso']
for col in cols_to_obj:
    prestamos_maestro[col] = prestamos_maestro[col].astype(object)

# 3. Verificar que los dtypes ahora coincidan
print("Dtypes actuales en Préstamos (Físico):")
print(prestamos_maestro[['anio', 'mes', 'mes_nombre', 'dia_semana', 'dia_nombre', 'hora', 'tipo_recurso']].dtypes)

In [ ]:
prestamos_maestro.sample(10)

In [ ]:
prestamos_maestro.dtypes

In [ ]:
consultas_maestro.dtypes

In [ ]:
consultas_maestro.sample(10)

Vamos a normalizar el nombre de las facultades ya que se identificó diferencias en estos.

In [ ]:
# 1. Función para limpiar el "ruido" (minúsculas, tildes y códigos raros)
def limpieza_inicial(texto):
    import re
    texto = str(texto).lower()
    # Quita códigos iniciales como "511-0-131-01 " si existen
    texto = re.sub(r'^\d+[-\d]*\s+', '', texto)
    # Quita tildes para evitar problemas de codificación
    remplazos = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", "ü": "u"}
    for original, limpio in remplazos.items():
        texto = texto.replace(original, limpio)
    return texto.strip()

In [ ]:
# 2. Aplicamos la limpieza básica a ambos maestros
prestamos_maestro['facultad'] = prestamos_maestro['facultad'].apply(limpieza_inicial)
consultas_maestro['facultad'] = consultas_maestro['facultad'].apply(limpieza_inicial)

In [ ]:
prestamos_maestro['carrera'].unique()

In [ ]:
consultas_maestro['carrera'].unique()

In [ ]:
# --- MAESTRO DE PRÉSTAMOS (Físico) ---
# Agregamos observed=False para silenciar el aviso de pandas
demanda_fisica_carrera = prestamos_maestro.groupby('carrera', observed=False).agg({
    'codigo': 'nunique',       # Estudiantes únicos
    'anio': 'count'            # Total de préstamos
}).rename(columns={'codigo': 'estudiantes_unicos', 'anio': 'total_prestamos'})

print("Resumen de Demanda Física por Facultad:")
display(demanda_fisica_carrera)

In [ ]:
# --- MAESTRO DE PRÉSTAMOS (Físico) ---
# Agregamos observed=False para silenciar el aviso de pandas
demanda_virtual_carrera = consultas_maestro.groupby('carrera', observed=False).agg({
    'codigo': 'nunique',       # Estudiantes únicos
    'anio': 'count'            # Total de préstamos
}).rename(columns={'codigo': 'estudiantes_unicos', 'anio': 'total_prestamos'})

print("Resumen de Demanda virtual por Facultad:")
display(demanda_virtual_carrera)

In [ ]:
# -----------------------------------------------------------------------------------------------------------------
# Vamos a organizar el nombre de los libros y bases de datos para hacer el paso de creación de diversity_recursos
# -----------------------------------------------------------------------------------------------------------------

consultas_maestro.rename(columns={'recurso_electronico': 'nombre_recurso'}, inplace=True)

prestamos_maestro.rename(columns={'titulo': 'nombre_recurso'}, inplace=True)

**TERMINACIÓN PASO 1. Limpieza y estandarización. Ya tenemos los dos "maestros" (Físico y Digital) con los mismos nombres de columnas y tipos de datos.**

### Análisis de maestros por aparte

¿Cuántos alumnos de cada facultad piden prestados libros en la biblioteca?
¿Cuál es el volumen de libros prestados?

In [ ]:
prestamos_maestro.columns.tolist

In [ ]:
consultas_maestro.columns.tolist

In [ ]:
# --- MAESTRO DE PRÉSTAMOS (Físico) ---
# Agregamos observed=False para silenciar el aviso de pandas
demanda_fisica_facultad = prestamos_maestro.groupby('facultad', observed=False).agg({
    'codigo': 'nunique',       # Estudiantes únicos
    'anio': 'count'            # Total de préstamos
}).rename(columns={'codigo': 'estudiantes_unicos', 'anio': 'total_prestamos'})

print("Resumen de Demanda Física por Facultad:")
display(demanda_fisica_facultad)



In [ ]:
# --- MAESTRO DE CONSULTAS (Digital) ---
demanda_digital_facultad = consultas_maestro.groupby('facultad', observed=False).agg({
    'codigo': 'nunique',
    'anio': 'count'
}).rename(columns={'codigo': 'estudiantes_unicos', 'anio': 'total_consultas'}).sort_values(by='total_consultas', ascending=False)

print("Resumen de Demanda Digital por Facultad:")
display(demanda_digital_facultad)

In [ ]:
consultas_maestro.sample(10)

- La Facultad de Salud se identifica como el principal motor de consumo digital de la biblioteca. Su volumen de consultas digitales supera exponencialmente a sus préstamos físicos, lo que sugiere un perfil de usuario que depende críticamente de bases de datos, revistas indexadas y recursos electrónicos de actualización constante.
- Tanto Ingeniería como Tecnología muestran una demanda robusta en ambos frentes. Aunque su consumo digital es alto, mantienen una interacción significativa con la colección física, lo que las define como facultades de perfil híbrido. Esto indica que sus procesos de aprendizaje aún requieren el soporte de textos técnicos físicos y laboratorios.
- La categoría de Apoyo Control presenta un comportamiento atípico: es la única donde la demanda física es superior a la digital. Esto revela una oportunidad de mejora o una brecha de digitalización en los procesos administrativos o de consulta de esta unidad, que sigue anclada al formato papel.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

# 1. Aseguramos que la columna 'fecha' sea tipo datetime y extraemos el mes
# (Si tus columnas se llaman distinto, ajusta el nombre 'fecha_prestamo' o similar)
prestamos_maestro['mes'] = pd.to_datetime(prestamos_maestro['fechas']).dt.month
consultas_maestro['mes'] = pd.to_datetime(consultas_maestro['fechas']).dt.month

# 2. Agrupamos por mes
uso_mensual_fisico = prestamos_maestro.groupby('mes').size().reset_index(name='cantidad')
uso_mensual_digital = consultas_maestro.groupby('mes').size().reset_index(name='cantidad')

# 3. Configuración estética UTP
plt.figure(figsize=(10, 4))
sns.set_theme(style="whitegrid")
color_fisico = "#A31D1D"
color_digital = "#2D5A27"

# 4. Graficamos las líneas
sns.lineplot(data=uso_mensual_fisico, x='mes', y='cantidad', color=color_fisico, marker='o', linewidth=3)
sns.lineplot(data=uso_mensual_digital, x='mes', y='cantidad', color=color_digital, marker='s', linewidth=3)

# 5. Personalización de la gráfica
plt.title('Evolución Mensual de la Demanda: Físico vs. Digital', fontsize=16, fontweight='bold')
plt.xlabel('Mes del Año', fontsize=12)
plt.ylabel('Total de Interacciones', fontsize=12)
plt.xticks(range(1, 13), ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'])

# Leyenda manual para que no falle el color
parche_fisico = mpatches.Patch(color=color_fisico, label='Préstamos Físicos')
parche_digital = mpatches.Patch(color=color_digital, label='Consultas Digitales')
plt.legend(handles=[parche_fisico, parche_digital])

plt.tight_layout()
plt.show()

Se observa una asimetría en la disponibilidad y uso de los servicios: mientras la demanda digital se mantiene activa durante los 12 meses del año, el servicio físico presenta periodos de inactividad total en diciembre y enero. Esto demuestra que la infraestructura digital de la biblioteca garantiza la continuidad académica incluso cuando las instalaciones físicas están cerradas por periodos administrativos o vacacionales.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Función para categorizar
def categorizar(n):
    if n <= 3: return 'Ocasional (1-3)'
    elif n <= 10: return 'Habitual (4-10)'
    else: return 'Frecuente (>10)'

# 2. Preparamos los datos
freq_fisica = prestamos_maestro.groupby('codigo').size().apply(categorizar).reset_index(name='tipo')
freq_fisica['Servicio'] = 'Físico'

freq_digital = consultas_maestro.groupby('codigo').size().apply(categorizar).reset_index(name='tipo')
freq_digital['Servicio'] = 'Digital'

# Unimos para graficar
df_tribus = pd.concat([freq_fisica, freq_digital])
orden = ['Ocasional (1-3)', 'Habitual (4-10)', 'Frecuente (>10)']

# 3. Gráfica de Barras Agrupadas
plt.figure(figsize=(10, 5))
sns.countplot(data=df_tribus, x='tipo', hue='Servicio', order=orden, palette=["#A31D1D", "#2D5A27"])

plt.title('Distribución de Usuarios por Nivel de Uso', fontsize=16, fontweight='bold')
plt.xlabel('Tipo de Usuario (Según cantidad de usos al año)', fontsize=12)
plt.ylabel('Número de Estudiantes', fontsize=12)
plt.legend(title='Servicio')
plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

Aquí no estamos contando cuántos libros se prestaron, sino cuántos estudiantes pertenecen a cada perfil de fidelidad

- La barra verde en la categoría "Frecuente (>10)" es la más alta de toda la gráfica (cerca de 9,000 estudiantes). Esto indica que el servicio digital no solo es el más usado, sino el que genera más lealtad.
- Nota que en la categoría "Ocasional", la barra roja supera a la verde. Esto sugiere que muchos estudiantes van a la biblioteca física "por necesidad del momento", pero no regresan con tanta frecuencia como lo hacen al portal digital.
- Mientras que en el mundo físico la diferencia entre usuarios ocasionales y frecuentes es pequeña, en el mundo digital hay una tendencia de usuarios frecuentes.

- En todas las facultades analizadas, el servicio Digital experimentó un crecimiento masivo entre 2022 y 2024. Por ejemplo, en SALUD, las consultas pasaron de 26,733 a un pico de 181,611 en solo dos años.
- Para 2024, la demanda digital superó drásticamente a la física en todos los casos. En INGENIERÍAS, el uso digital (76,381) fue casi 9 veces superior al préstamo físico (8,291) en ese mismo periodo.
- Existe una caída constante y generalizada en los Préstamos Físicos hacia 2025. En facultades como CIENCIAS_EDUCACIÓN, aunque el préstamo físico intentó mantenerse estable hasta 2024 (13,133), sufrió una caída abrupta en 2025 llegando a su punto más bajo (5,354).

* SALUD como Motor Digital: Es la facultad con mayor volumen de interacción total, estableciendo el estándar de consumo virtual de la institución.
* CIENCIAS_AMBIENTALES: Aunque tiene los volúmenes más bajos del grupo, sigue fielmente el patrón de migración hacia lo digital, alcanzando su pico en 2024 con 15,049 consultas.


Este gráfico demuestra que la biblioteca ha dejado de ser un espacio de circulación de objetos (libros físicos) para convertirse en un portal de acceso a información (recursos digitales). Los colores institucionales intensos subrayan que este cambio no es una moda pasajera, sino el nuevo estándar operativo de la universidad.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Función para encontrar la columna del año dinámicamente
def obtener_columna_anio(df):
    # Buscamos cualquier columna que diga "año" o "anio" (ignora mayúsculas)
    cols = [c for c in df.columns if 'año' in c.lower() or 'anio' in c.lower()]
    return cols[0] if cols else None

col_f = obtener_columna_anio(prestamos_maestro)
col_d = obtener_columna_anio(consultas_maestro)

# 2. Consolidación de datos (Copiamos para no alterar la base original)
f_anual = prestamos_maestro.copy()
d_anual = consultas_maestro.copy()

# Forzamos que sean números enteros para evitar errores de categorías
f_anual[col_f] = pd.to_numeric(f_anual[col_f], errors='coerce').fillna(0).astype(int)
d_anual[col_d] = pd.to_numeric(d_anual[col_d], errors='coerce').fillna(0).astype(int)

# Agrupamos usando el nombre real de la columna
resumen_f = f_anual.groupby(col_f).size().reset_index(name='Fisico')
resumen_d = d_anual.groupby(col_d).size().reset_index(name='Digital')

# Unimos las tablas usando los nombres detectados
df_evolucion = pd.merge(resumen_f, resumen_d, left_on=col_f, right_on=col_d, how='outer').fillna(0)
df_evolucion['Año_Final'] = df_evolucion[col_f].astype(int)
df_evolucion = df_evolucion.sort_values('Año_Final')

# 3. Gráfica con Colores Mate
plt.figure(figsize=(12, 6))
# Volvemos a los colores UTP intensos
color_f = '#A31D1D' # Rojo UTP
color_d = '#2D5A27' # Verde UTP

plt.plot(df_evolucion['Año_Final'], df_evolucion['Fisico'], 
         marker='o', linewidth=3, color=color_f, label='Préstamos Físicos')

plt.plot(df_evolucion['Año_Final'], df_evolucion['Digital'], 
         marker='s', linewidth=3, color=color_d, label='Consultas Digitales')

# 4. Etiquetas de valores organizadas
for x, y_f, y_d in zip(df_evolucion['Año_Final'], df_evolucion['Fisico'], df_evolucion['Digital']):
    # Etiquetas de Préstamos Físicos por debajo
    if y_f > 0:
        plt.text(x, y_f - 15000, f'{int(y_f):,}', ha='center', va='top', color=color_f, fontweight='bold')
    # Etiquetas de Consultas Digitales por encima
    if y_d > 0:
        plt.text(x, y_d + 15000, f'{int(y_d):,}', ha='center', va='bottom', color=color_d, fontweight='bold')

# Estética final
plt.title('Evolución Histórica de la Demanda (2022-2025)', fontsize=16, fontweight='bold', pad=25)
plt.xlabel('Año Académico', fontsize=12)
plt.ylabel('Total de Interacciones', fontsize=12)
plt.xticks(df_evolucion['Año_Final'].unique())

# Ajustamos límites del eje Y para que las etiquetas superiores no se corten
plt.ylim(0, df_evolucion['Digital'].max() + 50000)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(frameon=False, fontsize=11, loc='upper left')

plt.tight_layout()
plt.show()

Los resultados longitudinales confirman una transición estructural en el ecosistema de la Biblioteca UTP. El servicio ha evolucionado de un modelo híbrido equilibrado en 2022 hacia una hegemonía digital absoluta en 2024. Esta tendencia no solo valida la inversión en bases de datos virtuales, sino que plantea la necesidad de reconsiderar el uso de los espacios físicos de la biblioteca hacia áreas de coworking o aprendizaje colaborativo, dado el declive sistemático del préstamo domiciliario de libros.

Conclusión Parte 2

1. La "Muerte" del Modelo Tradicional (2022-2024)
La biblioteca ha dejado de ser un centro de intercambio de objetos físicos para convertirse en un hub de servicios digitales.

Dato de soporte: En solo dos años, la consulta digital pasó de ser el 50% de la actividad (2022) a representar casi el 85% del tráfico total en 2024.

Conclusión: El usuario de la UTP ya no busca el libro como objeto, sino la información como servicio inmediato.

2. Disparidad de Adopción por Áreas del Conocimiento
No todas las facultades consumen igual, lo que obliga a políticas diferenciadas:

Liderazgo Tecnológico (Salud e Ingenierías): Presentan una migración casi total. En Salud, la brecha es de 37 a 1 a favor de lo digital. Esto sugiere que sus contenidos pierden vigencia rápido y requieren la actualización constante de las bases de datos.

Resistencia Romántica (Educación y Bellas Artes): Son los únicos bastiones donde el libro físico mantiene cifras estables o caídas lentas. Aquí el formato impreso aún tiene un valor pedagógico o táctil esencial.

3. Eficiencia Operativa vs. Espacio Físico
El declive constante del préstamo físico (-8% anual sostenido) frente a la explosión digital (+400% acumulado) plantea una pregunta estratégica para la Universidad:

Conclusión: Si los libros físicos se prestan menos, el espacio físico de la biblioteca debe reconvertirse. Los estantes de libros están subutilizados, mientras que la demanda de conectividad y acceso remoto es la que sostiene la relevancia de la institución.

In [ ]:
# Verificar el estado real de los datos en cada maestro
print("=== PRÉSTAMOS — estadísticas básicas ===")
print(f"Registros: {prestamos_maestro.shape[0]:,}")
print(f"Usuarios únicos: {prestamos_maestro['codigo'].nunique():,}")
print(f"Rango de fechas: {prestamos_maestro['fechas'].min()} → {prestamos_maestro['fechas'].max()}")
print(f"\nValores nulos:")
nulos_p = prestamos_maestro.isnull().sum()
print(nulos_p[nulos_p > 0])

print("\n=== CONSULTAS — estadísticas básicas ===")
print(f"Registros: {consultas_maestro.shape[0]:,}")
print(f"Usuarios únicos: {consultas_maestro['codigo'].nunique():,}")
print(f"Rango de fechas: {consultas_maestro['fechas'].min()} → {consultas_maestro['fechas'].max()}")
print(f"\nValores nulos:")
nulos_c = consultas_maestro.isnull().sum()
print(nulos_c[nulos_c > 0])

print("\n=== CONSULTAS — tipo_de_usuario únicos ===")
print(consultas_maestro['tipo_de_usuario'].value_counts().head(10))

print("\n=== CONSULTAS — sesiones/busquedas/descargas describe ===")
print(consultas_maestro[['sesiones','busquedas','descargas']].describe())

print("\n=== PRÉSTAMOS — facultad (antes de limpiar) ===")
print(f"Categorías únicas de facultad: {prestamos_maestro['facultad'].nunique()}")
print(prestamos_maestro['facultad'].value_counts().head(10))

In [ ]:
# Corregir antes de correr los dashboards
prestamos_maestro['anio'] = pd.to_numeric(prestamos_maestro['anio'], errors='coerce')
consultas_maestro['anio'] = pd.to_numeric(consultas_maestro['anio'], errors='coerce')

# Verificar
print("Tipo anio préstamos:", prestamos_maestro['anio'].dtype)
print("Tipo anio consultas:", consultas_maestro['anio'].dtype)
print("Valores únicos préstamos:", sorted(prestamos_maestro['anio'].dropna().unique()))
print("Valores únicos consultas:", sorted(consultas_maestro['anio'].dropna().unique()))

In [ ]:
# ============================================================
# EDA INICIAL COMPLETO — VERSIÓN CORREGIDA
# Préstamos + Consultas con pies corregidos
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
import pandas as pd

COLOR_FIS = '#B71C1C'
COLOR_DIG = '#1B5E20'

prestamos_maestro['anio'] = pd.to_numeric(prestamos_maestro['anio'], errors='coerce')
consultas_maestro['anio'] = pd.to_numeric(consultas_maestro['anio'], errors='coerce')

meses_nombres = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
                 7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

rangos         = [0, 3, 10, 50, 99999]
etiquetas_freq = ['Ocasional\n(1-3)', 'Habitual\n(4-10)',
                  'Frecuente\n(11-50)', 'Superusuario\n(>50)']

# ============================================================
# DASHBOARD 1: PRÉSTAMOS FÍSICOS
# ============================================================
fig1 = plt.figure(figsize=(20, 16))
gs1  = gridspec.GridSpec(3, 3, figure=fig1, hspace=0.55, wspace=0.4)

# --- Panel 1: Calidad de datos ---
ax1 = fig1.add_subplot(gs1[0, 0])
cols_nulos = ['carrera', 'fechas', 'autor(es)', 'titulo', 'cod_ejemplar', 'facultad', 'tipo_trans']
nulos_pct  = [prestamos_maestro[c].isnull().sum() / len(prestamos_maestro) * 100
              if c in prestamos_maestro.columns else 0 for c in cols_nulos]
colores_n  = ['#E53935' if p > 20 else '#FF9800' if p > 5 else '#43A047' for p in nulos_pct]
bars1 = ax1.barh(cols_nulos, nulos_pct, color=colores_n, edgecolor='white', alpha=0.85)
for bar, val in zip(bars1, nulos_pct):
    if val > 0.1:
        ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8, fontweight='bold')
ax1.axvline(x=20, color='red', linestyle='--', alpha=0.4, linewidth=1)
ax1.set_title('Calidad de Datos\n(% Valores Nulos)', fontsize=10, fontweight='bold')
ax1.set_xlabel('% Nulos')
ax1.grid(axis='x', linestyle='--', alpha=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Panel 3: Top 8 facultades ---
ax3 = fig1.add_subplot(gs1[0, 2])
fac_p = prestamos_maestro['facultad'].value_counts().head(8)
fac_p_nombres = [str(f).replace('facultad de ','').replace('facultad ','')
                 .title()[:22] for f in fac_p.index]
bars3 = ax3.barh(fac_p_nombres[::-1], fac_p.values[::-1],
                 color=COLOR_FIS, edgecolor='white', alpha=0.85)
for bar, val in zip(bars3, fac_p.values[::-1]):
    ax3.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=7.5)
ax3.set_title('Top 8 Facultades\n(Préstamos Físicos)', fontsize=10, fontweight='bold')
ax3.set_xlabel('Total de Préstamos')
ax3.grid(axis='x', linestyle='--', alpha=0.3)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# --- Panel 4: Evolución anual ---
ax4 = fig1.add_subplot(gs1[1, 0])
anual_fis = prestamos_maestro.groupby('anio').size().reset_index(name='count')
anual_fis = anual_fis[anual_fis['anio'].between(2022, 2025)]
bars4 = ax4.bar(anual_fis['anio'].astype(str), anual_fis['count'],
                color=COLOR_FIS, edgecolor='white', alpha=0.85)
for i, (_, row) in enumerate(anual_fis.iterrows()):
    ax4.text(i, row['count'] + 500, f"{row['count']:,.0f}",
            ha='center', fontsize=8, fontweight='bold')
ax4.set_title('Préstamos por Año', fontsize=10, fontweight='bold')
ax4.set_ylabel('Total Préstamos')
ax4.grid(axis='y', linestyle='--', alpha=0.3)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

# --- Panel 5: Distribución mensual CORREGIDA ---
ax5 = fig1.add_subplot(gs1[1, 1])
mensual_fis = prestamos_maestro.dropna(subset=['mes']).groupby('mes').size().reset_index(name='count')
mensual_fis['mes']       = mensual_fis['mes'].astype(int)
mensual_fis['mes_label'] = mensual_fis['mes'].map(meses_nombres)
mensual_fis = mensual_fis.sort_values('mes')
ax5.bar(range(len(mensual_fis)), mensual_fis['count'],
        color=COLOR_FIS, edgecolor='white', alpha=0.85)
ax5.set_xticks(range(len(mensual_fis)))
ax5.set_xticklabels(mensual_fis['mes_label'], rotation=45, fontsize=8)
ax5.set_title('Préstamos por Mes\n(Acumulado 2022–2025)', fontsize=10, fontweight='bold')
ax5.set_ylabel('Total Préstamos')
ax5.grid(axis='y', linestyle='--', alpha=0.3)
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

# --- Panel 6: Hora ---
ax6 = fig1.add_subplot(gs1[1, 2])
hora_fis = prestamos_maestro.dropna(subset=['hora']).groupby('hora').size().reset_index(name='count')
hora_fis['hora'] = hora_fis['hora'].astype(int)
ax6.bar(hora_fis['hora'], hora_fis['count'],
        color=COLOR_FIS, edgecolor='white', alpha=0.85, width=0.8)
ax6.set_title('Préstamos por Hora del Día', fontsize=10, fontweight='bold')
ax6.set_xlabel('Hora')
ax6.set_ylabel('Total Préstamos')
ax6.set_xticks(range(0, 24, 2))
ax6.grid(axis='y', linestyle='--', alpha=0.3)
ax6.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)

# --- Panel 7: Frecuencia usuarios ---
ax7 = fig1.add_subplot(gs1[2, 0])
freq_usuario  = prestamos_maestro.groupby('codigo').size()
conteos_freq7 = pd.cut(freq_usuario, bins=rangos,
                       labels=etiquetas_freq).value_counts().reindex(etiquetas_freq)
colores_f7 = ['#FFCDD2', '#EF9A9A', COLOR_FIS, '#7B0000']
bars7 = ax7.bar(etiquetas_freq, conteos_freq7.values,
                color=colores_f7, edgecolor='white', alpha=0.9)
for bar, val in zip(bars7, conteos_freq7.values):
    ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:,}', ha='center', fontsize=8, fontweight='bold')
ax7.set_title('Segmentación por\nFrecuencia de Uso Físico', fontsize=10, fontweight='bold')
ax7.set_ylabel('N° de Usuarios')
ax7.grid(axis='y', linestyle='--', alpha=0.3)
ax7.spines['top'].set_visible(False)
ax7.spines['right'].set_visible(False)

# --- Panel 8: Completitud carrera ---
ax8 = fig1.add_subplot(gs1[2, 1])
n_con = prestamos_maestro['carrera'].notna().sum()
n_sin = prestamos_maestro['carrera'].isna().sum()
wedges8, _, autotexts8 = ax8.pie(
    [n_con, n_sin], labels=None,
    colors=['#43A047', '#E53935'],
    autopct='%1.1f%%', startangle=90, pctdistance=0.7,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts8:
    at.set_fontsize(11); at.set_fontweight('bold')
ax8.legend(wedges8,
           [f'Con carrera ({n_con:,})', f'Sin carrera ({n_sin:,})'],
           loc='lower center', bbox_to_anchor=(0.5, -0.12),
           ncol=1, fontsize=9, frameon=True)
ax8.set_title('Completitud de la\nVariable "Carrera"', fontsize=10, fontweight='bold')

# --- Panel 9: Complejidad categórica ---
ax9 = fig1.add_subplot(gs1[2, 2])
categorias = {
    'Facultades\n(préstamos)': prestamos_maestro['facultad'].nunique(),
    'Facultades\n(consultas)': consultas_maestro['facultad'].nunique(),
    'Carreras\n(préstamos)':   prestamos_maestro['carrera'].nunique(),
    'Carreras\n(consultas)':   consultas_maestro['carrera'].nunique(),
}
colores_cat = [COLOR_FIS, COLOR_DIG, COLOR_FIS, COLOR_DIG]
bars9 = ax9.bar(categorias.keys(), categorias.values(),
                color=colores_cat, edgecolor='white', alpha=0.85)
for bar, val in zip(bars9, categorias.values()):
    ax9.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', fontsize=10, fontweight='bold')
ax9.set_title('Complejidad Categórica\n(Antes de Normalizar)', fontsize=10, fontweight='bold')
ax9.set_ylabel('N° Categorías Únicas')
ax9.tick_params(axis='x', labelsize=8)
ax9.grid(axis='y', linestyle='--', alpha=0.3)
ax9.spines['top'].set_visible(False)
ax9.spines['right'].set_visible(False)

plt.suptitle('EDA Inicial — Maestro de Préstamos Físicos\nBiblioteca Jorge Roa Martínez UTP (2022–2025)',
             fontsize=15, fontweight='bold')
plt.savefig('eda_prestamos_inicial.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard préstamos guardado")

In [ ]:
# ============================================================
# DASHBOARD 2: CONSULTAS DIGITALES
# ============================================================
fig2 = plt.figure(figsize=(20, 16))
gs2  = gridspec.GridSpec(3, 3, figure=fig2, hspace=0.55, wspace=0.4)

# --- Panel 1: Calidad datos consultas ---
ax1 = fig2.add_subplot(gs2[0, 0])
cols_nc   = ['facultad', 'nivel_academico', 'carrera', 'tipo_de_usuario']
nulos_c   = [consultas_maestro[c].isnull().sum() / len(consultas_maestro) * 100
             for c in cols_nc]
colores_nc = ['#E53935' if p > 20 else '#FF9800' if p > 5 else '#43A047' for p in nulos_c]
bars1c = ax1.barh(cols_nc, nulos_c, color=colores_nc, edgecolor='white', alpha=0.85)
for bar, val in zip(bars1c, nulos_c):
    ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
ax1.axvline(x=10, color='orange', linestyle='--', alpha=0.4)
ax1.set_title('Calidad de Datos\n(% Valores Nulos)', fontsize=10, fontweight='bold')
ax1.set_xlabel('% Nulos')
ax1.grid(axis='x', linestyle='--', alpha=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Panel 2: Pie tipo usuario CORREGIDO ---
ax2 = fig2.add_subplot(gs2[0, 1])
tipo_u = consultas_maestro['tipo_de_usuario'].value_counts().head(6)
etiq_u = ['Estudiante', 'Egresado', 'Doc. Cátedra', 'Anónimo', 'Doc. Planta', 'Docente']
wedges2, _, autotexts2 = ax2.pie(
    tipo_u.values, labels=None, # type: ignore
    colors=[COLOR_DIG, '#388E3C', '#66BB6A', '#A5D6A7', '#81C784', '#C8E6C9'],
    autopct='%1.1f%%', startangle=90, pctdistance=0.78,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts2:
    at.set_fontsize(9); at.set_fontweight('bold')
ax2.legend(wedges2,
           [f"{et} ({v:,})" for et, v in zip(etiq_u[:len(tipo_u)], tipo_u.values)],
           loc='lower center', bbox_to_anchor=(0.5, -0.18),
           ncol=2, fontsize=7.5, frameon=True)
ax2.set_title('Distribución por\nTipo de Usuario', fontsize=10, fontweight='bold')

# --- Panel 3: Top 8 facultades consultas ---
ax3 = fig2.add_subplot(gs2[0, 2])
fac_d = consultas_maestro['facultad'].value_counts().head(8)
fac_d_nombres = [str(f)[:22].title() for f in fac_d.index]
bars3d = ax3.barh(fac_d_nombres[::-1], fac_d.values[::-1],
                  color=COLOR_DIG, edgecolor='white', alpha=0.85)
for bar, val in zip(bars3d, fac_d.values[::-1]):
    ax3.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=7.5)
ax3.set_title('Top 8 Facultades\n(Consultas Digitales)', fontsize=10, fontweight='bold')
ax3.set_xlabel('Total Consultas')
ax3.grid(axis='x', linestyle='--', alpha=0.3)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# --- Panel 4: Evolución anual consultas ---
ax4 = fig2.add_subplot(gs2[1, 0])
anual_dig = consultas_maestro.groupby('anio').size().reset_index(name='count')
anual_dig = anual_dig[anual_dig['anio'].between(2022, 2025)]
bars4d = ax4.bar(anual_dig['anio'].astype(str), anual_dig['count'],
                 color=COLOR_DIG, edgecolor='white', alpha=0.85)
for i, (_, row) in enumerate(anual_dig.iterrows()):
    ax4.text(i, row['count'] + 2000, f"{row['count']:,.0f}",
            ha='center', fontsize=8, fontweight='bold')
ax4.set_title('Consultas Digitales por Año', fontsize=10, fontweight='bold')
ax4.set_ylabel('Total Consultas')
ax4.grid(axis='y', linestyle='--', alpha=0.3)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

# --- Panel 5: Variables cuantitativas CORREGIDO ---
ax5 = fig2.add_subplot(gs2[1, 1])
vars_dig = ['sesiones', 'busquedas', 'descargas']
medias   = [consultas_maestro[v].mean() for v in vars_dig]
pct_pos  = [(consultas_maestro[v] > 0).sum() / len(consultas_maestro) * 100
             for v in vars_dig]
x = np.arange(3)
bars5a = ax5.bar(x, medias, color=COLOR_DIG, alpha=0.8,
                 edgecolor='white', label='Media por registro', width=0.5)
for bar, val in zip(bars5a, medias):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=9, fontweight='bold', color=COLOR_DIG)

# Porcentaje como texto debajo
for i, pct in enumerate(pct_pos):
    ax5.text(i, -0.04, f'{pct:.0f}%\ncon valor',
            ha='center', fontsize=8, color='gray', style='italic')

ax5.set_xticks(x)
ax5.set_xticklabels(['Sesiones', 'Búsquedas', 'Descargas'], fontsize=10)
ax5.set_ylabel('Media por registro')
ax5.set_ylim(-0.08, max(medias) * 1.3)
ax5.set_title('Variables Cuantitativas Digitales\n(Media y % con valor > 0)',
              fontsize=10, fontweight='bold')
ax5.grid(axis='y', linestyle='--', alpha=0.3)
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

# --- Panel 6: Hora consultas ---
ax6 = fig2.add_subplot(gs2[1, 2])
hora_dig = consultas_maestro.dropna(subset=['hora']).groupby('hora').size().reset_index(name='count')
hora_dig['hora'] = hora_dig['hora'].astype(int)
ax6.bar(hora_dig['hora'], hora_dig['count'],
        color=COLOR_DIG, edgecolor='white', alpha=0.85, width=0.8)
ax6.set_title('Consultas Digitales por Hora del Día', fontsize=10, fontweight='bold')
ax6.set_xlabel('Hora')
ax6.set_ylabel('Total Consultas')
ax6.set_xticks(range(0, 24, 2))
ax6.grid(axis='y', linestyle='--', alpha=0.3)
ax6.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)

# --- Panel 7: Frecuencia usuarios digitales ---
ax7 = fig2.add_subplot(gs2[2, 0])
freq_dig      = consultas_maestro.groupby('codigo').size()
conteos_freq7d = pd.cut(freq_dig, bins=rangos,
                        labels=etiquetas_freq).value_counts().reindex(etiquetas_freq)
colores_f7d = ['#C8E6C9', '#81C784', COLOR_DIG, '#1B5E20']
bars7d = ax7.bar(etiquetas_freq, conteos_freq7d.values,
                 color=colores_f7d, edgecolor='white', alpha=0.9)
for bar, val in zip(bars7d, conteos_freq7d.values):
    ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:,}', ha='center', fontsize=8, fontweight='bold')
ax7.set_title('Segmentación por\nFrecuencia de Uso Digital', fontsize=10, fontweight='bold')
ax7.set_ylabel('N° de Usuarios')
ax7.grid(axis='y', linestyle='--', alpha=0.3)
ax7.spines['top'].set_visible(False)
ax7.spines['right'].set_visible(False)

# --- Panel 8: Distribución mensual consultas ---
ax8 = fig2.add_subplot(gs2[2, 1])
mensual_dig = consultas_maestro.dropna(subset=['mes']).groupby('mes').size().reset_index(name='count')
mensual_dig['mes']       = mensual_dig['mes'].astype(int)
mensual_dig['mes_label'] = mensual_dig['mes'].map(meses_nombres)
mensual_dig = mensual_dig.sort_values('mes')
ax8.bar(range(len(mensual_dig)), mensual_dig['count'],
        color=COLOR_DIG, edgecolor='white', alpha=0.85)
ax8.set_xticks(range(len(mensual_dig)))
ax8.set_xticklabels(mensual_dig['mes_label'], rotation=45, fontsize=8)
ax8.set_title('Consultas Digitales por Mes\n(Acumulado 2022–2025)',
              fontsize=10, fontweight='bold')
ax8.set_ylabel('Total Consultas')
ax8.grid(axis='y', linestyle='--', alpha=0.3)
ax8.spines['top'].set_visible(False)
ax8.spines['right'].set_visible(False)

# --- Panel 9: Nivel académico CORREGIDO ---
ax9 = fig2.add_subplot(gs2[2, 2])
nivel_raw = consultas_maestro['nivel_academico'].value_counts().head(5)
etiq_nivel_map = {
    nivel_raw.index[0]: 'Pregrado',
    nivel_raw.index[1]: 'Posgrado Espec.',
    nivel_raw.index[2]: 'Posgrado Maestría',
    nivel_raw.index[3]: 'Otro / NA',
    nivel_raw.index[4]: 'Doctorado'
} if len(nivel_raw) >= 5 else {k: k[:15] for k in nivel_raw.index}
etiq_nivel = [etiq_nivel_map.get(i, str(i)[:15]) for i in nivel_raw.index]

wedges9, _, autotexts9 = ax9.pie(
    nivel_raw.values, labels=None,
    colors=[COLOR_DIG, '#388E3C', '#66BB6A', '#A5D6A7', '#C8E6C9'],
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts9:
    at.set_fontsize(9); at.set_fontweight('bold')
ax9.legend(wedges9,
           [f"{et} ({v:,})" for et, v in zip(etiq_nivel, nivel_raw.values)],
           loc='lower center', bbox_to_anchor=(0.5, -0.18),
           ncol=2, fontsize=7.5, frameon=True)
ax9.set_title('Distribución por\nNivel Académico', fontsize=10, fontweight='bold')

plt.suptitle('EDA Inicial — Maestro de Consultas Digitales\nBiblioteca Jorge Roa Martínez UTP (2022–2025)',
             fontsize=15, fontweight='bold')
plt.savefig('eda_consultas_inicial.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard consultas guardado")

In [ ]:
consultas_maestro.columns.tolist()

In [ ]:
prestamos_maestro.columns.tolist()

### ORGANIZACIÓN PARA EL CLUSTERING

In [ ]:
consultas_maestro.dtypes

In [ ]:
display(consultas_maestro)

In [ ]:
prestamos_maestro.dtypes

vamos a dejar iguales las variables en ambos maestros para realizar la unión

In [ ]:
# 1. Creamos las columnas que le faltan a Préstamos pero que sí tiene Consultas
prestamos_maestro['sesiones'] = 0
prestamos_maestro['busquedas'] = 0
prestamos_maestro['descargas'] = 0

# 2. Asegúrate de que el nombre del nivel académico sea idéntico en ambos
# (Usaremos el que tiene guion bajo para evitar errores de sintaxis después)
prestamos_maestro['nivel_academico'] = 'pregrado'

# 3. 

In [ ]:
# Limpieza profunda en Préstamos
prestamos_maestro['codigo'] = prestamos_maestro['codigo'].astype(str).str.strip().str.lower()

# Limpieza profunda en Consultas
consultas_maestro['codigo'] = consultas_maestro['codigo'].astype(str).str.strip().str.lower()

In [ ]:
# 3. Diccionario extendido basado en tus capturas (Agrupación de "Vivienda", "Gastos", etc.)
diccionario_tesis_facultades = {
    # ÁREAS ACADÉMICAS
    'facultad de ingenierias': 'ingenierias',
    'facultad de ingenieria mecanica': 'mecanica',
    'facultad de tecnologias': 'tecnologia',
    'facultad de ciencias de la salud': 'salud',
    'facultad de bellas artes y humanidades': 'bellas_artes',
    'facultad de ciencias empresariales' : 'ciencias_empresariales',
    'facultad de ciencias de la educacion' : 'ciencias_educacion',
    'facultad de ciencias agrarias y agroindustria' : 'ciencias_agrarias',
    'facultad de ciencias ambientales' : 'ciencias_ambientales',
    'facultad de ingenierias electrica, electronica, fisica y ciencias de la computacion' : 'ingenierias',
    'licenciatura en musica': 'bellas_artes',
    'ingenieria industrial': 'ciencias_empresariales',
    'administracion de empresas': 'ciencias_empresariales',
    'escuela de musica': 'bellas_artes',
    'escuela de filosofia' : 'bellas_artes',
    'departamento de ciencias clinicas' : 'salud',
    'escuela tecnologia industrial' : 'tecnologia',
    'facultad ingenieria fisica, electrica, electronica y ciencias de la computacion' : 'ingenierias',
    'licenciatura en bilinguismo con enfasis en ingles':'bellas_artes',
    'departamento de humanidades e idiomas':'bellas_artes',
    'escuela de español y comunicacion a.v.':'ciencias_educacion',
    'administracion ambiental': 'ciencias_ambientales',
    'departamento de ciencias basicas ambientales':'ciencias_ambientales',
    'escuela de ciencias sociales':'ciencias_educacion',
    'doctorado en ciencias ambientales':'ciencias_ambientales',
    'departamento de psicopedagogia':'ciencias_educacion',
    'tecnologia en atencion prehospitalaria' : 'salud',
    'departamento de medicina comunitaria' : 'salud',
    'departamento de ciencias basicas de medicina' : 'salud',
    'medicina veterinaria' : 'salud',
    'medicina veterinaria y zootecnia' : 'salud',
    'laboratorio de genetica medica' : 'salud',
    'escuela de deporte y recreacion' : 'salud',
    'ingenieria mecanica':'mecanica',
    'departamento de fisica': 'ingenierias',
    'ingenieria de sistemas y computacion': 'ingenierias',
    'ingenieria electrica': 'ingenierias',
    'ingenieria de sistemas y computacion': 'ingenierias',
    'ingenieria fisica': 'ingenierias',
    'escuela tecnologia electrica':'tecnologia',
    'tecnologia electrica':'tecnologia',
    'tecnologia quimica':'tecnologia',
    'escuela de tecnologia quimica':'tecnologia',
    'facultad de tecnologia':'tecnologia',
    'tecnologia en produccion horticola': 'ciencias_agrarias',
    'ingenieria en procesos sostenibles de las maderas': 'ciencias_agrarias',
    'ingenieria en procesos agroindustriales': 'ciencias_agrarias',
    'departamento de ciencias basicas de la medicina' : 'salud',
    'facultad de mecanica aplicada': 'mecanica',
    'laboratorio financiero (middle office en el proceso de gestion de inversiones y riesgos financieros del modelo tipo endowment de la utp)': 'ciencias_empresariales',
    'mecatronica': 'tecnologia',
    'ingenieria de manufactura': 'tecnologia',
    'facultad de ciencias basicas': 'ciencias_basicas',
    'departamento de matematicas': 'ciencias_basicas',
    'escuela tecnologia mecanica': 'mecanica',
    
    
    
    
    # ÁREAS ADMINISTRATIVAS / APOYO (Aquí agrupamos lo que mencionas)
    'gastos generales': 'apoyo_administrativo',
    'administrativo': 'apoyo_administrativo',
    'fondo de ahorro y vivienda': 'apoyo_administrativo',
    'vivienda': 'apoyo_administrativo',
    'vicerrectoria administrativa y financiera': 'apoyo_administrativo',
    'vicerrectoria administrativa': 'apoyo_administrativo',
    'recursos humanos': 'apoyo_administrativo',
    'gestion de la calidad': 'apoyo_administrativo',
    'laboratorio de gestion': 'apoyo_administrativo',
    'bienestar universitario': 'apoyo_administrativo',
    'vicerrectoria de responsabilidad social y bienestar universitario':'apoyo_administrativo',
    'vicerectoria administrativa': 'apoyo_administrativo',
    'vicerrect. respons. social y bienestar universitar': 'apoyo_administrativo',
    'vicerrectoria academica': 'apoyo_administrativo',
    'vicerrectoria de investigaciones, innovacion y extension': 'apoyo_administrativo',
    'juridica': 'apoyo_administrativo',
    'gestion de tecnologias informaticas y sistemas de informacion': 'apoyo_administrativo',
    'biblioteca e informacion cientifica':'biblioteca',
   
    
    # OTROS
    'anonimo': 'externo',
    'sin departamento': 'externo',
    'jubilado': 'externo',
    'usuario lector' : 'externo',
    'centro de biblioteca' : 'apoyo_administrativo',
    'direccion general de posgrados' : 'apoyo_administrativo',
    'gestion financiera' : 'apoyo_administrativo',
    'gastos generales jornadas especiales': 'apoyo_administrativo',
    'recursos informaticos y educativos': 'apoyo_administrativo',
    'administracion u.t.p.': 'apoyo_administrativo',
    'programa solo para estudiantes visitantes': 'apoyo_administrativo',
    'univirtual': 'univirtual',
    'instituto de lenguas extranjeras - ilex': 'ilex',
    'facultad': 'apoyo_administrativo',
    'laboratorio de analisis de aguas y alimentos': 'apoyo_administrativo',
    'programas de pregrado': 'apoyo_administrativo',
    'usuario externo': 'externo',
}

In [ ]:
# Definimos la lista de columnas que vamos a dejar.
columnas_finales = ['codigo', 'facultad', 'carrera','nombre_recurso','nivel_academico','fechas', 'anio', 'mes', 'dia_semana', 'hora', 'tipo_recurso', 'sesiones', 'busquedas', 'descargas']

# Filtramos ambas tablas para que sean idénticas
prestamos_clustering = prestamos_maestro[columnas_finales]
consultas_clustering = consultas_maestro[columnas_finales]

In [ ]:
# Unión de los dos maestros
df_unificado_clustering = pd.concat([prestamos_clustering, consultas_clustering], axis=0) # type: ignore

In [ ]:
df_unificado_clustering.sample(5)

In [ ]:
df_unificado_clustering.shape

In [ ]:
# ============================================================
# PALETA INSTITUCIONAL — usar en todas las figuras de la tesis
# ============================================================
PALETA = {
    'digital':      '#1A3A5C',  # Azul oscuro institucional
    'fisico':       '#C0392B',  # Rojo vino
    'cluster_0':    '#1A3A5C',  # Azul oscuro
    'cluster_1':    '#2E86AB',  # Azul medio
    'cluster_2':    '#C0392B',  # Rojo
    'cluster_3':    '#8E6B3E',  # Café dorado
    'acento':       '#F39C12',  # Amarillo acento
    'fondo':        '#F7F9FC',  # Gris muy claro
    'grid':         '#DDE3EA',  # Gris para grillas
    'texto':        '#1C2833',  # Casi negro
}

# Fuente base para todas las figuras
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['font.family']      = 'Arial'
mpl.rcParams['axes.spines.top']  = False
mpl.rcParams['axes.spines.right']= False
mpl.rcParams['axes.grid']        = True
mpl.rcParams['grid.linestyle']   = '--'
mpl.rcParams['grid.alpha']       = 0.4
mpl.rcParams['grid.color']       = '#DDE3EA'

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA 1A — Evolución Histórica de la Demanda (2022–2025)
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df_anual_tipo = df_unificado_clustering.groupby(
    ['anio', 'tipo_recurso']
).size().reset_index(name='interacciones')

fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')

for tipo, color, label, marker in [
    ('digital', PALETA['digital'], 'Consultas Digitales', 'o'),
    ('fisico',  PALETA['fisico'],  'Préstamos Físicos',   's')
]:
    subset = df_anual_tipo[df_anual_tipo['tipo_recurso'] == tipo].sort_values('anio')
    ax.plot(subset['anio'], subset['interacciones'],
            marker=marker, color=color, linewidth=2.5,
            markersize=9, label=label, zorder=3)
    for _, row in subset.iterrows():
        # En 2022 separar: digital va arriba, físico va abajo
        if row['anio'] == 2022:
            offset = (0, 13) if tipo == 'digital' else (0, -20)
        else:
            offset = (0, 13)
        ax.annotate(
            f"{row['interacciones']:,.0f}",
            (row['anio'], row['interacciones']),
            textcoords="offset points", xytext=offset,
            ha='center', fontsize=10, color=color, fontweight='bold'
        )

ax.set_title('Figura 1. Evolución Histórica de la Demanda (2022–2025)',
             fontsize=13, fontweight='bold', color=PALETA['texto'], pad=14)
ax.set_xlabel('Año Académico', fontsize=11, color=PALETA['texto'])
ax.set_ylabel('Total de Interacciones', fontsize=11, color=PALETA['texto'])
ax.set_xticks([2022, 2023, 2024, 2025])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.legend(frameon=False, fontsize=10, loc='upper left')
ax.set_facecolor('white')
ax.tick_params(colors=PALETA['texto'])

plt.tight_layout()
plt.savefig('figura_1a_evolucion_anual.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura 1a guardada")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA 1B — Distribución Mensual Acumulada (2022–2025)
# ══════════════════════════════════════════════════════════════
df_mensual = df_unificado_clustering.groupby(
    ['mes', 'tipo_recurso']
).size().reset_index(name='interacciones')

meses_nombres = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
                 7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')

for tipo, color, label, marker in [
    ('digital', PALETA['digital'], 'Consultas Digitales', 'o'),
    ('fisico',  PALETA['fisico'],  'Préstamos Físicos',   's')
]:
    subset = df_mensual[df_mensual['tipo_recurso'] == tipo].sort_values('mes')
    ax.plot(
        [meses_nombres[int(m)] for m in subset['mes']],
        subset['interacciones'],
        marker=marker, color=color, linewidth=2.5,
        markersize=7, label=label, zorder=3
    )

ax.set_title('Figura 2. Distribución Mensual Acumulada (2022–2025)',
             fontsize=13, fontweight='bold', color=PALETA['texto'], pad=14)
ax.set_xlabel('Mes del Año', fontsize=11, color=PALETA['texto'])
ax.set_ylabel('Total de Interacciones', fontsize=11, color=PALETA['texto'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.legend(frameon=False, fontsize=10, loc='upper right')
ax.set_facecolor('white')
ax.tick_params(colors=PALETA['texto'])

plt.tight_layout()
plt.savefig('figura_1b_distribucion_mensual.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura 1b guardada")

In [ ]:
# Agregar los que faltaban al diccionario principal
adicionales = {
    'ingenieria electronica': 'ingenierias',
    'administracion del turismo sostenible': 'ciencias_empresariales',
    'tecnologia en regencia de farmacia': 'salud',
    'ciencias del deporte y recreacion': 'salud',
    'escuela de artes plasticas': 'bellas_artes',
    'movilidad academica': 'externo',
    'administracion industrial': 'ciencias_empresariales',
    'escuela de español y comunicacion audiov.': 'ciencias_educacion',
    'tecnologia en produccion agricola': 'ciencias_agrarias',
    'turismo sostenible': 'ciencias_empresariales',
    'vicerrectoria de investigaciones innovacion y extension': 'apoyo_administrativo',
    'estudiantes visitantes posgrados': 'externo',
    'laboratorio gastronomico sostenible (servicio academico)': 'apoyo_administrativo',
    'oficina de planeacion': 'apoyo_administrativo',
    'gf - gestion de tesoreria': 'apoyo_administrativo',
    'ingenieria civil': 'ingenierias',
    'tecnologia en desarrollo de software': 'tecnologia',
    'gsi - administracion del almacen general e inventarios': 'apoyo_administrativo',
    'gestion del talento humano': 'apoyo_administrativo',
    'departamento de dibujo': 'bellas_artes',
}
diccionario_tesis_facultades.update(adicionales)

# Aplicar el diccionario completo
df_unificado_clustering['facultad'] = df_unificado_clustering['facultad'].replace(diccionario_tesis_facultades)

# Corregir nulos
df_unificado_clustering['facultad'] = df_unificado_clustering['facultad'].replace(
    ['nan', 'Missing value', 'None'], np.nan
)

print("--- CONTEO DE NULOS ---")
print(df_unificado_clustering['facultad'].isna().sum())
print(f"\nFacultades únicas: {df_unificado_clustering['facultad'].nunique()}")
print(df_unificado_clustering['facultad'].value_counts())

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA 2 — Distribución digital vs físico por facultad
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

canal_fac = (df_unificado_clustering
             .groupby(['facultad', 'tipo_recurso'])
             .size()
             .reset_index(name='registros'))

canal_fac['pct'] = (canal_fac.groupby('facultad')['registros']
                   .transform(lambda x: x / x.sum() * 100))

pivot = canal_fac.pivot_table(index='facultad', 
                               columns='tipo_recurso', 
                               values='pct', 
                               fill_value=0).reset_index()

pivot = pivot.sort_values('digital', ascending=True)

nombres = {
    'salud': 'Salud',
    'ingenierias': 'Ingenierías',
    'ciencias_educacion': 'Ciencias de la Educación',
    'tecnologia': 'Tecnología',
    'apoyo_administrativo': 'Apoyo Administrativo',
    'bellas_artes': 'Bellas Artes',
    'ciencias_empresariales': 'Ciencias Empresariales',
    'fac_desconocida': 'Facultad Desconocida',
    'ciencias_agrarias': 'Ciencias Agrarias',
    'mecanica': 'Mecánica',
    'ciencias_ambientales': 'Ciencias Ambientales',
    'externo': 'Externo',
    'ciencias_basicas': 'Ciencias Básicas',
    'biblioteca': 'Biblioteca',
    'rectoria': 'Rectoría',
    'univirtual': 'Univirtual',
    'ilex': 'ILEX',
}
pivot['facultad_label'] = pivot['facultad'].map(nombres).fillna(pivot['facultad'])

fig, ax = plt.subplots(figsize=(11, 8), facecolor='white')

ax.barh(pivot['facultad_label'], pivot['digital'],
        color=PALETA['digital'], height=0.6)
ax.barh(pivot['facultad_label'], pivot.get('fisico', 0),
        left=pivot['digital'],
        color=PALETA['fisico'], height=0.6)

# Etiquetas dentro de las barras
for i, (_, row) in enumerate(pivot.iterrows()):
    if row['digital'] > 8:
        ax.text(row['digital'] / 2, i,
                f"{row['digital']:.1f}%",
                ha='center', va='center',
                fontsize=8.5, color='white', fontweight='bold')
    fisico_val = row.get('fisico', 0)
    if fisico_val > 8:
        ax.text(row['digital'] + fisico_val / 2, i,
                f"{fisico_val:.1f}%",
                ha='center', va='center',
                fontsize=8.5, color='white', fontweight='bold')

# Leyenda manual fuera del área de la gráfica
patch_digital = mpatches.Patch(color=PALETA['digital'], label='Digital')
patch_fisico  = mpatches.Patch(color=PALETA['fisico'],  label='Físico')
ax.legend(handles=[patch_digital, patch_fisico],
          frameon=False, fontsize=10,
          loc='lower right',
          bbox_to_anchor=(1.0, -0.08),
          ncol=2)

ax.set_title('Figura 2. Distribución Digital vs. Físico por Facultad',
             fontsize=13, fontweight='bold', color=PALETA['texto'], pad=14)
ax.set_xlabel('Porcentaje de Interacciones (%)', fontsize=11, color=PALETA['texto'])
ax.set_xlim(0, 100)
ax.axvline(x=50, color=PALETA['grid'], linestyle='--', linewidth=1, alpha=0.7)
ax.set_facecolor('white')
ax.tick_params(colors=PALETA['texto'])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.tight_layout()
plt.savefig('figura_2_digital_fisico_facultad.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura 2 guardada")

In [ ]:
df_unificado_clustering.sample(20)

Se aplicó un proceso de reducción de dimensionalidad manual, eliminando variables descriptivas de inventario (como autores, títulos y signaturas topográficas) que no aportan valor predictivo al modelo de comportamiento. Al estandarizar los esquemas de ambas fuentes, garantizamos que el algoritmo de clustering se enfoque exclusivamente en las métricas de frecuencia, intensidad y modalidad de acceso

Vamos a hacer las limpiezas necesarias para el modelo del clustering

1. Facultad.
2. Carrera.
3. Nivel Académico.

1. FACULTADES

Creamos un diccionario para aplicar toda la información de ambos maestros a estos para estandarizar.

In [ ]:
# 3. Diccionario extendido basado en tus capturas (Agrupación de "Vivienda", "Gastos", etc.)
diccionario_tesis_facultades = {
    # ÁREAS ACADÉMICAS
    'facultad de ingenierias': 'ingenierias',
    'facultad de ingenieria mecanica': 'mecanica',
    'facultad de tecnologias': 'tecnologia',
    'facultad de ciencias de la salud': 'salud',
    'facultad de bellas artes y humanidades': 'bellas_artes',
    'facultad de ciencias empresariales' : 'ciencias_empresariales',
    'facultad de ciencias de la educacion' : 'ciencias_educacion',
    'facultad de ciencias agrarias y agroindustria' : 'ciencias_agrarias',
    'facultad de ciencias ambientales' : 'ciencias_ambientales',
    'facultad de ingenierias electrica, electronica, fisica y ciencias de la computacion' : 'ingenierias',
    'licenciatura en musica': 'bellas_artes',
    'ingenieria industrial': 'ciencias_empresariales',
    'administracion de empresas': 'ciencias_empresariales',
    'escuela de musica': 'bellas_artes',
    'escuela de filosofia' : 'bellas_artes',
    'departamento de ciencias clinicas' : 'salud',
    'escuela tecnologia industrial' : 'tecnologia',
    'facultad ingenieria fisica, electrica, electronica y ciencias de la computacion' : 'ingenierias',
    'licenciatura en bilinguismo con enfasis en ingles':'bellas_artes',
    'departamento de humanidades e idiomas':'bellas_artes',
    'escuela de español y comunicacion a.v.':'ciencias_educacion',
    'administracion ambiental': 'ciencias_ambientales',
    'departamento de ciencias basicas ambientales':'ciencias_ambientales',
    'escuela de ciencias sociales':'ciencias_educacion',
    'doctorado en ciencias ambientales':'ciencias_ambientales',
    'departamento de psicopedagogia':'ciencias_educacion',
    'tecnologia en atencion prehospitalaria' : 'salud',
    'departamento de medicina comunitaria' : 'salud',
    'departamento de ciencias basicas de medicina' : 'salud',
    'medicina veterinaria' : 'salud',
    'medicina veterinaria y zootecnia' : 'salud',
    'laboratorio de genetica medica' : 'salud',
    'escuela de deporte y recreacion' : 'salud',
    'ingenieria mecanica':'mecanica',
    'departamento de fisica': 'ingenierias',
    'ingenieria de sistemas y computacion': 'ingenierias',
    'ingenieria electrica': 'ingenierias',
    'ingenieria de sistemas y computacion': 'ingenierias',
    'ingenieria fisica': 'ingenierias',
    'escuela tecnologia electrica':'tecnologia',
    'tecnologia electrica':'tecnologia',
    'tecnologia quimica':'tecnologia',
    'escuela de tecnologia quimica':'tecnologia',
    'facultad de tecnologia':'tecnologia',
    'tecnologia en produccion horticola': 'ciencias_agrarias',
    'ingenieria en procesos sostenibles de las maderas': 'ciencias_agrarias',
    'ingenieria en procesos agroindustriales': 'ciencias_agrarias',
    'departamento de ciencias basicas de la medicina' : 'salud',
    'facultad de mecanica aplicada': 'mecanica',
    'laboratorio financiero (middle office en el proceso de gestion de inversiones y riesgos financieros del modelo tipo endowment de la utp)': 'ciencias_empresariales',
    'mecatronica': 'tecnologia',
    'ingenieria de manufactura': 'tecnologia',
    'facultad de ciencias basicas': 'ciencias_basicas',
    'departamento de matematicas': 'ciencias_basicas',
    'escuela tecnologia mecanica': 'mecanica',
    
    
    
    
    # ÁREAS ADMINISTRATIVAS / APOYO (Aquí agrupamos lo que mencionas)
    'gastos generales': 'apoyo_administrativo',
    'administrativo': 'apoyo_administrativo',
    'fondo de ahorro y vivienda': 'apoyo_administrativo',
    'vivienda': 'apoyo_administrativo',
    'vicerrectoria administrativa y financiera': 'apoyo_administrativo',
    'vicerrectoria administrativa': 'apoyo_administrativo',
    'recursos humanos': 'apoyo_administrativo',
    'gestion de la calidad': 'apoyo_administrativo',
    'laboratorio de gestion': 'apoyo_administrativo',
    'bienestar universitario': 'apoyo_administrativo',
    'vicerrectoria de responsabilidad social y bienestar universitario':'apoyo_administrativo',
    'vicerectoria administrativa': 'apoyo_administrativo',
    'vicerrect. respons. social y bienestar universitar': 'apoyo_administrativo',
    'vicerrectoria academica': 'apoyo_administrativo',
    'vicerrectoria de investigaciones, innovacion y extension': 'apoyo_administrativo',
    'juridica': 'apoyo_administrativo',
    'gestion de tecnologias informaticas y sistemas de informacion': 'apoyo_administrativo',
    'biblioteca e informacion cientifica':'biblioteca',
   
    
    # OTROS
    'anonimo': 'externo',
    'sin departamento': 'externo',
    'jubilado': 'externo',
    'usuario lector' : 'externo',
    'centro de biblioteca' : 'apoyo_administrativo',
    'direccion general de posgrados' : 'apoyo_administrativo',
    'gestion financiera' : 'apoyo_administrativo',
    'gastos generales jornadas especiales': 'apoyo_administrativo',
    'recursos informaticos y educativos': 'apoyo_administrativo',
    'administracion u.t.p.': 'apoyo_administrativo',
    'programa solo para estudiantes visitantes': 'apoyo_administrativo',
    'univirtual': 'univirtual',
    'instituto de lenguas extranjeras - ilex': 'ilex',
    'facultad': 'apoyo_administrativo',
    'laboratorio de analisis de aguas y alimentos': 'apoyo_administrativo',
    'programas de pregrado': 'apoyo_administrativo',
    'usuario externo': 'externo',
}

In [ ]:
# Agregar las que faltan al diccionario existente
adicionales = {
    'ingenieria electronica': 'ingenierias',
    'administracion del turismo sostenible': 'ciencias_empresariales',
    'tecnologia en regencia de farmacia': 'salud',
    'ciencias del deporte y recreacion': 'salud',
    'escuela de artes plasticas': 'bellas_artes',
    'movilidad academica': 'externo',
    'administracion industrial': 'ciencias_empresariales',
    'escuela de español y comunicacion audiov.': 'ciencias_educacion',
    'tecnologia en produccion agricola': 'ciencias_agrarias',
    'turismo sostenible': 'ciencias_empresariales',
    'vicerrectoria de investigaciones innovacion y extension': 'apoyo_administrativo',
    'estudiantes visitantes posgrados': 'externo',
    'laboratorio gastronomico sostenible (servicio academico)': 'apoyo_administrativo',
    'oficina de planeacion': 'apoyo_administrativo',
    'gf - gestion de tesoreria': 'apoyo_administrativo',
    'ingenieria civil': 'ingenierias',
    'tecnologia en desarrollo de software': 'tecnologia',
    'gsi - administracion del almacen general e inventarios': 'apoyo_administrativo',
    'gestion del talento humano': 'apoyo_administrativo',
    'departamento de dibujo': 'bellas_artes',
}

diccionario_tesis_facultades.update(adicionales)

# Volver a aplicar
df_unificado_clustering['facultad'] = df_unificado_clustering['facultad'].map(
    diccionario_tesis_facultades
).fillna(df_unificado_clustering['facultad'])

print(df_unificado_clustering['facultad'].value_counts())
print(f"\nFacultades únicas: {df_unificado_clustering['facultad'].nunique()}")

Aplicamos el diccionario a ambas bases de datos

In [ ]:
# Agregar los que faltaban al diccionario principal
adicionales = {
    'ingenieria electronica': 'ingenierias',
    'administracion del turismo sostenible': 'ciencias_empresariales',
    'tecnologia en regencia de farmacia': 'salud',
    'ciencias del deporte y recreacion': 'salud',
    'escuela de artes plasticas': 'bellas_artes',
    'movilidad academica': 'externo',
    'administracion industrial': 'ciencias_empresariales',
    'escuela de español y comunicacion audiov.': 'ciencias_educacion',
    'tecnologia en produccion agricola': 'ciencias_agrarias',
    'turismo sostenible': 'ciencias_empresariales',
    'vicerrectoria de investigaciones innovacion y extension': 'apoyo_administrativo',
    'estudiantes visitantes posgrados': 'externo',
    'laboratorio gastronomico sostenible (servicio academico)': 'apoyo_administrativo',
    'oficina de planeacion': 'apoyo_administrativo',
    'gf - gestion de tesoreria': 'apoyo_administrativo',
    'ingenieria civil': 'ingenierias',
    'tecnologia en desarrollo de software': 'tecnologia',
    'gsi - administracion del almacen general e inventarios': 'apoyo_administrativo',
    'gestion del talento humano': 'apoyo_administrativo',
    'departamento de dibujo': 'bellas_artes',
}
diccionario_tesis_facultades.update(adicionales)

# Aplicar el diccionario completo
df_unificado_clustering['facultad'] = df_unificado_clustering['facultad'].replace(diccionario_tesis_facultades)

# Corregir nulos
df_unificado_clustering['facultad'] = df_unificado_clustering['facultad'].replace(
    ['nan', 'Missing value', 'None'], np.nan
)

print("--- CONTEO DE NULOS ---")
print(df_unificado_clustering['facultad'].isna().sum())
print(f"\nFacultades únicas: {df_unificado_clustering['facultad'].nunique()}")
print(df_unificado_clustering['facultad'].value_counts())

In [ ]:
df_unificado_clustering['facultad'].value_counts()

In [ ]:
df_unificado_clustering['facultad'].isnull().sum()

In [ ]:
df_unificado_clustering.sample(20)

In [ ]:
# 1. Convertimos los nulos restantes en una categoría clara
df_unificado_clustering['facultad'] = df_unificado_clustering['facultad'].fillna('fac_desconocida')

# 2. Verificamos que ya no existan nulos reales
print(f"Nulos finales en Facultad: {df_unificado_clustering['facultad'].isna().sum()}")

# 3. Miramos cómo quedó la distribución para tu tesis
print("\n--- DISTRIBUCIÓN FINAL POR FACULTAD ---")
print(df_unificado_clustering['facultad'].value_counts())

2. CARRERAS

In [ ]:
# Sacamos las carreras, convertimos todo a texto y quitamos los valores nulos (NaN)
lista_carreras = df_unificado_clustering['carrera'].dropna().unique().astype(str)

# Ahora sí podemos ordenar alfabéticamente
lista_carreras.sort()

# La imprimimos con un contador para que sea fácil de leer
print(f"Se encontraron {len(lista_carreras)} carreras distintas:\n")
for i, carrera in enumerate(lista_carreras):
    print(f"{i}. {carrera}")

In [ ]:
import re

def limpieza_basica_carrera(texto):
    if pd.isna(texto): return "No Identificado"
    texto = str(texto).upper().strip()
    # 1. Quitamos los códigos numéricos del inicio (ej: "01-", "14 ")
    texto = re.sub(r'^\d+[\s-]*', '', texto)
    # 2. Quitamos espacios múltiples
    texto = re.sub(r'\s+', ' ', texto)
    return texto

# Creamos la columna organizada
df_unificado_clustering['carrera_limpia'] = df_unificado_clustering['carrera'].apply(limpieza_basica_carrera)

# Veamos cuántas quedaron ahora
print(f"Carreras después de quitar códigos: {df_unificado_clustering['carrera_limpia'].nunique()}")

Pasaste de 428 nombres a 311. Eso significa que eliminaste 117 variantes que solo eran diferentes por tener un número al principio (como el "01-" o "14-").

In [ ]:
# Mejoramos la función para quitar los signos de interrogación y símbolos raros
def limpieza_profunda_caracteres(texto):
    import re
    if pd.isna(texto): return "No Identificado"
    texto = str(texto).upper()
    
    # Quitamos cualquier cosa que no sea letra (A-Z), número (0-9) o espacio
    # Esto borrará los '?', las tildes rotas y símbolos extraños
    texto = re.sub(r'[^A-Z0-9\s]', '', texto)
    
    # Quitamos espacios dobles que hayan quedado tras borrar los símbolos
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# Aplicamos esta limpieza sobre lo que ya tenías
df_unificado_clustering['carrera_limpia'] = df_unificado_clustering['carrera_limpia'].apply(limpieza_profunda_caracteres)

# Verificamos si bajó un poco más el número de carreras únicas
print(f"Carreras tras quitar símbolos: {df_unificado_clustering['carrera_limpia'].nunique()}")

In [ ]:
# Ver las primeras filas del dataset unificado con la columna limpia
df_unificado_clustering[['carrera', 'carrera_limpia']].sample(15)

In [ ]:
def gran_limpieza_carreras_utp(valor):
    # --- 0. MANEJO DE NULOS (DESCONOCIDOS) ---
    # Si el valor es nulo (NaN), devolvemos 'DESCONOCIDO'
    if pd.isna(valor):
        return 'DESCONOCIDO'
    
    nombre = str(valor).upper().strip()
    
    # --- 1. FILTRO DE CARGOS Y NO-ESTUDIANTES ---
    # Incluimos 'ANONIMO' aquí para que también sea 'DESCONOCIDO' o 'PERSONAL ADM'
    # Si prefieres que ANONIMO sea DESCONOCIDO, muévelo a la sección 0.
    cargos = ['GRADO', 'DIRECTOR', 'ADMINISTRATIVO', 'CATEDRATICO', 'PLANTA', 'ANONIMO', 
              'ASISTENCIAL', 'PROFESIONAL ND', 'TECNICO ND', 'VISITANTES', 'ORDEN DE SERVICIO']
    if any(c in nombre for c in cargos):
        return 'PERSONAL ADM / OTROS'

    # --- 2. MAESTRÍAS ---
    if 'MAESTRA' in nombre or 'MAGISTER' in nombre:
        if any(x in nombre for x in ['SISTEMAS', 'ELECTRONICA', 'ELECTRICA', 'MECANICA', 'INSTRUMENTACION', 'COMPUTACION', 'AUTOMATICOS', 'SOFTWARE']):
            return 'MAESTRIA EN INGENIERIA Y TECH'
        if any(x in nombre for x in ['EDUCACION', 'DIDACTICA', 'ENSENANZA', 'PEDAGOGIA', 'INFANCIA', 'BILINGUE']):
            return 'MAESTRIA EN EDUCACION'
        if any(x in nombre for x in ['ADMINISTRACION', 'GERENCIA', 'ECONOMICA', 'FINANCIERA', 'DESARROLLO HUMANO', 'AGRONEGOCIOS', 'SUMINISTRO', 'PROYECTOS', 'CALIDAD']):
            return 'MAESTRIA EN ADMINISTRACION Y GESTION'
        if any(x in nombre for x in ['LITERATURA', 'LINGUISTICA', 'HISTORIA', 'FILOSOFIA', 'ESTETICA', 'CULTURALES', 'NARRATIVAS']):
            return 'MAESTRIA EN HUMANIDADES Y CIENCIAS SOC.'
        if any(x in nombre for x in ['BIOLOGIA', 'QUIMICA', 'MATEMATICA', 'FISICA', 'BIOTECNOLOGIA', 'AMBIENTALES', 'ECOTECNOLOGIA', 'AGROINDUSTRIAL']):
            return 'MAESTRIA EN CIENCIAS EXACTAS Y NAT.'
        return 'MAESTRIA - OTRAS'

    # --- 3. DOCTORADOS ---
    if 'DOCTORADO' in nombre:
        if 'INGENIERIA' in nombre: return 'DOCTORADO EN INGENIERIA'
        if 'EDUCACION' in nombre or 'DIDACTICA' in nombre: return 'DOCTORADO EN EDUCACION'
        return 'DOCTORADO - OTROS'

    # --- 4. ESPECIALIZACIONES ---
    if 'ESPECIALIZA' in nombre:
        if any(x in nombre for x in ['MEDICINA', 'PSIQUIATRIA', 'PEDIATRIA', 'PEDIATR', 'GINECOLOGIA', 'INTERNA', 'RADIOLOGIA', 'SALUD']):
            return 'ESPECIALIZACION EN SALUD'
        return 'ESPECIALIZACION - OTRAS'

    # --- 5. TECNOLOGÍAS ---
    if 'TECNOLOGIA' in nombre or 'TECNOLOGA' in nombre:
        if 'MECANICA' in nombre or 'MECNICA' in nombre or 'FABRICACION' in nombre: return 'TECNOLOGIA MECANICA'
        if 'ELECTRICA' in nombre or 'ELCTRICA' in nombre: return 'TECNOLOGIA ELECTRICA'
        if 'INDUSTRIAL' in nombre: return 'TECNOLOGIA INDUSTRIAL'
        if 'QUIMICA' in nombre or 'QUMICA' in nombre: return 'TECNOLOGIA QUIMICA'
        if 'SOFTWARE' in nombre: return 'TECNOLOGIA EN SOFTWARE'
        if 'TURISMO' in nombre: return 'TECNOLOGIA EN TURISMO'
        if 'PREHOSPITALARIA' in nombre or 'FARMACIA' in nombre: return 'TECNOLOGIA EN SALUD'
        return 'OTRA TECNOLOGIA'

    # --- 6. INGENIERÍAS ---
    if 'INGENIER' in nombre or 'ING ' in nombre:
        if 'MECANICA' in nombre or 'MECNICA' in nombre: return 'INGENIERIA MECANICA'
        if 'SISTEMAS' in nombre or 'COMPUTACION' in nombre: return 'INGENIERIA DE SISTEMAS'
        if 'INDUSTRIAL' in nombre: return 'INGENIERIA INDUSTRIAL'
        if 'ELECTRICA' in nombre or 'ELCTRICA' in nombre: return 'INGENIERIA ELECTRICA'
        if 'ELECTRONICA' in nombre or 'ELECTRNICA' in nombre: return 'INGENIERIA ELECTRONICA'
        if 'MECATRONICA' in nombre or 'MECATRÓNICA' in nombre: return 'INGENIERIA MECATRONICA'
        if 'FISICA' in nombre or 'FíSICA' in nombre: return 'INGENIERIA FISICA'
        if 'CIVIL' in nombre: return 'INGENIERIA CIVIL'
        return 'OTRA INGENIERIA'

    # --- 7. LICENCIATURAS ---
    if 'LICENCIATURA' in nombre or 'LIC ' in nombre or 'LICEN' in nombre:
        if 'MUSICA' in nombre or 'MÊSICA' in nombre: return 'LICENCIATURA EN MUSICA'
        if 'ARTES' in nombre: return 'LICENCIATURA EN ARTES'
        if 'BILING' in nombre: return 'LICENCIATURA EN BILINGUISMO'
        if 'PEDAGOGIA' in nombre or 'INFANTIL' in nombre or 'EDUCACION' in nombre or 'EDUCACIN' in nombre: return 'LICENCIATURA EN EDUCACION'
        if 'MATEMATICA' in nombre or 'FISICA' in nombre: return 'LICENCIATURA EN CIENCIAS'
        if 'ESPANOL' in nombre or 'LITERATURA' in nombre or 'CASTELLANA' in nombre: return 'LICENCIATURA EN LENGUAS'
        return 'OTRA LICENCIATURA'

    # --- 8. ADMINISTRACIÓN Y SALUD ---
    if 'ADMINISTRACION' in nombre or 'ADMINISTRACIN' in nombre:
        if 'TURISMO' in nombre: return 'ADMINISTRACION TURISMO'
        if 'AMBIENTAL' in nombre or 'MEDIO AMBIENTE' in nombre: return 'ADMINISTRACION AMBIENTAL'
        return 'ADMINISTRACION INDUSTRIAL'
    
    if 'MEDICINA' in nombre:
        if 'VETERINARIA' in nombre: return 'MEDICINA VETERINARIA'
        return 'MEDICINA'

    return 'OTRO / POR REVISAR'

# Aplicamos la función definitiva usando la columna original 'carrera'
# Importante: Asegúrate de tener importado pandas como 'pd' o cambia 'pandas' por 'pd' arriba
import pandas as pd
df_unificado_clustering['carrera_final'] = df_unificado_clustering['carrera'].apply(gran_limpieza_carreras_utp)

# Verificamos resultados
print(f"CATEGORÍAS FINALES: {df_unificado_clustering['carrera_final'].nunique()}")
print(df_unificado_clustering['carrera_final'].value_counts())

In [ ]:
df_unificado_clustering['carrera_final'].unique()

In [ ]:
df_unificado_clustering['carrera_final'].value_counts()

In [ ]:
df_unificado_clustering.columns.tolist()

Eliminamos las dos variables que usamos para transformar carrera y dejamos solo la que vamos a usar "carrera_final"

In [ ]:
# Eliminamos las versiones "sucias" e intermedias de la carrera
cols_a_eliminar = ['carrera', 'carrera_limpia']

# Usamos errors='ignore' por si alguna ya la habías borrado antes
df_unificado_clustering.drop(columns=cols_a_eliminar, inplace=True, errors='ignore')

print("--- LIMPIEZA DE COLUMNAS FINALIZADA ---")
print(f"Columnas resultantes: {df_unificado_clustering.columns.tolist()}")

3. NIVEL ACADEMICO

Esta nueva columna llamada nivel_academico está basada en las etiquetas que ya se limpiaron en carrera_final. Esto nos permitirá segmentar el comportamiento de un estudiante de pregrado frente al de un doctorado.

In [ ]:
def asignar_nivel_academico(carrera):
    c = str(carrera).upper()
    
    if 'DOCTORADO' in c:
        return 'POSGRADO - DOCTORADO'
    if 'MAESTRIA' in c:
        return 'POSGRADO - MAESTRIA'
    if 'ESPECIALIZACION' in c or 'ESP.' in c:
        return 'POSGRADO - ESPECIALIZACION'
    if any(x in c for x in ['INGENIERIA', 'LICENCIATURA', 'ADMINISTRACION', 'MEDICINA', 'TECNOLOGIA', 'QUIMICA', 'FISICA']):
        return 'PREGRADO'
    if 'DESCONOCIDO' in c:
        return 'DESCONOCIDO'
    
    return 'OTRO / NO APLICA'

# Aplicamos la función
df_unificado_clustering['nivel_academico_final'] = df_unificado_clustering['carrera_final'].apply(asignar_nivel_academico)

# Verificamos cómo quedó la distribución
print("--- DISTRIBUCIÓN POR NIVEL ACADÉMICO ---")
conteo_nivel = df_unificado_clustering['nivel_academico_final'].value_counts()
print(conteo_nivel)

# Porcentaje para tu análisis de tesis
print("\n--- PORCENTAJE POR NIVEL ---")
print(df_unificado_clustering['nivel_academico_final'].value_counts(normalize=True) * 100)

In [ ]:
df_unificado_clustering.sample(10)

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA 3 — Clasificación de usuarios por nivel de recurrencia
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# Calcular total de interacciones por usuario y tipo
recurrencia = (df_unificado_clustering
               .groupby(['codigo', 'tipo_recurso'])
               .size()
               .reset_index(name='total_interacciones'))

# Clasificar en categorías
def clasificar(n):
    if n <= 3:   return '1. Esporádico\n(1–3)'
    elif n <= 10: return '2. Ocasional\n(4–10)'
    elif n <= 30: return '3. Frecuente\n(11–30)'
    else:         return '4. Habitual\n(+30)'

recurrencia['categoria'] = recurrencia['total_interacciones'].apply(clasificar)

# Contar usuarios por categoría y tipo
conteo = (recurrencia
          .groupby(['categoria', 'tipo_recurso'])['codigo']
          .nunique()
          .reset_index(name='usuarios'))

# Pivotar
pivot = conteo.pivot_table(index='categoria', 
                            columns='tipo_recurso', 
                            values='usuarios', 
                            fill_value=0).reset_index()

categorias = ['1. Esporádico\n(1–3)', '2. Ocasional\n(4–10)', 
              '3. Frecuente\n(11–30)', '4. Habitual\n(+30)']
pivot = pivot.set_index('categoria').reindex(categorias).reset_index()

x      = np.arange(len(categorias))
width  = 0.35

fig, ax = plt.subplots(figsize=(11, 6), facecolor='white')

bars_d = ax.bar(x - width/2, pivot.get('digital', 0),
                width, color=PALETA['digital'], label='Digital')
bars_f = ax.bar(x + width/2, pivot.get('fisico', 0),
                width, color=PALETA['fisico'], label='Físico')

# Etiquetas encima de cada barra
for bar in bars_d:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 80,
                f'{int(h):,}', ha='center', va='bottom',
                fontsize=9.5, color=PALETA['digital'], fontweight='bold')

for bar in bars_f:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 80,
                f'{int(h):,}', ha='center', va='bottom',
                fontsize=9.5, color=PALETA['fisico'], fontweight='bold')

ax.set_title('Figura 4. Clasificación de Usuarios por Nivel de Recurrencia según Modalidad',
             fontsize=13, fontweight='bold', color=PALETA['texto'], pad=14)
ax.set_xlabel('Nivel de Recurrencia', fontsize=11, color=PALETA['texto'])
ax.set_ylabel('Número de Usuarios', fontsize=11, color=PALETA['texto'])
ax.set_xticks(x)
ax.set_xticklabels(categorias, fontsize=10, color=PALETA['texto'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_facecolor('white')
ax.tick_params(colors=PALETA['texto'])

patch_d = mpatches.Patch(color=PALETA['digital'], label='Digital')
patch_f = mpatches.Patch(color=PALETA['fisico'],  label='Físico')
ax.legend(handles=[patch_d, patch_f], frameon=False, fontsize=10,
          loc='upper right')

plt.tight_layout()
plt.savefig('figura_3_recurrencia.png', dpi=300, 
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura 3 guardada")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA 3B — Clasificación por recurrencia PROMEDIO ANUAL
# ══════════════════════════════════════════════════════════════

# Calcular interacciones por usuario, tipo y año
recurrencia_anual = (df_unificado_clustering
                     .groupby(['codigo', 'tipo_recurso', 'anio'])
                     .size()
                     .reset_index(name='interacciones_anio'))

# Promedio anual por usuario y tipo
promedio_anual = (recurrencia_anual
                  .groupby(['codigo', 'tipo_recurso'])['interacciones_anio']
                  .mean()
                  .reset_index(name='promedio_interacciones'))

# Clasificar
def clasificar(n):
    if n <= 3:    return '1. Esporádico\n(1–3)'
    elif n <= 10: return '2. Ocasional\n(4–10)'
    elif n <= 30: return '3. Frecuente\n(11–30)'
    else:         return '4. Habitual\n(+30)'

promedio_anual['categoria'] = promedio_anual['promedio_interacciones'].apply(clasificar)

# Contar usuarios por categoría y tipo
conteo = (promedio_anual
          .groupby(['categoria', 'tipo_recurso'])['codigo']
          .nunique()
          .reset_index(name='usuarios'))

pivot = conteo.pivot_table(index='categoria',
                            columns='tipo_recurso',
                            values='usuarios',
                            fill_value=0).reset_index()

categorias = ['1. Esporádico\n(1–3)', '2. Ocasional\n(4–10)',
              '3. Frecuente\n(11–30)', '4. Habitual\n(+30)']
pivot = pivot.set_index('categoria').reindex(categorias).reset_index()

x     = np.arange(len(categorias))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6), facecolor='white')

bars_d = ax.bar(x - width/2, pivot.get('digital', 0),
                width, color=PALETA['digital'], label='Digital')
bars_f = ax.bar(x + width/2, pivot.get('fisico', 0),
                width, color=PALETA['fisico'], label='Físico')

for bar in bars_d:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 50,
                f'{int(h):,}', ha='center', va='bottom',
                fontsize=9.5, color=PALETA['digital'], fontweight='bold')

for bar in bars_f:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 50,
                f'{int(h):,}', ha='center', va='bottom',
                fontsize=9.5, color=PALETA['fisico'], fontweight='bold')

ax.set_title('Figura 5. Clasificación de Usuarios por Nivel de Recurrencia\nPromedio Anual según Modalidad',
             fontsize=13, fontweight='bold', color=PALETA['texto'], pad=14)
ax.set_xlabel('Nivel de Recurrencia', fontsize=11, color=PALETA['texto'])
ax.set_ylabel('Número de Usuarios', fontsize=11, color=PALETA['texto'])
ax.set_xticks(x)
ax.set_xticklabels(categorias, fontsize=10, color=PALETA['texto'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_facecolor('white')
ax.tick_params(colors=PALETA['texto'])

patch_d = mpatches.Patch(color=PALETA['digital'], label='Digital')
patch_f = mpatches.Patch(color=PALETA['fisico'],  label='Físico')
ax.legend(handles=[patch_d, patch_f], frameon=False, fontsize=10,
          loc='upper right')

plt.tight_layout()
plt.savefig('figura_3b_recurrencia_promedio.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura 3b guardada")

In [ ]:
df_unificado_clustering.drop(columns=['nivel_academico'], inplace=True)

In [ ]:
df_unificado_clustering.sample(10)

In [ ]:
df_unificado_clustering.columns.tolist()

In [ ]:
# Obtenemos los conjuntos de códigos únicos de cada tabla
codigos_fisicos = set(prestamos_maestro['codigo'])
codigos_digitales = set(consultas_maestro['codigo'])

# Encontramos la intersección (los que están en AMBOS)
codigos_comunes = codigos_fisicos.intersection(codigos_digitales)

print(f"Usuarios en Préstamos: {len(codigos_fisicos)}")
print(f"Usuarios en Consultas: {len(codigos_digitales)}")
print(f"Usuarios Híbridos (en ambos): {len(codigos_comunes)}")

Inicialmente, los datos presentaban una inconsistencia de formato que ocultaba la realidad del comportamiento del usuario. Tras aplicar técnicas de normalización de identificadores, logramos descubrir que 4,618 usuarios (aproximadamente el 51% de quienes usan la biblioteca física) interactúan de forma omnicanal. Este hallazgo valida la necesidad de un modelo de clustering que no solo separe lo físico de lo digital, sino que analice la profundidad de esta hibridación"

In [ ]:
print(f"Total de registros en la tabla unificada: {len(df_unificado_clustering)}")
print("-" * 30)
print("Distribución por tipo de recurso:")
print(df_unificado_clustering['tipo_recurso'].value_counts())

In [ ]:
df_unificado_clustering.sample()

#### TRANSFORMACIONES DE LAS VARIABLES DEL UNIFICADO

VARIABLE: codigo y facultad -> **id_perfil**
- Se realiza una concatenación del codigo del estudiante con la facultad

* Justificación:

Contextualización del Hábito: El comportamiento del usuario no es solo personal, es curricular. Un mismo estudiante actúa distinto si estudia Medicina o si estudia Artes; el ID de perfil captura esa diferencia.

Independencia de Trayectorias: Permite tratar de forma separada los registros de un usuario que cambió de carrera, evitando que hábitos antiguos sesguen el perfil actual.

Llave de Agregación: Es la llave primaria sintética necesaria para reducir 1.5 millones de transacciones a una matriz de perfiles únicos lista para el clustering.

In [ ]:
# 1. Creamos el ID de Perfil combinando Código y Facultad
df_unificado_clustering['id_perfil'] = (
    df_unificado_clustering['codigo'].astype(str) + 
    "_" + 
    df_unificado_clustering['facultad'].astype(str)
)

# 2. Verificamos cómo queód
print("Muestra de los nuevos IDs de Perfil:")
print(df_unificado_clustering[['codigo', 'facultad', 'id_perfil']].head())

# 3. ¿Cuántos perfiles únicos logramos identificar?
n_perfiles = df_unificado_clustering['id_perfil'].nunique()
print(f"\nTotal de perfiles académicos únicos: {n_perfiles}")

Ya no tengo 1.5 millones de filas anónimas, hay 18,944 carpetas de comportamiento listas.

Ahora cada acción está amarrada a su facultad. Si ese usuario llega a aparecer en "ciencias" más adelante, será un perfil distinto y no arruinará los promedios.

VARIABLE: tipo_recuso -> **clasificacion_recurso**
- 1: consulta en base de datos.
- 0: préstamo físico.

In [ ]:
# Creamos la columna 'clasificacion_recurso' (1 si es digital, 0 si es fisico)
df_unificado_clustering['clasificacion_recurso'] = df_unificado_clustering['tipo_recurso'].map({'digital': 1, 'fisico': 0})

# Verificamos rápidamente los primeros registros
print(df_unificado_clustering[['tipo_recurso', 'clasificacion_recurso']].sample(10))

VARIABLE: hora a **hora_sin** y **hora_cos**

Para que el modelo no se confunda con las horas (pensando que las 23h y las 00h están lejos), vamos a aplicar la transformación cíclica.

Nombre: Cyclic Encoding: Sine and Cosine Transformations for Periodic Features

In [ ]:
# Transformación cíclica de la hora
df_unificado_clustering['hora_sin'] = np.sin(2 * np.pi * df_unificado_clustering['hora'] / 24)
df_unificado_clustering['hora_cos'] = np.cos(2 * np.pi * df_unificado_clustering['hora'] / 24)

# Ya no usaremos la columna 'hora' original para el modelo, sino estas dos nuevas

In [ ]:
df_unificado_clustering.sample(10)

In [ ]:
df_unificado_clustering.shape

In [ ]:
import matplotlib.pyplot as plt

# Tomamos una muestra para no saturar el gráfico
sample = df_unificado_clustering.sample(1000)

plt.figure(figsize=(6,6))
plt.scatter(sample['hora_cos'], sample['hora_sin'], c=sample['hora'], cmap='twilight')
plt.title("Reloj de Actividad de la Biblioteca (Transformación Cíclica)")
plt.xlabel("Hora Coseno")
plt.ylabel("Hora Seno")
plt.grid(True)
plt.show()

VARIABLE: mes a **mes_sin** y **mes_cos**

Si no transformamos el mes, la inteligencia artificial pensará que diciembre (12) y enero (1) están a 11 meses de distancia, cuando en realidad son meses seguidos. Al aplicar el Seno y Coseno, los unimos en el ciclo anual tal cual como se hizo con las horas.

In [ ]:
import numpy as np

# Transformamos el MES (Ciclo de 12 meses)
df_unificado_clustering['mes_sin'] = np.sin(2 * np.pi * df_unificado_clustering['mes'] / 12)
df_unificado_clustering['mes_cos'] = np.cos(2 * np.pi * df_unificado_clustering['mes'] / 12)

# Verificamos los nuevos valores
print("Transformación del Mes lista:")
print(df_unificado_clustering[['mes', 'mes_sin', 'mes_cos']].drop_duplicates().sort_values('mes'))

Complementando la ingeniería de variables temporales, se normalizó la variable estacional (mes) mediante proyecciones trigonométricas. Esto garantiza que el modelo de clustering reconozca la continuidad del año académico, permitiendo una segmentación precisa de comportamientos vinculados a periodos de alta demanda bibliográfica, como cierres de semestre y procesos de matriculación, sin las distorsiones propias de una escala lineal

VARIABLE: facultad **Se hace un One-Hot Encoding**

Para la variable categórica Facultad, se aplicó una transformación de codificación One-Hot. Esta técnica evita el sesgo de jerarquía que produciría una codificación numérica simple (Label Encoding), permitiendo que el algoritmo de clustering trate a cada unidad académica como una dimensión independiente en el espacio vectorial de análisis

In [ ]:
# Creamos variables "dummies" para cada facultad
df_facultades = pd.get_dummies(df_unificado_clustering['facultad'], prefix='fac')

# Unimos estas nuevas columnas a nuestro dataframe principal
df_unificado_clustering = pd.concat([df_unificado_clustering, df_facultades], axis=1)

# Mostramos cómo quedaron las nuevas columnas (serán muchos 0 y un 1)
df_facultades.sample(10)

In [ ]:
df_unificado_clustering.sample(10)

In [ ]:
df_unificado_clustering['facultad'].isnull().sum()

In [ ]:
# Convertimos los True/False en 1/0 en todo el dataframe
#df_unificado_clustering = df_unificado_clustering.fillna(0) # Aprovechamos para quitar los NaNs que vimos antes
#df_unificado_clustering = df_unificado_clustering * 1 # El truco del almendruco: multiplicar por 1 convierte True en

VARIABLES: nivel_academico **Se hace un One Hot Encoding**

Se aplicó una codificación de variables ficticias (Dummies) a la columna 'Nivel Académico'. Este tratamiento es indispensable para algoritmos de aprendizaje no supervisado, ya que permite representar categorías multiclase en un espacio euclidiano sin introducir un orden de magnitud artificial, garantizando que el modelo identifique clústeres basados puramente en la similitud de perfiles.

In [ ]:
import pandas as pd

# 1. Creamos las columnas One-Hot para Nivel Académico
# prefix='nivel' para que las columnas se llamen 'nivel_Pregrado', 'nivel_Posgrado', etc.
df_niveles_dummies = pd.get_dummies(df_unificado_clustering['nivel_academico_final'], prefix='nivel')

# 2. Las unimos al dataframe principal
df_unificado_clustering = pd.concat([df_unificado_clustering, df_niveles_dummies], axis=1)

# Verificamos qué niveles se crearon
print("Nuevas columnas de Nivel Académico:")
print(df_niveles_dummies.columns.tolist())
print("\nPrimeras filas con la transformación:")
print(df_unificado_clustering[df_niveles_dummies.columns].head())

In [ ]:
df_unificado_clustering.sample(20)

In [ ]:
# Contamos cuántas veces aparece cada id_perfil en todo el dataset unificado
# transform('count') pone el resultado de ese conteo en cada fila correspondiente
df_unificado_clustering['total_interacciones'] = df_unificado_clustering.groupby('id_perfil')['id_perfil'].transform('count')

# Verificamos los primeros 5 para ver que funcionó
print(df_unificado_clustering[['id_perfil', 'total_interacciones']].head(15))

In [ ]:
df_unificado_clustering.sample(10)

VARIABLES: carrera_final **Se hace un One Hot Encoding**

In [ ]:
# 1. Creamos las columnas One-Hot para Carrera
# Usamos el prefijo 'car' para identificarlas fácilmente
df_carreras_dummies = pd.get_dummies(df_unificado_clustering['carrera_final'], prefix='car')

# 2. Las unimos al dataframe principal
df_unificado_clustering = pd.concat([df_unificado_clustering, df_carreras_dummies], axis=1)

print(f"Nuevas columnas de Carrera creadas: {len(df_carreras_dummies.columns)}")
print("\nMuestra de las primeras columnas de carrera:")
print(df_carreras_dummies.columns[:10].tolist())

In [ ]:
df_unificado_clustering.sample(20)

In [ ]:
df_unificado_clustering.columns.tolist

#### Unificaciones para el modelo (hasta acá todo corre bien)
Consolidamos toda la historia de cada usuario en una sola fila. Usamos la media para el comportamiento y el valor máximo para las categorías (dummies), ya que si un usuario es de "Ingeniería" una vez, lo es siempre.

*Selección de variables y estandarización*

In [ ]:
# Definimos las columnas que ya tienes transformadas
cols_comportamiento = ['sesiones', 'busquedas', 'descargas', 'total_interacciones', 
                       'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos', 'clasificacion_recurso']

# Seleccionamos todas las que creamos con get_dummies (facultad, carrera, nivel)
cols_dummies = [c for c in df_unificado_clustering.columns if c.startswith(('fac_', 'niv_', 'car_'))]

# AGRUPAMOS: Aquí es donde creamos los "Perfiles"
df_perfiles = df_unificado_clustering.groupby('id_perfil').agg({
    **{col: 'mean' for col in cols_comportamiento},
    **{col: 'max' for col in cols_dummies}
}).fillna(0)

print(f"Total de usuarios únicos para el modelo: {len(df_perfiles)}")

In [ ]:
df_perfiles.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

# Creamos el escalador
scaler = StandardScaler()

# Ajustamos y transformamos en un solo paso
X_scaled = scaler.fit_transform(df_perfiles)

# Lo convertimos en un DataFrame limpio para el modelo
X_final = pd.DataFrame(X_scaled, columns=df_perfiles.columns)

print("¡Estandarización única completada!")
X_final.head()

In [ ]:
X_final.shape

In [ ]:
df_perfiles.columns.tolist()

In [ ]:
print(df_unificado_clustering.shape)
df_unificado_clustering.columns.tolist

Organización de sesiones, descargas y búsquedas para usr en el RFM

In [ ]:
# Creamos la variable de esfuerzo digital total (Sesiones + Búsquedas + Descargas)
df_unificado_clustering['volumen_digital'] = df_unificado_clustering['sesiones'] + df_unificado_clustering['busquedas'] + df_unificado_clustering['descargas']

In [ ]:
# Fechas de semestres UTP — CONFIRMA ESTAS CON TU DIRECTOR
semestres_utp = [
    (pd.Timestamp('2022-02-07'), pd.Timestamp('2022-05-28')),
    (pd.Timestamp('2022-08-08'), pd.Timestamp('2022-11-19')),
    (pd.Timestamp('2023-02-13'), pd.Timestamp('2023-06-03')),
    (pd.Timestamp('2023-08-08'), pd.Timestamp('2023-11-25')),
    (pd.Timestamp('2024-02-05'), pd.Timestamp('2024-06-01')),
    (pd.Timestamp('2024-08-12'), pd.Timestamp('2024-11-30')),
    (pd.Timestamp('2025-02-03'), pd.Timestamp('2025-05-31')),
    (pd.Timestamp('2025-08-11'), pd.Timestamp('2025-11-29')),
]

def asignar_semana(fecha):
    for inicio, fin in semestres_utp:
        if inicio <= fecha <= fin:
            semana = int((fecha - inicio).days / 7) + 1
            return min(semana, 16)
    return None  # vacaciones o fuera de semestre

df_unificado_clustering['semana_semestre'] = (
    df_unificado_clustering['fechas'].apply(asignar_semana)
)

# Verificar
print(df_unificado_clustering['semana_semestre'].value_counts().sort_index())
print(f"\nRegistros fuera de semestre (vacaciones): "
      f"{df_unificado_clustering['semana_semestre'].isna().sum():,}")


# Para semestres
# Función para etiquetar a qué semestre pertenece la transacción
def asignar_periodo_academico(fecha):
    for inicio, fin in semestres_utp:
        if inicio <= fecha <= fin:
            # Identifica si es el primer o segundo semestre del año
            periodo = "1" if inicio.month < 6 else "2"
            return f"{inicio.year}-{periodo}" # Ej: Retorna "2022-1" o "2024-2"
    return 'Vacaciones'

# Lo aplicas creando una nueva columna para tus metadatos
df_unificado_clustering['periodo_academico'] = (
    df_unificado_clustering['fechas'].apply(asignar_periodo_academico)
)

Ahora el paso clave: cruzar las semanas con los clústeres. Corre esto:

Creación de variables de semanas por semestre y semestre para caracterización al final

In [ ]:
# Fechas de semestres UTP — CONFIRMA ESTAS CON TU DIRECTOR
semestres_utp = [
    (pd.Timestamp('2022-02-07'), pd.Timestamp('2022-05-28')),
    (pd.Timestamp('2022-08-08'), pd.Timestamp('2022-11-19')),
    (pd.Timestamp('2023-02-13'), pd.Timestamp('2023-06-03')),
    (pd.Timestamp('2023-08-08'), pd.Timestamp('2023-11-25')),
    (pd.Timestamp('2024-02-05'), pd.Timestamp('2024-06-01')),
    (pd.Timestamp('2024-08-12'), pd.Timestamp('2024-11-30')),
    (pd.Timestamp('2025-02-03'), pd.Timestamp('2025-05-31')),
    (pd.Timestamp('2025-08-11'), pd.Timestamp('2025-11-29')),
]

def asignar_semana(fecha):
    for inicio, fin in semestres_utp:
        if inicio <= fecha <= fin:
            semana = int((fecha - inicio).days / 7) + 1
            return min(semana, 16)
    return None  # vacaciones o fuera de semestre

df_unificado_clustering['semana_semestre'] = (
    df_unificado_clustering['fechas'].apply(asignar_semana)
)

# Verificar
print(df_unificado_clustering['semana_semestre'].value_counts().sort_index())
print(f"\nRegistros fuera de semestre (vacaciones): "
      f"{df_unificado_clustering['semana_semestre'].isna().sum():,}")


# Para semestres
# Función para etiquetar a qué semestre pertenece la transacción
def asignar_periodo_academico(fecha):
    for inicio, fin in semestres_utp:
        if inicio <= fecha <= fin:
            # Identifica si es el primer o segundo semestre del año
            periodo = "1" if inicio.month < 6 else "2"
            return f"{inicio.year}-{periodo}" # Ej: Retorna "2022-1" o "2024-2"
    return 'Vacaciones'

# Lo aplicas creando una nueva columna para tus metadatos
df_unificado_clustering['periodo_academico'] = (
    df_unificado_clustering['fechas'].apply(asignar_periodo_academico)
)

# MODELO FINAL (OBJETIVO 3)

In [ ]:
# =============================================================================
# RUTA A — SEGMENTACIÓN POR COMPORTAMIENTO REAL (ENFOQUE RFM ADAPTADO)
# Biblioteca Jorge Roa Martínez — UTP
# Retoma desde: df_unificado_clustering (ya limpio y unificado)
# =============================================================================
# FUNDAMENTO METODOLÓGICO:
# En lugar de agrupar transacciones individuales, construimos un vector
# de comportamiento real por usuario (tipo RFM adaptado a bibliotecas).
# Esto resuelve el problema anterior donde variables redundantes como
# sesiones/búsquedas (r=0.87) y el desbalance físico/digital (1:4)
# distorsionaban los clusters.
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# CELDA 1 — Imports
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

# Paleta institucional UTP
COLOR_FISICO  = '#B71C1C'
COLOR_DIGITAL = '#1B5E20'
PALETTE_CLUSTERS = ['#1565C0', '#E65100', '#2E7D32', '#6A1B9A', '#00838F']

print("✅ Librerías cargadas correctamente.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 2 — Verificación del punto de partida
# ─────────────────────────────────────────────────────────────────────────────
# Verificamos que df_unificado_clustering esté disponible y tiene lo que necesitamos

columnas_requeridas = ['codigo', 'fechas', 'tipo_recurso', 'facultad',
                       'sesiones', 'busquedas', 'descargas',
                       'mes', 'hora', 'anio']

cols_faltantes = [c for c in columnas_requeridas if c not in df_unificado_clustering.columns]

if cols_faltantes:
    print(f"⚠️  Faltan columnas: {cols_faltantes}")
else:
    print(f"   df_unificado_clustering listo.")
    print(f"   Registros: {len(df_unificado_clustering):,}")
    print(f"   Usuarios únicos: {df_unificado_clustering['codigo'].nunique():,}")
    print(f"   Rango de fechas: {df_unificado_clustering['fechas'].min().date()} → {df_unificado_clustering['fechas'].max().date()}")
    print(f"   Tamaño de los datos {df_unificado_clustering.shape}")


In [ ]:
df_unificado_clustering.sample(10)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 3 — Construcción del vector RFM adaptado por usuario
# ─────────────────────────────────────────────────────────────────────────────
# DECISIÓN METODOLÓGICA:
# Calculamos 6 features de comportamiento independientes que capturan
# dimensiones distintas del uso de la biblioteca. Esto reemplaza las
# 25+ variables anteriores que tenían alta multicolinealidad.

FECHA_CORTE = df_unificado_clustering['fechas'].max()

print("Construyendo features de comportamiento por usuario...")

df_rfm = df_unificado_clustering.groupby('codigo').agg(
    # --- RECENCY: días desde la última interacción ---
    # Usuarios activos recientemente son más relevantes para decisiones de corto plazo
    recency_dias = ('fechas', lambda x: (FECHA_CORTE - x.max()).days),

    # --- FREQUENCY: número de días únicos con actividad ---
    # Más robusto que contar transacciones porque un usuario que genera
    # 1000 sesiones en un día no es más "frecuente" que uno que va 50 días distintos
    frequency_dias_activos = ('fechas', lambda x: x.dt.date.nunique()),

    # --- DIVERSITY: recursos distintos usados ---
    # Captura amplitud intelectual: ¿explorador o especialista?
    # Nota metodológica: Para recursos físicos cuenta títulos únicos prestados. 
    # Para recursos digitales cuenta bases de datos/plataformas únicas consultadas.
    diversity_recursos = ('nombre_recurso', 'nunique'),

    # --- RATIO DIGITAL: proporción de interacciones digitales ---
    # Eje clave del comportamiento: ¿qué canal prefiere el usuario?
    # 0.0 = solo físico | 1.0 = solo digital | valores intermedios = híbrido
    ratio_digital = ('tipo_recurso', lambda x: (x == 'digital').mean()),

    # --- INTENSIDAD DIGITAL: promedio de sesiones/búsquedas/descargas ---
    # Solo aplica a interacciones digitales (en físico es 0, por diseño)
    # Usamos la suma normalizada para no crear correlación artificial
    intensidad_digital = ('volumen_digital', lambda x: x[x > 0].mean() if (x > 0).any() else 0),

    # --- TEMPORALIDAD: hora promedio de actividad (reconstruida del seno/coseno) ---
    # Identifica usuarios diurnos vs. nocturnos, relevante para horarios de atención
    hora_sin_promedio = ('hora_sin', 'mean'),
    hora_cos_promedio = ('hora_cos', 'mean'),

    # Metadatos adicionales (no entran al modelo, pero sí a la caracterización)
    facultad_moda = ('facultad', lambda x: x.mode()[0] if not x.mode().empty else 'No Registra'),
    anio_primera  = ('anio', 'min'),
    anio_ultima   = ('anio', 'max'),
    total_registros = ('codigo', 'count'),
    #También usaremos los semestres y semanas del semestre de cada año (Variable creada anteriormente en la caracterización del objetivo 2)

).reset_index()

# 2. AQUÍ, después, en línea separada:
df_rfm['prop_fisica'] = 1 - df_rfm['ratio_digital']

print(f"✅ Vector RFM construido: {df_rfm.shape[0]:,} usuarios × {df_rfm.shape[1]} columnas")
print(f"\nPrimeras filas:")
display(df_rfm.head())
display(df_rfm.columns.to_list)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CREACIÓN DE VARIABLE: TENURE Y ACTUALIZACIÓN FINAL DEL MODELO
# ─────────────────────────────────────────────────────────────────────────────
print("--- Calculando la métrica: Tenure (Tiempo de vida del usuario) ---")

# 1. Buscamos la fecha de la PRIMERA visita de cada estudiante
fecha_primera_visita = df_unificado_clustering.groupby('codigo')['fechas'].min()

# 2. Buscamos la fecha de corte de la base de datos (último registro global)
fecha_max_global = df_unificado_clustering['fechas'].max()

# 3. Calculamos el Tenure (Diferencia en días)
# sumamos +1 para que si vino hoy y el corte es hoy, sea 1 día de vida y no 0
dias_tenure = (fecha_max_global - fecha_primera_visita).dt.days + 1

# 4. Mapeamos la nueva variable a nuestro dataframe limpio
if 'codigo' in df_rfm.columns:
    df_rfm['tenure_dias'] = df_rfm['codigo'].map(dias_tenure)
else:
    df_rfm['tenure_dias'] = df_rfm.index.map(dias_tenure)

print("✅ 'tenure_dias' calculada con éxito.")


In [ ]:
# Proporción de meses distintos con actividad sobre el total de meses 
# en que el usuario estuvo activo (entre su primera y última fecha)
# Un valor cercano a 1 = uso distribuido todo el año
# Un valor cercano a 0 = uso concentrado en pocos meses (solo parciales)

# PASO 1 — Calcular consistencia_uso desde el dataframe original
consistencia = df_unificado_clustering.groupby('codigo')['fechas'].apply(
    lambda x: x.dt.to_period('M').nunique() / max(((x.max() - x.min()).days / 30), 1)
).clip(upper=1).reset_index()

consistencia.columns = ['codigo', 'consistencia_uso']

# Verificar que salió bien antes de unir
print(consistencia.head())
print(consistencia['consistencia_uso'].describe())


In [ ]:
# PASO 2 — Unir a df_rfm_limpio
df_rfm = df_rfm.reset_index().merge(
    consistencia, on='codigo', how='left'
).set_index('codigo')


In [ ]:
# PASO 1 — Calcular frequency_fisica
frequency_fisica = (
    df_unificado_clustering[df_unificado_clustering['tipo_recurso'] == 'fisico']
    .groupby('codigo')['fechas']
    .nunique()
    .reset_index()
)
frequency_fisica.columns = ['codigo', 'frequency_fisica']

# Verificar que salió bien
print(frequency_fisica.head())
print(f"Registros: {len(frequency_fisica)}")

In [ ]:
# PASO 2 — Merge
df_rfm = df_rfm.reset_index().merge(
    frequency_fisica, on='codigo', how='left'
).set_index('codigo')

df_rfm['frequency_fisica'] = df_rfm['frequency_fisica'].fillna(0)

In [ ]:
# Simplemente si el usuario ALGUNA VEZ usó el servicio físico
df_rfm['usa_fisico'] = (df_rfm['frequency_fisica'] > 0).astype(int)

print(df_rfm['usa_fisico'].value_counts())
print(df_rfm['usa_fisico'].corr(df_rfm['intensidad_digital']).round(2))

Esta variable nos muestra que solo 9,843 usuarios tienen registro físico de 17,864 totales — casi la mitad nunca usó el servicio físico. Eso es exactamente la dimensión que faltaba en el modelo.

1. recency_dias = Esta línea calcula el tiempo (en días) que ha transcurrido desde la última vez que un usuario específico interactuó con la biblioteca (ya sea prestando un libro físico o consultando una base de datos digital) hasta una fecha de referencia (FECHA_CORTE).

2. frequency_dias_activos = Esta línea está calculando la Frecuencia (la "F" del modelo RFM). En lugar de contar cuántas veces el usuario hizo "clic" o pidió un libro, cuenta cuántos días distintos el usuario utilizó la biblioteca.

3. diversity_recursos: La cantidad de recursos únicos (títulos de libros diferentes o plataformas distintas) que el usuario consumió en su historia.

4. ratio_digital: proporción de la actividad de un usuario que ocurre en el entorno virtual. 
   - 0.0: Significa 0% digital. El usuario es 100% físico (solo saca libros impresos).
   - 0.5: Significa 50% digital y 50% físico (un usuario perfectamente equilibrado).
   - 1.0: Significa 100% digital. El usuario nunca ha tocado un libro de papel de la biblioteca.

5. intensidad_digital: Nos dice qué tanta energía gasta o qué volumen de información consume en cada una de esas visitas.

6. hora sin y cos: Identifica el cronotipo de estudio del usuario (diurno, vespertino o nocturno) calculando el punto de máxima concentración de su actividad en un ciclo continuo de 24 horas. (ransformación Trigonométrica: Para evitar este salto matemático, la hora se proyecta en un círculo continuo bidimensional utilizando funciones de seno y coseno)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 4 — Diagnóstico de distribuciones y skewness (CORREGIDA)
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

features_modelo = ['recency_dias', 'frequency_dias_activos',
                   'diversity_recursos', 
                   'ratio_digital',
                   'intensidad_digital', #'hora_sin_promedio', 'hora_cos_promedio', 
                   'tenure_dias', 
                   'consistencia_uso','usa_fisico']

fig, axes = plt.subplots(2, len(features_modelo), figsize=(25, 8)) 
fig.suptitle('Distribución de Features — Diagnóstico de Skewness', fontsize=14, fontweight='bold')

nombres_display = ['Recency\n(días)', 'Frequency\n(días activos)',
                   'Diversity\n(recursos únicos)', 'Ratio\nDigital',
                   'Intensidad\nDigital', 'tenure_dias',
                    'consistencia_uso','usa_fisico']

for idx, (col, nombre) in enumerate(zip(features_modelo, nombres_display)):
    datos = df_rfm[col].dropna()
    skew_orig = datos.skew()

    # Fila superior: distribución original
    axes[0, idx].hist(datos, bins=50, color='#455A64', alpha=0.8, edgecolor='none')
    axes[0, idx].set_title(f'{nombre}\nskew = {skew_orig:.2f}', fontsize=10)
    axes[0, idx].set_ylabel('Frecuencia' if idx == 0 else '')

    # Fila inferior: distribución log1p (Solo para las que aplican)
    if col in ['hora_sin_promedio', 'hora_cos_promedio']:
        axes[1, idx].text(0.5, 0.5, 'No aplica log1p\n(Coordenadas)',
                          horizontalalignment='center', verticalalignment='center',
                          transform=axes[1, idx].transAxes, fontsize=12, color='gray')
        axes[1, idx].set_axis_off()
    else:
        # Aplicamos el logaritmo solo a valores > 0 por seguridad
        # Para evitar valores negativos extremos si los hubiera
        datos_log = np.log1p(np.clip(datos, 0, None)) 
        skew_log = datos_log.skew()
        
        # Manejo seguro para evitar el error de "NA is ambiguous"
        if pd.isna(skew_log):
            color = '#9E9E9E'  # Gris si el skew no se puede calcular
            skew_str = "NaN"
        else:
            color = '#2E7D32' if abs(skew_log) < 1.5 else '#E65100' # Verde (Aceptable) o Naranja (Alto Sesgo)
            skew_str = f"{skew_log:.2f}"

        axes[1, idx].hist(datos_log, bins=50, color=color, alpha=0.8, edgecolor='none')
        axes[1, idx].set_title(f'log1p({nombre.split(chr(10))[0]})\nskew = {skew_str}', fontsize=10)
        axes[1, idx].set_ylabel('Frecuencia' if idx == 0 else '')

axes[0, 0].set_ylabel('Original', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Transformada (log1p)', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('distribucion_features_RFM.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado como 'distribucion_features_RFM.png'")

- frequency skew 0.82 → el log la mejora, sí aplicar
- tenure_dias skew -1.84 → el log la empeora, queda con cola izquierda fuerte, no aplicar
- intensidad_digital skew -0.97 → el log la empeora también, no aplicar
- consistencia_uso skew -0.28 → ya está bien desde el original, no aplicar
- recency_dias skew 0.78 → moderado, dejarla sin log

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

df_rfm['consistencia_ponderada'] = (
    df_rfm['consistencia_uso'] * 
    (1 - np.exp(-df_rfm['frequency_dias_activos'] / 5))
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_rfm['consistencia_uso'], bins=50, color='steelblue')
axes[0].set_title(f"Original\nskew={df_rfm['consistencia_uso'].skew():.2f}")

axes[1].hist(df_rfm['consistencia_ponderada'], bins=50, color='green')
axes[1].set_title(f"Ponderada\nskew={df_rfm['consistencia_ponderada'].skew():.2f}")

plt.tight_layout()
plt.show()

# Casos concretos
print("=== Efecto en usuarios de baja frecuencia ===")
print(df_rfm[df_rfm['frequency_dias_activos'] <= 3][
    ['frequency_dias_activos','consistencia_uso','consistencia_ponderada']
].head(10).round(3))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FIGURA 5 — Diagnóstico de distribuciones — 4 variables finales del modelo
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

features_modelo  = ['frequency_dias_activos', 'recency_dias',
                    'intensidad_digital', 'consistencia_uso']

nombres_display  = ['Frecuencia\ndías activos', 'Recencia\n(días)',
                    'Intensidad\ndigital', 'Consistencia\nde uso']

fig, axes = plt.subplots(2, 4, figsize=(16, 7), facecolor='white')
fig.patch.set_facecolor('white')

for idx, (col, nombre) in enumerate(zip(features_modelo, nombres_display)):
    datos     = df_rfm[col].dropna()
    skew_orig = round(datos.skew(), 2)
    datos_log = np.log1p(np.clip(datos, 0, None))
    skew_log  = round(datos_log.skew(), 2)

    # ── Fila 0: distribución original ──
    axes[0, idx].hist(datos, bins=50,
                      color=PALETA['digital'], alpha=0.85, edgecolor='none')
    axes[0, idx].set_title(f'{nombre}\nskew = {skew_orig}',
                            fontsize=10, fontweight='bold', color=PALETA['texto'])
    axes[0, idx].set_facecolor('white')
    axes[0, idx].tick_params(colors=PALETA['texto'], labelsize=8)
    axes[0, idx].spines['top'].set_visible(False)
    axes[0, idx].spines['right'].set_visible(False)

    # ── Fila 1: distribución log1p ──
    # Verde si mejora, gris si no mejora
    color_log = PALETA['digital'] if abs(skew_log) < abs(skew_orig) else '#9E9E9E'
    axes[1, idx].hist(datos_log, bins=50,
                      color=color_log, alpha=0.85, edgecolor='none')
    axes[1, idx].set_title(f'log1p\nskew = {skew_log}',
                            fontsize=10, fontweight='bold', color=PALETA['texto'])
    axes[1, idx].set_facecolor('white')
    axes[1, idx].tick_params(colors=PALETA['texto'], labelsize=8)
    axes[1, idx].spines['top'].set_visible(False)
    axes[1, idx].spines['right'].set_visible(False)

# Etiquetas de filas
axes[0, 0].set_ylabel('Original', fontsize=11, fontweight='bold',
                       color=PALETA['texto'])
axes[1, 0].set_ylabel('Transformada (log1p)', fontsize=11, fontweight='bold',
                       color=PALETA['texto'])

plt.tight_layout()
plt.savefig('figura_5_distribucion.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ Figura 5 guardada")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FIGURA 5a-5d — Distribución individual por variable
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

variables = [
    ('frequency_dias_activos', 'Frecuencia días activos'),
    ('recency_dias',           'Recencia (días)'),
    ('intensidad_digital',     'Intensidad digital'),
    ('consistencia_uso',       'Consistencia de uso'),
]

letras = ['5a', '5b', '5c', '5d']

for (col, nombre), letra in zip(variables, letras):
    datos     = df_rfm[col].dropna()
    skew_orig = round(datos.skew(), 2)
    datos_log = np.log1p(np.clip(datos, 0, None))
    skew_log  = round(datos_log.skew(), 2)
    mejora    = abs(skew_log) < abs(skew_orig)
    color_log = PALETA['digital'] if mejora else '#9E9E9E'

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='white')
    fig.patch.set_facecolor('white')

    # Panel izquierdo — original
    axes[0].hist(datos, bins=50, color=PALETA['digital'],
                 alpha=0.85, edgecolor='none')
    axes[0].set_title(f'Distribución original\nskew = {skew_orig}',
                      fontsize=11, fontweight='bold', color=PALETA['texto'])
    axes[0].set_xlabel(nombre, fontsize=10, color=PALETA['texto'])
    axes[0].set_ylabel('Frecuencia', fontsize=10, color=PALETA['texto'])
    axes[0].set_facecolor('white')
    axes[0].tick_params(colors=PALETA['texto'])
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

    # Panel derecho — log1p
    axes[1].hist(datos_log, bins=50, color=color_log,
                 alpha=0.85, edgecolor='none')
    estado = '✓ Mejora con log1p' if mejora else '✗ No mejora con log1p'
    axes[1].set_title(f'Transformación log1p — {estado}\nskew = {skew_log}',
                      fontsize=11, fontweight='bold', color=PALETA['texto'])
    axes[1].set_xlabel(f'log1p({nombre})', fontsize=10, color=PALETA['texto'])
    axes[1].set_ylabel('Frecuencia', fontsize=10, color=PALETA['texto'])
    axes[1].set_facecolor('white')
    axes[1].tick_params(colors=PALETA['texto'])
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)

    plt.tight_layout()
    nombre_archivo = f'figura_{letra}_dist_{col}.png'
    plt.savefig(nombre_archivo, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"✅ Figura {letra} guardada — {nombre_archivo}")

In [ ]:
df_rfm[['frequency_dias_activos', 'recency_dias', 
        'intensidad_digital', 'consistencia_uso',
        'prop_fisica', 'usa_fisico']].corr().round(2)

- frequency_dias_activos (con log) — ¿qué tan habitual es este usuario? No cuenta transacciones sino días únicos con actividad, lo que lo hace robusto. Un usuario que generó 500 sesiones en un día no es más frecuente que uno que fue 30 días distintos. El logaritmo suaviza a los usuarios extremadamente frecuentes para que no distorsionen los clusters.
- recency_dias — ¿este usuario sigue siendo usuario activo hoy? Mide cuántos días han pasado desde su última interacción con la biblioteca. Es la variable que separa a los usuarios vigentes de los que se fueron — un usuario que lleva 800 días sin aparecer tiene un perfil de comportamiento actual completamente distinto al que vino la semana pasada, sin importar cuánto haya usado la biblioteca antes.
- intensidad_digital — ¿qué tan profundo es su uso de los recursos digitales? Captura el promedio de volumen digital (sesiones, búsquedas, descargas) en las interacciones donde hubo actividad digital. Diferencia al usuario que solo entra al portal del usuario que realmente consume bases de datos e investiga. Es la variable más independiente del modelo — correlación máxima de 0.29 con las demás.
- consistencia_uso — ¿viene todo el año o solo en épocas de parciales? Mide qué proporción de los meses posibles dentro de su ventana de actividad tuvo al menos una interacción. Un valor cercano a 1 significa uso distribuido y sostenido a lo largo del tiempo. Un valor cercano a 0 significa que aparece en rachas cortas y desaparece. Esta variable es la que distingue al usuario con hábito real de biblioteca del usuario de emergencia académica.

Se identificó una correlación crítica de 0.88 entre las variables ratio_digital e intensidad_digital. Se optó por conservar únicamente la Intensidad Digital. Mientras que el Ratio es una proporción porcentual (0 a 1), la Intensidad captura el volumen real de interacción (búsquedas y descargas). La Intensidad "contiene" la información del Ratio pero con mayor profundidad analítica.

Se decidió conservar Frecuencia y Diversidad a pesar de su correlación (0.77), ya que representan constructos psicológicos distintos: el hábito frente a la curiosidad intelectual. Mantener ambas permite al modelo diferenciar entre usuarios con necesidades altamente especializadas y usuarios con comportamientos de exploración multidisciplinaria, enriqueciendo la segmentación final.

In [ ]:
# 1. Definimos el destino de cada variable basado en el diagnóstico de la Celda 4
features_log = ['frequency_dias_activos'] #,'diversity_recursos' # Las únicas que mejoraron con log
# Sacamos ratio_digital porque tiene 0.88 de correlación con intensidad
features_lineales = ['recency_dias', 'intensidad_digital', 'consistencia_uso'
                     ]

# 2. Aplicamos log1p solo a las variables que lo necesitan
X = df_rfm.copy()
for col in features_log:
    X[col] = np.log1p(X[col])

# 3. Seleccionamos solo las columnas que van al modelo
cols_modelo = features_log + features_lineales
X_final = X[cols_modelo].fillna(0)

# 4. REVISIÓN DE CORRELACIONES (Antes de escalar)
print("--- MATRIZ DE CORRELACIONES ---")
corr = X_final.corr().round(2)
display(corr)

# Alerta de multicolinealidad
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            print(f"⚠️ Alta correlación: {corr.columns[i]} ↔ {corr.columns[j]}: {corr.iloc[i, j]}")

# 5. ESCALADO (StandardScaler) - Paso obligatorio para K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

# Convertimos a DataFrame para que sea fácil de leer
X_scaled_df = pd.DataFrame(X_scaled, columns=cols_modelo, index=X_final.index)

print(f"\n✅ Matriz escalada lista para K-Means: {X_scaled_df.shape[0]:,} usuarios × {X_scaled_df.shape[1]} variables")
display(X_scaled_df.head())

In [ ]:
# Ver distribución de las 4 variables del modelo
cols_modelo = ['frequency_dias_activos', 'recency_dias', 
               'intensidad_digital', 'consistencia_uso']

print(df_rfm[cols_modelo].describe().round(2))
print(f"\nSkewness:")
print(df_rfm[cols_modelo].skew().round(2))
print(f"\nCorrelación:")
print(df_rfm[cols_modelo].corr().round(2))

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA 6 — Matriz de Correlación — Variables del Modelo RFM
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

cols_modelo = ['frequency_dias_activos', 'recency_dias', 
               'intensidad_digital', 'consistencia_uso']

labels_corr = ['Frecuencia\ndías activos', 'Recencia\n(días)', 
               'Intensidad\ndigital', 'Consistencia\nde uso']

import numpy as np

# Aplicar log1p igual que en el modelo
X_corr = df_rfm[cols_modelo].copy()
X_corr['frequency_dias_activos'] = np.log1p(X_corr['frequency_dias_activos'])

corr = X_corr.corr().round(2)

# El resto del código igual...

fig, ax = plt.subplots(figsize=(7, 6), facecolor='white')

cmap = mcolors.LinearSegmentedColormap.from_list(
    'custom', ['#C0392B', 'white', '#1A3A5C'], N=256)

im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(labels_corr)))
ax.set_yticks(range(len(labels_corr)))
ax.set_xticklabels(labels_corr, fontsize=10, color='#1C2833')
ax.set_yticklabels(labels_corr, fontsize=10, color='#1C2833')

for i in range(len(cols_modelo)):
    for j in range(len(cols_modelo)):
        val = corr.values[i, j]
        color_txt = 'white' if abs(val) > 0.5 else '#1C2833'
        ax.text(j, i, f'{val:.2f}',
                ha='center', va='center',
                fontsize=12, fontweight='bold', color=color_txt)

plt.colorbar(im, ax=ax, shrink=0.8, label='Correlación de Pearson')

ax.set_title('Figura 6. Matriz de Correlación — Variables del Modelo RFM',
             fontsize=12, fontweight='bold', color='#1C2833', pad=14)

ax.set_facecolor('white')
plt.tight_layout()
plt.savefig('figura_6_correlacion_FINAL.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Guardada como figura_6_correlacion_FINAL.png")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA 5 — Distribución y skewness de las 4 variables
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np

PALETA = {
    'digital':   '#1A3A5C',
    'fisico':    '#C0392B',
    'acento':    '#F39C12',
    'fondo':     '#F7F9FC',
    'grid':      '#DDE3EA',
    'texto':     '#1C2833',
}

cols_modelo = ['frequency_dias_activos', 'recency_dias',
               'intensidad_digital', 'consistencia_uso']

labels_dist = ['Frecuencia días activos', 'Recencia (días)',
               'Intensidad digital', 'Consistencia de uso']

skews = df_rfm[cols_modelo].skew().round(2)

fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor='white')
fig.suptitle('Figura 5. Diagnóstico de Distribución — Variables del Modelo RFM',
             fontsize=13, fontweight='bold', color=PALETA['texto'], y=1.01)

for idx, (col, label) in enumerate(zip(cols_modelo, labels_dist)):
    # Fila 0 — original
    ax_orig = axes[0, idx]
    data = df_rfm[col].dropna()
    ax_orig.hist(data, bins=40, color=PALETA['digital'], 
                 edgecolor='white', linewidth=0.3, alpha=0.85)
    ax_orig.set_title(f'{label}\nskew = {skews[col]}',
                      fontsize=9.5, fontweight='bold', color=PALETA['texto'])
    ax_orig.set_ylabel('Frecuencia' if idx == 0 else '', fontsize=9)
    ax_orig.set_facecolor('white')
    ax_orig.tick_params(colors=PALETA['texto'], labelsize=8)
    if idx == 0:
        axes[0, 0].set_ylabel('Original', fontsize=10, fontweight='bold',
                               color=PALETA['texto'])

    # Fila 1 — transformada con log1p
    ax_log = axes[1, idx]
    data_log = np.log1p(data)
    skew_log = round(data_log.skew(), 2)
    color_log = PALETA['digital'] if abs(skew_log) < abs(skews[col]) else '#888888'
    ax_log.hist(data_log, bins=40, color=color_log,
                edgecolor='white', linewidth=0.3, alpha=0.85)
    ax_log.set_title(f'log1p({label})\nskew = {skew_log}',
                     fontsize=9.5, fontweight='bold', color=PALETA['texto'])
    ax_log.set_facecolor('white')
    ax_log.tick_params(colors=PALETA['texto'], labelsize=8)
    if idx == 0:
        axes[1, 0].set_ylabel('Log-transformada', fontsize=10, fontweight='bold',
                               color=PALETA['texto'])

plt.tight_layout()
plt.savefig('figura_5_distribucion.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura 5 guardada")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 6 — Selección del número óptimo de clusters (Codo + Silhouette)
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import silhouette_score
import numpy as np

# IMPORTANTE: Usamos los datos que ya escalamos en la Celda 5
data_final = X_scaled_df.values 

rango_k = range(2, 11)
inercias = []
silhouettes = []

print(f"Evaluando K de 2 a 10 sobre {len(data_final):,} usuarios...")

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=15, max_iter=300)
    labels = km.fit_predict(data_final)
    
    inercias.append(km.inertia_)
    
    # Silhouette sobre muestra de 5000 para no agotar la RAM
    sil = silhouette_score(data_final, labels, sample_size=5000, random_state=42)
    silhouettes.append(sil)
    print(f"  k={k}: Inercia={km.inertia_:,.0f} | Silhouette={sil:.4f}")

# Gráfico dual profesional
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Diagnóstico de Agrupamiento: Codo vs. Silueta', fontsize=14, fontweight='bold')

# 1. Gráfico del Codo
axes[0].plot(rango_k, inercias, 'o-', color='#1565C0', linewidth=2, markersize=8)
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inercia (Variación interna)')
axes[0].set_title('Método del Codo\n(Busca el punto de inflexión)')
axes[0].grid(alpha=0.3)

# 2. Gráfico de Silueta
axes[1].plot(rango_k, silhouettes, 's-', color='#2E7D32', linewidth=2, markersize=8)
best_k = rango_k[np.argmax(silhouettes)]
axes[1].axvline(x=best_k, color='red', linestyle='--', alpha=0.6, label=f'Max Silhouette (K={best_k})')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Coeficiente de Silueta')
axes[1].set_title('Coeficiente de Silueta\n(Más alto = mejor separación)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('seleccion_k_clusters.png', dpi=150)
plt.show()

De manera preliminar vemos que el modelo da unos resultados muy bajos en su precisión.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PASO HÍBRIDO A — Limpieza de Outliers con DBSCAN
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.cluster import DBSCAN

# eps: Distancia máxima entre dos muestras para que se consideren vecinas
# min_samples: Mínimo de personas para formar un "barrio" denso
dbscan = DBSCAN(eps=1.5, min_samples=10) 
db_labels = dbscan.fit_predict(X_scaled_df)

# Añadimos la etiqueta temporal al DataFrame
df_rfm['dbscan_label'] = db_labels

n_outliers = (db_labels == -1).sum()
print(f"📊 Usuarios analizados: {len(df_rfm):,}")
print(f"🚨 Outliers detectados (Ruido): {n_outliers:,} ({n_outliers/len(df_rfm)*100:.2f}%)")

# Creamos el dataset "Limpio" (solo los que NO son -1)
X_limpio_scaled = X_scaled_df[db_labels != -1]
df_rfm_limpio = df_rfm[db_labels != -1].copy()

print(f"✅ Dataset filtrado listo para K-Means: {len(X_limpio_scaled):,} usuarios")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELDA MAESTRA — Correr siempre después de reiniciar el kernel
# ══════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

PALETA = {
    'digital': '#1A3A5C', 'fisico': '#C0392B', 'acento': '#F39C12',
    'fondo': '#F7F9FC', 'grid': '#DDE3EA', 'texto': '#1C2833',
}
colores_cluster = {0: '#C0392B', 1: '#1A3A5C', 2: '#2E86AB', 3: '#F39C12'}
nombres_fac = {
    'salud': 'Salud', 'ingenierias': 'Ingenierías',
    'ciencias_educacion': 'Cs. Educación', 'tecnologia': 'Tecnología',
    'apoyo_administrativo': 'Apoyo Adm.', 'bellas_artes': 'Bellas Artes',
    'ciencias_empresariales': 'Cs. Empresariales', 'ciencias_agrarias': 'Cs. Agrarias',
    'mecanica': 'Mecánica', 'ciencias_ambientales': 'Cs. Ambientales',
    'externo': 'Externo', 'ciencias_basicas': 'Cs. Básicas',
}
facultades_validas = list(nombres_fac.keys())
cols_modelo = ['frequency_dias_activos', 'recency_dias',
               'intensidad_digital', 'consistencia_uso']

# Reconstruir modelo
X = df_rfm_limpio[cols_modelo].fillna(0).copy()
X['frequency_dias_activos'] = np.log1p(X['frequency_dias_activos'])
scaler = StandardScaler()
X_limpio_scaled = scaler.fit_transform(X)
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=25)
df_rfm_limpio['cluster'] = kmeans_final.fit_predict(X_limpio_scaled)

# Reconstruir df_plot y df_fac
df_plot = df_unificado_clustering.merge(
    df_rfm_limpio[['cluster']], left_on='codigo', right_index=True, how='inner'
)
df_fac = df_plot[df_plot['facultad'].isin(facultades_validas)].copy()

print("✅ Celda maestra lista")
print(df_rfm_limpio['cluster'].value_counts().sort_index())
print(f"df_plot: {df_plot.shape} | df_fac: {df_fac.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PASO HÍBRIDO B — Nuevo Diagnóstico sobre Datos Limpios
# ─────────────────────────────────────────────────────────────────────────────
inercias_l = []
silhouettes_l = []
rango_k = range(2, 11)

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=15)
    labels = km.fit_predict(X_limpio_scaled)
    inercias_l.append(km.inertia_)
    sil = silhouette_score(X_limpio_scaled, labels, sample_size=5000, random_state=42)
    silhouettes_l.append(sil)
    print(f"  k={k}: Silhouette Limpio = {sil:.4f}")

# Graficamos para comparar
plt.figure(figsize=(10, 5))
plt.plot(rango_k, silhouettes_l, 's-', color='#E64A19', label='Con Datos Limpios (DBSCAN)')
plt.title('Mejora de la Silueta tras Limpieza de Ruido')
plt.legend()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FIGURA 11 — Elbow + Silhouette
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

rango_k = list(range(2, 11))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='white')
fig.patch.set_facecolor('white')

# ── Panel 1: Método del Codo ──────────────────────────────────
axes[0].plot(rango_k, inercias, 'o-',
             color=PALETA['digital'], linewidth=2.5, markersize=8)
axes[0].set_xlabel('Número de Clusters (K)', fontsize=11, color=PALETA['texto'])
axes[0].set_ylabel('Inercia', fontsize=11, color=PALETA['texto'])
axes[0].set_title('Método del Codo', fontsize=12,
                  fontweight='bold', color=PALETA['texto'])
axes[0].set_facecolor('white')
axes[0].tick_params(colors=PALETA['texto'])
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Panel 2: Coeficiente de Silueta ──────────────────────────
axes[1].plot(rango_k, silhouettes_l, 's-',
             color=PALETA['digital'], linewidth=2.5, markersize=8)

# Marcar K=4 como el óptimo interpretable
axes[1].axvline(x=4, color=PALETA['fisico'], linestyle='--',
                linewidth=1.8, label='K=4 seleccionado (0.4269)')
axes[1].scatter([4], [silhouettes_l[2]], color=PALETA['fisico'],
                zorder=5, s=100)

axes[1].set_xlabel('Número de Clusters (K)', fontsize=11, color=PALETA['texto'])
axes[1].set_ylabel('Coeficiente de Silueta', fontsize=11, color=PALETA['texto'])
axes[1].set_title('Coeficiente de Silueta', fontsize=12,
                  fontweight='bold', color=PALETA['texto'])
axes[1].legend(frameon=False, fontsize=10)
axes[1].set_facecolor('white')
axes[1].tick_params(colors=PALETA['texto'])
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figura_11_elbow_silhouette.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura 11 guardada")

Se seleccionó $K=4$ como número óptimo de clústeres.

Se evaluó la detección de outliers mediante DBSCAN (eps=1.5, min_samples=10) como paso previo a la segmentación. El algoritmo identificó únicamente 2 usuarios atípicos sobre un total de 17,864 (0.01%), lo que indica que la población presenta un comportamiento homogéneo sin valores extremos que distorsionen el modelo. Dado que la exclusión de estos usuarios no mejoró el coeficiente de silueta en ningún valor de K evaluado, se decidió trabajar con el dataset completo de 17,864 usuarios, utilizando los resultados sin filtrado DBSCAN como base del modelo definitivo con K=4 y silueta de 0.4204.

In [ ]:
# Entrenamos el modelo definitivo sobre los datos que limpiaste con DBSCAN
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=25)
clusters_finales = kmeans_final.fit_predict(X_limpio_scaled)

# Guardamos los resultados en tu dataframe
df_rfm_limpio['cluster'] = clusters_finales

In [ ]:
cols_modelo = ['frequency_dias_activos', 'recency_dias', 
               'intensidad_digital', 'consistencia_uso']

print("=== Correlación actual df_rfm ===")
print(df_rfm[cols_modelo].corr().round(2))

print("\n=== Correlación df_rfm_limpio ===")
print(df_rfm_limpio[cols_modelo].corr().round(2))

print("\n=== Shape df_rfm ===", df_rfm.shape)
print("=== Shape df_rfm_limpio ===", df_rfm_limpio.shape)

print("\n=== ¿frequency ya tiene log1p aplicado? ===")
print(df_rfm['frequency_dias_activos'].describe().round(2))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ─────────────────────────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────────────────────────
K_FINAL  = 4
cols_modelo = ['frequency_dias_activos', 'recency_dias', 
               'intensidad_digital', 'consistencia_uso']
labels      = ['Frecuencia', 'Recencia', 'Int. Digital', 'Consistencia']
colors      = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd']

# ─────────────────────────────────────────────────────────────────
# PASO 1 — Escalar igual que el modelo
# ─────────────────────────────────────────────────────────────────
X_radar = df_rfm_limpio[cols_modelo].fillna(0).copy()
X_radar['frequency_dias_activos'] = np.log1p(X_radar['frequency_dias_activos'])

scaler       = StandardScaler()
X_radar_scaled = scaler.fit_transform(X_radar)

# ─────────────────────────────────────────────────────────────────
# PASO 2 — Calcular centroides por cluster
# ─────────────────────────────────────────────────────────────────
radar_data = (
    pd.DataFrame(X_radar_scaled, columns=cols_modelo, index=df_rfm_limpio.index)
    .assign(cluster=df_rfm_limpio['cluster'].values)
    .groupby('cluster')[cols_modelo]
    .mean()
    .reset_index()
)

print(radar_data)

# ─────────────────────────────────────────────────────────────────
# PASO 3 — Graficar
# ─────────────────────────────────────────────────────────────────
n_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, n_vars, endpoint=False).tolist()
angles += angles[:1]

v_min = radar_data[cols_modelo].values.min() - 0.2
v_max = radar_data[cols_modelo].values.max() + 0.2

fig, axes = plt.subplots(figsize=(16, 4), nrows=1, ncols=K_FINAL,
                         subplot_kw=dict(polar=True))

for i, ax in enumerate(axes):
    values = radar_data.iloc[i][cols_modelo].tolist()
    values += values[:1]

    ax.plot(angles, values, color=colors[i], linewidth=2.2)
    ax.fill(angles, values, color=colors[i], alpha=0.3)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=11, fontweight='bold')
    ax.set_ylim(v_min, v_max)
    ax.set_title(f'Cluster {i}', size=14, color='black', y=1.15, fontweight='bold')

plt.tight_layout()
plt.savefig('radar_clusters_tesis_real.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
cols_modelo = ['frequency_dias_activos', 'recency_dias',
               'intensidad_digital', 'consistencia_uso']

print("=== MEDIANAS POR CLUSTER ===")
print(df_rfm_limpio.groupby('cluster')[cols_modelo].median().round(2))

print("\n=== CONTEO DE USUARIOS ===")
print(df_rfm_limpio['cluster'].value_counts().sort_index())

print("\n=== % USO FÍSICO POR CLUSTER ===")
uso_fisico = df_plot.groupby('cluster').apply(
    lambda x: (x['tipo_recurso'] == 'fisico').mean() * 100
).round(1)
print(uso_fisico)

In [ ]:
# ══════════════════════════════════════════════════════════════
# RADARES — Usar clusters YA asignados, sin reentrenar
# ══════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

PALETA = {
    'digital': '#1A3A5C', 'fisico': '#C0392B',
    'fondo': '#F7F9FC', 'grid': '#DDE3EA', 'texto': '#1C2833',
}
colores_cluster = {
    0: '#C0392B', 1: '#1A3A5C', 2: '#2E86AB', 3: '#F39C12'
}
letras = {0: '12a', 1: '12b', 2: '12c', 3: '12d'}

cols_modelo = ['frequency_dias_activos', 'recency_dias',
               'intensidad_digital', 'consistencia_uso']
labels_radar = ['Frecuencia\ndías activos', 'Recencia\n(días)',
                'Intensidad\ndigital', 'Consistencia\nde uso']

# ── Verificar que los clusters ya existen ──────────────────────
print("Verificando clusters existentes:")
print(df_rfm_limpio['cluster'].value_counts().sort_index())

# ── Centroides desde los clusters ya asignados ────────────────
centroides = df_rfm_limpio.groupby('cluster')[cols_modelo].median()
print("\nCentroides (medianas reales):")
print(centroides.round(2))

# ── Normalizar para el radar ───────────────────────────────────
scaler_radar = MinMaxScaler()
centroides_norm = pd.DataFrame(
    scaler_radar.fit_transform(centroides),
    columns=cols_modelo,
    index=centroides.index
)

# ── Ángulos ────────────────────────────────────────────────────
angles = np.linspace(0, 2 * np.pi, len(cols_modelo), endpoint=False).tolist()
angles += angles[:1]

# ── Generar un radar por cluster ───────────────────────────────
for cluster_id in [0, 1, 2, 3]:
    values = centroides_norm.loc[cluster_id, cols_modelo].tolist()
    values += values[:1]
    color  = colores_cluster[cluster_id]

    fig, ax = plt.subplots(figsize=(7, 6),
                           subplot_kw=dict(polar=True),
                           facecolor='white')
    fig.patch.set_facecolor('white')
    ax.set_position([0.1, 0.18, 0.8, 0.75])

    ax.plot(angles, values, color=color, linewidth=2.5)
    ax.fill(angles, values, color=color, alpha=0.25)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids([])

    for angle, label in zip(np.degrees(angles[:-1]), labels_radar):
        angle_rad = np.deg2rad(angle)
        ha = 'center'
        if np.sin(angle_rad) > 0.1:
            ha = 'left'
        elif np.sin(angle_rad) < -0.1:
            ha = 'right'
        ax.text(angle_rad, 1.32, label,
                ha=ha, va='center',
                fontsize=11, fontweight='bold',
                color=PALETA['texto'],
                transform=ax.transData)

    ax.set_ylim(0, 1.15)
    ax.set_yticks([0.25, 0.50, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.0'],
                       fontsize=7.5, color='gray')
    ax.grid(color=PALETA['grid'], linestyle='--', linewidth=0.8)
    ax.spines['polar'].set_color(PALETA['grid'])
    ax.set_facecolor('white')

    medianas = centroides.loc[cluster_id]
    nota = (f"Frecuencia: {medianas['frequency_dias_activos']:.0f} días  |  "
            f"Recencia: {medianas['recency_dias']:.0f} días\n"
            f"Intensidad: {medianas['intensidad_digital']:.2f}  |  "
            f"Consistencia: {medianas['consistencia_uso']:.2f}")
    fig.text(0.5, 0.02, nota, ha='center', fontsize=9,
             color=PALETA['texto'], style='italic', linespacing=1.6)

    letra = letras[cluster_id]
    plt.savefig(f'figura_{letra}_radar_cluster{cluster_id}.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"✅ Figura {letra} — Cluster {cluster_id} guardada")

# CARACTERIZACIÓN DE CLUSTERS (OBJETIVO 4)

In [ ]:
print(df_rfm_limpio.shape)
print('cluster' in df_rfm_limpio.columns)

In [ ]:
# ── RECONSTRUIR df_plot ──────────────────────────────────────
# Paso 1: asegurarse de que df_rfm_limpio tiene clusters
# (si no, corre primero la celda del KMeans final)
print(df_rfm_limpio['cluster'].value_counts().sort_index())

# Paso 2: unir clusters al dataframe transaccional
df_plot = df_unificado_clustering.merge(
    df_rfm_limpio[['cluster']],
    left_on='codigo',
    right_index=True,
    how='inner'
)

print(f"df_plot creado: {df_plot.shape}")
print(df_plot.columns.tolist())
print(df_plot['cluster'].value_counts().sort_index())

In [ ]:
import pandas as pd
# Ver todos los dataframes que existen en memoria
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {obj.shape}")

In [ ]:
print(df_plot.groupby('cluster')['codigo'].nunique())

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA — Uso por plataforma digital según perfil
# ══════════════════════════════════════════════════════════════
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

PALETA = {
    0: '#F39C12',   # Físico Perdido
    1: '#1A3A5C',   # Digital Consolidado
    2: '#2E86AB',   # Digital Perdido
    3: '#C0392B',   # Físico Activo
}
NOMBRES_PERFIL = {
    0: 'P0 · Físico Perdido',
    1: 'P1 · Digital Consolidado',
    2: 'P2 · Digital Perdido',
    3: 'P3 · Físico Activo',
}

# Solo registros digitales
df_digital = df_plot[df_plot['tipo_recurso'] == 'digital'].copy()

# Top 10 plataformas globales
top_plataformas = (
    df_digital['nombre_recurso']
    .value_counts()
    .head(10)
    .index.tolist()
)
print("Top 10 plataformas:")
print(df_digital['nombre_recurso'].value_counts().head(10))

# Proporción dentro de cada perfil
df_top = df_digital[df_digital['nombre_recurso'].isin(top_plataformas)]

tabla = (
    df_top.groupby(['cluster', 'nombre_recurso'])
    .size()
    .reset_index(name='registros')
)

total_por_cluster = (
    df_digital.groupby('cluster')
    .size()
    .reset_index(name='total_cluster')
)

tabla = tabla.merge(total_por_cluster, on='cluster')
tabla['pct'] = (tabla['registros'] / tabla['total_cluster']) * 100

pivot = tabla.pivot(index='nombre_recurso', columns='cluster', values='pct').fillna(0)
pivot = pivot.reindex(top_plataformas)

# Figura
fig, ax = plt.subplots(figsize=(15, 7), facecolor='#F7F9FC')
ax.set_facecolor('#F7F9FC')

x = np.arange(len(top_plataformas))
ancho = 0.2
clusters = sorted(pivot.columns.tolist())

for i, cluster in enumerate(clusters):
    offset = (i - 1.5) * ancho
    valores = pivot[cluster].values if cluster in pivot.columns else np.zeros(len(top_plataformas))
    bars = ax.bar(
        x + offset, valores, width=ancho,
        color=PALETA[cluster], label=NOMBRES_PERFIL[cluster],
        alpha=0.88, edgecolor='white', linewidth=0.5
    )
    for bar, val in zip(bars, valores):
        if val > 1.5:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{val:.0f}%', ha='center', va='bottom',
                    fontsize=11, color='#1C2833')

ax.set_xticks(x)
ax.set_xticklabels(
    [p[:25] + '…' if len(p) > 25 else p for p in top_plataformas],
    rotation=32, ha='right', fontsize=13, color='#1C2833'
)
ax.set_ylabel('% del total de registros digitales del perfil', fontsize=11, color='#1C2833')
ax.set_title('Uso por plataforma digital según perfil — Biblioteca Jorge Roa Martínez',
             fontsize=20, fontweight='bold', color='#1C2833', pad=14)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))
ax.grid(axis='y', color='#DDE3EA', linewidth=0.6, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc='upper right', fontsize=13, framealpha=0.8, edgecolor='#DDE3EA')

plt.tight_layout()
plt.savefig('figura_plataformas_por_perfil.png', dpi=150, bbox_inches='tight', facecolor='#F7F9FC')
plt.show()
print("✅ Guardada")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA — Top libros físicos más prestados por perfil
# Celda autónoma
# ══════════════════════════════════════════════════════════════
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

PALETA = {
    0: '#F39C12',   # Físico Perdido
    1: '#1A3A5C',   # Digital Consolidado
    2: '#2E86AB',   # Digital Perdido
    3: '#C0392B',   # Físico Activo
}
NOMBRES_PERFIL = {
    0: 'P0 · Físico Perdido',
    1: 'P1 · Digital Consolidado',
    2: 'P2 · Digital Perdido',
    3: 'P3 · Físico Activo',
}

# Solo registros físicos
df_fisico = df_plot[df_plot['tipo_recurso'] == 'fisico'].copy()

# Top 8 títulos por perfil
fig, axes = plt.subplots(2, 2, figsize=(18, 12), facecolor='#F7F9FC')
axes = axes.flatten()

for idx, cluster_id in enumerate([0, 1, 2, 3]):
    ax = axes[idx]
    ax.set_facecolor('#F7F9FC')

    df_c = df_fisico[df_fisico['cluster'] == cluster_id]
    top = (
        df_c['nombre_recurso']
        .value_counts()
        .head(8)
        .reset_index()
    )
    top.columns = ['titulo', 'prestamos']
    top = top.sort_values('prestamos')

    top['titulo_corto'] = top['titulo'].apply(
        lambda x: x[:45] + '…' if len(str(x)) > 45 else x
    )

    bars = ax.barh(
        top['titulo_corto'], top['prestamos'],
        color=PALETA[cluster_id], alpha=0.85,
        edgecolor='white', linewidth=0.5
    )

    for bar, val in zip(bars, top['prestamos']):
        ax.text(bar.get_width() + top['prestamos'].max() * 0.01,
                bar.get_y() + bar.get_height()/2,
                f'{int(val):,}', va='center', fontsize=12, color='#1C2833')

    ax.set_title(NOMBRES_PERFIL[cluster_id],
                 fontsize=11, fontweight='bold',
                 color=PALETA[cluster_id], pad=10)
    ax.set_xlabel('Número de préstamos', fontsize=12, color='#1C2833')
    ax.tick_params(axis='y', labelsize=16)
    ax.grid(axis='x', color='#DDE3EA', linewidth=0.6, linestyle='--')
    ax.spines[['top', 'right']].set_visible(False)

    print(f"\n{NOMBRES_PERFIL[cluster_id]}")
    print(top[['titulo', 'prestamos']].sort_values(
        'prestamos', ascending=False).to_string(index=False))

fig.suptitle('Top 8 títulos físicos más prestados por perfil — Biblioteca Jorge Roa Martínez',
             fontsize=14, fontweight='bold', color='#1C2833', y=1.01)

plt.tight_layout()
plt.savefig('figura_libros_fisicos_por_perfil.png', dpi=150,
            bbox_inches='tight', facecolor='#F7F9FC')
plt.show()
print("\n✅ Guardada como figura_libros_fisicos_por_perfil.png")

In [ ]:
print(df_rfm_limpio['cluster'].value_counts().sort_index())
print(df_rfm_limpio.groupby('cluster')[['frequency_dias_activos','recency_dias','intensidad_digital','consistencia_uso']].median().round(2))

In [ ]:
df_plot[(df_plot['nombre_recurso'].str.contains('Game of thrones', case=False)) & 
        (df_plot['cluster'] == 2)]['facultad'].value_counts()

In [ ]:
vars_desc = ['tenure_dias', 'prop_fisica', 'frequency_fisica',
             'usa_fisico', 'total_registros']

print("=== VARIABLES DEL MODELO (mediana) ===")
print(df_rfm_limpio.groupby('cluster')[cols_modelo].median().round(2))

print("\n=== VARIABLES DESCRIPTIVAS (mediana) ===")
print(df_rfm_limpio.groupby('cluster')[vars_desc].median().round(2))

print("\n=== TAMAÑO ===")
tamano = df_rfm_limpio['cluster'].value_counts().sort_index().reset_index()
tamano.columns = ['cluster', 'n_usuarios']
tamano['pct'] = (tamano['n_usuarios'] / len(df_rfm_limpio) * 100).round(1)
print(tamano)

print("\n=== FACULTAD DOMINANTE ===")
for c in sorted(df_rfm_limpio['cluster'].unique()):
    top3 = (df_rfm_limpio[df_rfm_limpio['cluster'] == c]['facultad_moda']
            .value_counts(normalize=True).head(3) * 100).round(1)
    print(f"\nCluster {c}:")
    print(top3.to_string())

print("\n=== USO FÍSICO VS DIGITAL ===")
uso = (df_rfm_limpio.groupby('cluster')['usa_fisico']
       .value_counts(normalize=True)
       .unstack() * 100).round(1)
print(uso)

In [ ]:
print(df_rfm_limpio.groupby('cluster')[
    ['frequency_dias_activos','recency_dias',
     'intensidad_digital','consistencia_uso']].median().round(2))
print(df_rfm_limpio['cluster'].value_counts().sort_index())

In [ ]:
# Verificar que df_plot existe y tiene clusters
from seaborn import displot


print(df_plot.columns.tolist())
print(df_plot.shape)
print(df_plot['cluster'].value_counts().sort_index())

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np

K_FINAL = 4
cols_modelo = ['frequency_dias_activos', 'recency_dias', 
               'intensidad_digital', 'consistencia_uso']

# Escalar sobre df_rfm completo
X = df_rfm[cols_modelo].fillna(0).copy()
X['frequency_dias_activos'] = np.log1p(X['frequency_dias_activos'])

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Entrenar y guardar en df_rfm
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=25)
df_rfm['cluster'] = kmeans.fit_predict(X_scaled)

# Verificar
print(f"✅ Usuarios totales: {len(df_rfm):,}")
print(df_rfm['cluster'].value_counts().sort_index())

# Verificar silueta
from sklearn.metrics import silhouette_score
sil = silhouette_score(X_scaled, df_rfm['cluster'], sample_size=5000, random_state=42)
print(f"Silhouette: {sil:.4f}")

In [ ]:
PALETA = {
    0: '#C0392B',
    1: '#1A3A5C',
    2: '#2E86AB',
    3: '#F39C12'
}

# Claves de texto para gráficos
PALETA['digital'] = '#1A3A5C'
PALETA['fisico']  = '#C0392B'
PALETA['texto']   = '#2C3E50'
PALETA['grid']    = '#BDC3C7'

In [ ]:
import matplotlib.pyplot as plt

# 1. Paleta para las BARRAS (los canales dentro de cada facultad)
PALETA = {
    'digital': '#1A3A5C', # Azul oscuro (Digital)
    'fisico': '#C0392B',  # Rojo (Físico)
    'texto': '#333333',
    'grid': '#E0E0E0'
}

# 2. Tus colores exactos para los TÍTULOS de cada perfil
colores_perfil = {
    0: '#F39C12',   # Físico Perdido
    1: '#1A3A5C',   # Digital Consolidado
    2: '#2E86AB',   # Digital Perdido
    3: '#C0392B',   # Físico Activo
}

nombres_perfil = {
    0: 'P0 · Físico Perdido',
    1: 'P1 · Digital Consolidado',
    2: 'P2 · Digital Perdido',
    3: 'P3 · Físico Activo',
}

# (Asumo que nombres_fac ya está definido en una celda anterior, si no, define un diccionario vacío o bórralo del map)
# nombres_fac = {} 

# ══════════════════════════════════════════════════════════════
# FIGURA 13B — Uso por tipo de recurso dentro de cada perfil
# por facultad (top 6 facultades más grandes) — figuras individuales
# ══════════════════════════════════════════════════════════════

top6_fac = (df_fac.groupby('facultad')['codigo']
            .nunique().nlargest(6).index.tolist())

for cluster_id in [0, 1, 2, 3]:

    fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
    fig.patch.set_facecolor('white')

    # ── Filtro ────────────────────────────────────────────────
    df_c = df_fac[
        (df_fac['cluster'] == cluster_id) &
        (df_fac['facultad'].isin(top6_fac))
    ]

    # ── Cálculo de porcentajes por canal ──────────────────────
    canal = (df_c.groupby(['facultad', 'tipo_recurso'])
             .size().reset_index(name='registros'))
    canal['pct'] = (canal.groupby('facultad')['registros']
                    .transform(lambda x: x / x.sum() * 100))

    pivot = canal.pivot_table(
        index='facultad', columns='tipo_recurso',
        values='pct', fill_value=0
    ).reset_index()

    # Si nombres_fac existe, lo mapea. Si no, deja el nombre original.
    pivot['label'] = pivot['facultad'].map(nombres_fac).fillna(pivot['facultad'])

    # ── Fallback seguro si falta algún canal ──────────────────
    pivot['_digital'] = pivot['digital'].values if 'digital' in pivot.columns else 0
    pivot['_fisico']  = pivot['fisico'].values  if 'fisico'  in pivot.columns else 0
    pivot = pivot.sort_values('_digital', ascending=True)

    # ── Barras ────────────────────────────────────────────────
    y = range(len(pivot))

    ax.barh(list(y), pivot['_digital'],
            color=PALETA['digital'], height=0.6, label='Digital')
    ax.barh(list(y), pivot['_fisico'],
            left=pivot['_digital'],
            color=PALETA['fisico'], height=0.6, label='Físico')

    # ── Estética ──────────────────────────────────────────────
    ax.set_yticks(list(y))
    ax.set_yticklabels(pivot['label'], fontsize=10, color=PALETA['texto'])
    ax.set_xlabel('% de registros de la facultad', fontsize=10, color=PALETA['texto'])
    ax.set_title(nombres_perfil[cluster_id], fontsize=12,
                 fontweight='bold', color=colores_perfil[cluster_id], pad=10)
    ax.set_xlim(0, 100)
    ax.axvline(x=50, color=PALETA['grid'], linestyle='--', linewidth=0.8)
    ax.set_facecolor('white')
    ax.tick_params(colors=PALETA['texto'])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── Leyenda fuera del área, bajo el título ─────────────────
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels,
               loc='upper right',
               bbox_to_anchor=(0.98, 0.97),
               frameon=False, fontsize=9, ncol=1)

    plt.tight_layout()
    plt.savefig(f'figura_13b_perfil_{cluster_id}.png', dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    print(f"✅ Figura 13b — Perfil {cluster_id} guardada con éxito")

In [ ]:
# ============================================================
# FIGURA 14 — Dinámica de perfiles de usuario por semestre
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

PALETA = {
    'fondo': '#F7F9FC',
    'grid':  '#DDE3EA',
    'texto': '#1C2833',
}

colores_perfil = {
    0: '#F39C12',   # Físico Perdido
    1: '#1A3A5C',   # Digital Consolidado
    2: '#2E86AB',   # Digital Perdido
    3: '#C0392B',   # Físico Activo
}

nombres_perfil = {
    0: 'P0 · Físico Perdido',
    1: 'P1 · Digital Consolidado',
    2: 'P2 · Digital Perdido',
    3: 'P3 · Físico Activo',
}

# ------ PASO 1: Semestre ------
df_plot['semestre_num'] = df_plot['mes'].apply(lambda m: 1 if m <= 6 else 2)
df_plot['semestre_label'] = df_plot['anio'].astype(str) + '-S' + df_plot['semestre_num'].astype(str)
orden_semestres = sorted(df_plot['semestre_label'].unique())

# ------ PASO 2: Composición por semestre ------
usuarios_semestre = (
    df_plot.groupby(['semestre_label', 'cluster'])['codigo']
    .nunique()
    .reset_index()
    .rename(columns={'codigo': 'usuarios'})
)

pivot_comp = usuarios_semestre.pivot(
    index='semestre_label', columns='cluster', values='usuarios'
).fillna(0).reindex(orden_semestres)

pivot_pct = pivot_comp.div(pivot_comp.sum(axis=1), axis=0) * 100

# ------ PASO 3: Primera aparición ------
primera_vez = (
    df_plot.sort_values('semestre_label')
    .groupby('codigo')
    .agg(primer_semestre=('semestre_label', 'first'),
         cluster=('cluster', 'first'))
    .reset_index()
)

nuevos_por_semestre = (
    primera_vez.groupby(['primer_semestre', 'cluster'])
    .size()
    .reset_index(name='nuevos_usuarios')
)

pivot_nuevos = nuevos_por_semestre.pivot(
    index='primer_semestre', columns='cluster', values='nuevos_usuarios'
).fillna(0).reindex(orden_semestres)

# ------ PASO 4: Figura ------
fig = plt.figure(figsize=(16, 10), facecolor=PALETA['fondo'])
gs = GridSpec(2, 1, figure=fig, hspace=0.45)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# --- Panel A: área apilada ---
colores_orden = [colores_perfil[c] for c in sorted(colores_perfil.keys())]

ax1.stackplot(
    range(len(orden_semestres)),
    [pivot_pct[c].values for c in sorted(pivot_pct.columns)],
    labels=[nombres_perfil[c] for c in sorted(colores_perfil.keys())],
    colors=colores_orden,
    alpha=0.88
)
ax1.set_xticks(range(len(orden_semestres)))
ax1.set_xticklabels(orden_semestres, rotation=35, ha='right',
                    fontsize=9, color=PALETA['texto'])
ax1.set_ylabel('Proporción de usuarios activos (%)',
               fontsize=10, color=PALETA['texto'])
ax1.set_title('Panel A — Composición de perfiles por semestre académico',
              fontsize=11, fontweight='bold', color=PALETA['texto'], pad=10)
ax1.set_ylim(0, 100)
ax1.grid(axis='y', color=PALETA['grid'], linewidth=0.6, linestyle='--')
ax1.set_facecolor(PALETA['fondo'])
ax1.spines[['top', 'right']].set_visible(False)
ax1.legend(loc='upper left', fontsize=8.5, framealpha=0.7,
           ncol=2, bbox_to_anchor=(0.0, -0.18))

# Línea tendencia Digital Perdido
ax1_twin = ax1.twinx()
ax1_twin.plot(
    range(len(orden_semestres)),
    pivot_pct[2].values,
    color=colores_perfil[2], linewidth=2, linestyle='--',
    marker='o', markersize=4, alpha=0.7, label='% Digital Perdido (tendencia)'
)
ax1_twin.set_ylabel('% Digital Perdido', fontsize=9, color=colores_perfil[2])
ax1_twin.tick_params(axis='y', labelcolor=colores_perfil[2])
ax1_twin.set_ylim(0, 100)

# --- Panel B: barras nuevos usuarios ---
x = np.arange(len(orden_semestres))
ancho = 0.2

for i, cluster in enumerate(sorted(pivot_nuevos.columns)):
    offset = (i - 1.5) * ancho
    ax2.bar(
        x + offset,
        pivot_nuevos[cluster].values,
        width=ancho,
        color=colores_perfil[cluster],
        label=nombres_perfil[cluster],
        alpha=0.85,
        edgecolor='white',
        linewidth=0.5
    )

ax2.set_xticks(x)
ax2.set_xticklabels(orden_semestres, rotation=35, ha='right',
                    fontsize=9, color=PALETA['texto'])
ax2.set_ylabel('Usuarios que aparecen por primera vez',
               fontsize=10, color=PALETA['texto'])
ax2.set_title('Panel B — Incorporación de nuevos usuarios por perfil y semestre',
              fontsize=11, fontweight='bold', color=PALETA['texto'], pad=10)
ax2.grid(axis='y', color=PALETA['grid'], linewidth=0.6, linestyle='--')
ax2.set_facecolor(PALETA['fondo'])
ax2.spines[['top', 'right']].set_visible(False)
ax2.legend(loc='upper right', fontsize=8.5, framealpha=0.7, ncol=2)

fig.suptitle(
    'Figura 14. Dinámica temporal de perfiles de usuario — Biblioteca Jorge Roa Martínez (2022–2025)',
    fontsize=12, fontweight='bold', color=PALETA['texto'], y=1.01
)

plt.tight_layout()
plt.savefig('figura_14_dinamica_perfiles_temporal.png',
            dpi=150, bbox_inches='tight', facecolor=PALETA['fondo'])
plt.show()
print("✅ Figura 14 guardada")

In [ ]:
PALETA['digital'] = '#1A3A5C'
PALETA['fisico']  = '#C0392B'
PALETA['texto']   = '#2C3E50'
PALETA['grid']    = '#BDC3C7'
PALETA['acento']  = '#E74C3C'  # rojo para destacar períodos importantes

In [ ]:
import matplotlib.pyplot as plt

# 1. Aseguramos que la PALETA tenga la llave 'acento' y 'texto'
PALETA = {
    'texto': '#333333',
    'acento': '#C0392B', # Un tono rojo/naranja para resaltar los parciales finales
    # (Si usas otras llaves en el resto de tu código, puedes mantenerlas aquí)
}

# 2. Tus colores exactos para los TÍTULOS de cada perfil
colores_perfil = {
    0: '#F39C12',   # Físico Perdido
    1: '#1A3A5C',   # Digital Consolidado
    2: '#2E86AB',   # Digital Perdido
    3: '#C0392B',   # Físico Activo
}

nombres_perfil = {
    0: 'P0 · Físico Perdido',
    1: 'P1 · Digital Consolidado',
    2: 'P2 · Digital Perdido',
    3: 'P3 · Físico Activo',
}

# ══════════════════════════════════════════════════════════════
# FIGURA 18 — Comportamiento por semana del semestre
# ══════════════════════════════════════════════════════════════
df_semana = (df_plot.groupby(['semana_semestre', 'cluster'])
             .size().reset_index(name='registros'))

# Normalizar por perfil
df_semana['pct'] = (df_semana.groupby('cluster')['registros']
                    .transform(lambda x: x / x.sum() * 100))

fig, ax = plt.subplots(figsize=(12, 5), facecolor='white')
fig.patch.set_facecolor('white')

for ci in range(4):
    subset = df_semana[df_semana['cluster'] == ci].sort_values('semana_semestre')
    ax.plot(subset['semana_semestre'], subset['pct'],
            color=colores_perfil[ci], linewidth=2.5,
            label=nombres_perfil[ci].replace('\n', ' '))

# Marcar semanas de parciales
ax.axvspan(7, 9, alpha=0.07, color='gray', label='Parciales midterm')
ax.axvspan(14, 16, alpha=0.07, color=PALETA['acento'], label='Parciales finales')

ax.set_title('Concentración de actividad por semana del semestre (% normalizado)',
             fontsize=12, fontweight='bold', color=PALETA['texto'])
ax.set_xlabel('Semana del semestre', fontsize=10, color=PALETA['texto'])
ax.set_ylabel('% de actividad del perfil', fontsize=10, color=PALETA['texto'])
ax.legend(frameon=False, fontsize=9)
ax.set_facecolor('white')
ax.tick_params(colors=PALETA['texto'])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figura_18_semana_semestre.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura 18 guardada con éxito")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA — Nivel académico por perfil
# Celda autónoma
# ══════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PALETA = {'texto': '#333333'}

colores_perfil = {
    0: '#F39C12',   # Físico Perdido
    1: '#1A3A5C',   # Digital Consolidado
    2: '#2E86AB',   # Digital Perdido
    3: '#C0392B',   # Físico Activo
}

nombres_perfil = {
    0: 'P0 · Físico Perdido',
    1: 'P1 · Digital Consolidado',
    2: 'P2 · Digital Perdido',
    3: 'P3 · Físico Activo',
}

niveles_validos = ['PREGRADO', 'POSGRADO - ESPECIALIZACION', 'POSGRADO - DOCTORADO']
nombres_nivel = {
    'PREGRADO':                   'Pregrado',
    'POSGRADO - ESPECIALIZACION': 'Especialización',
    'POSGRADO - DOCTORADO':       'Doctorado',
}
colores_nivel = ['#1A3A5C', '#C0392B', '#F39C12']

# --- DATOS ---
df_nivel = df_plot[df_plot['nivel_academico_final'].isin(niveles_validos)]

nivel_cluster = (
    df_nivel.groupby(['cluster', 'nivel_academico_final'])
    .size()
    .reset_index(name='n')
)
nivel_cluster['pct'] = (
    nivel_cluster.groupby('cluster')['n']
    .transform(lambda x: x / x.sum() * 100)
)

# --- FIGURA ---
x     = np.arange(4)
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5), facecolor='white')
fig.patch.set_facecolor('white')

for ji, niv in enumerate(niveles_validos):
    vals = []
    for ci in range(4):
        v = nivel_cluster[
            (nivel_cluster['cluster'] == ci) &
            (nivel_cluster['nivel_academico_final'] == niv)
        ]['pct']
        vals.append(v.values[0] if len(v) > 0 else 0)

    bars = ax.bar(x + ji * width, vals, width,
                  color=colores_nivel[ji],
                  label=nombres_nivel[niv])

    for bar, val in zip(bars, vals):
        if val > 3:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5,
                    f'{val:.1f}%',
                    ha='center', va='bottom',
                    fontsize=11, color=colores_nivel[ji],
                    fontweight='bold')

ax.set_title('Distribución de nivel académico por perfil',
             fontsize=13, fontweight='bold', color=PALETA['texto'])
ax.set_xlabel('Perfil', fontsize=11, color=PALETA['texto'])
ax.set_ylabel('% dentro del perfil', fontsize=11, color=PALETA['texto'])
ax.set_xticks(x + width)
ax.set_xticklabels(
    [nombres_perfil[i] for i in range(4)],
    fontsize=10, color=PALETA['texto']
)
ax.legend(frameon=False, fontsize=10)
ax.set_facecolor('white')
ax.tick_params(colors=PALETA['texto'])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figura_nivel_academico.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura nivel académico guardada")

In [ ]:
# Resumen completo por cluster
total_usuarios = df_rfm_limpio['cluster'].value_counts().sort_index()
total_interacciones = df_plot.groupby('cluster').size().sort_index()
total_general = len(df_plot)
total_u = total_usuarios.sum()

print("=== RESUMEN POR CLUSTER ===")
for c in range(4):
    u = total_usuarios[c]
    i = total_interacciones[c]
    print(f"\nCluster {c}:")
    print(f"  Usuarios:      {u:,} ({u/total_u*100:.1f}%)")
    print(f"  Interacciones: {i:,} ({i/total_general*100:.1f}%)")
    print(f"  Promedio int/usuario: {i/u:.0f}")

print(f"\n=== TOTALES ===")
print(f"Usuarios únicos:     {total_u:,}")
print(f"Total interacciones: {total_general:,}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA — Usuarios e interacciones por cluster
# Celda autónoma
# ══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

PALETA = {
    0: '#F39C12' , 1: '#1A3A5C', 2: '#2E86AB', 3: '#C0392B',
}
NOMBRES = {
    0: 'Cluster 0\nFísico Perdido',
    1: 'Cluster 1\nDigital Consolidado',
    2: 'Cluster 2\nDigital Perdido',
    3: 'Cluster 3\nFísico Activo',
}

# --- DATOS ---
conteo_usuarios = df_rfm_limpio['cluster'].value_counts().sort_index()
conteo_interact = df_plot.groupby('cluster').size().sort_index()
total_u = conteo_usuarios.sum()
total_i = conteo_interact.sum()

print("Verificación antes de graficar:")
for c in range(4):
    print(f"  Cluster {c}: {conteo_usuarios[c]:,} usuarios | {conteo_interact[c]:,} interacciones")

# --- FIGURA CON DOS EJES Y ---
fig, ax1 = plt.subplots(figsize=(13, 7), facecolor='#F7F9FC')
ax1.set_facecolor('#F7F9FC')
ax2 = ax1.twinx()

x = np.arange(4)
ancho = 0.35

# Barras de usuarios (eje izquierdo)
bars1 = ax1.bar(x - ancho/2, conteo_usuarios.values,
                width=ancho, color=[PALETA[i] for i in range(4)],
                edgecolor='white', linewidth=1.5, alpha=0.95,
                label='Usuarios únicos')

# Barras de interacciones (eje derecho) — mismo color, más transparente
bars2 = ax2.bar(x + ancho/2, conteo_interact.values,
                width=ancho, color=[PALETA[i] for i in range(4)],
                edgecolor='white', linewidth=1.5, alpha=0.45,
                label='Interacciones', hatch='//')

# Etiquetas barras de usuarios
for bar, val in zip(bars1, conteo_usuarios.values):
    pct = val / total_u * 100
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + total_u * 0.004,
             f'{val:,}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=10,
             fontweight='bold', color='#1C2833')

# Etiquetas barras de interacciones
for bar, val in zip(bars2, conteo_interact.values):
    pct = val / total_i * 100
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + total_i * 0.004,
             f'{val:,.0f}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=10,
             fontweight='bold', color='#555555')

# Ejes
ax1.set_xticks(x)
ax1.set_xticklabels([NOMBRES[i] for i in range(4)], fontsize=11, color='#1C2833')
ax1.set_ylabel('Número de usuarios únicos', fontsize=11, color='#1C2833')
ax2.set_ylabel('Número de interacciones', fontsize=11, color='#555555')
ax1.set_ylim(0, conteo_usuarios.max() * 1.25)
ax2.set_ylim(0, conteo_interact.max() * 1.25)
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.tick_params(axis='y', labelcolor='#1C2833')
ax2.tick_params(axis='y', labelcolor='#555555')
ax1.grid(axis='y', color='#DDE3EA', linewidth=0.6, linestyle='--')
ax1.spines[['top']].set_visible(False)
ax2.spines[['top']].set_visible(False)

ax1.set_title('Distribución de usuarios e interacciones por cluster\nBiblioteca Jorge Roa Martínez',
              fontsize=13, fontweight='bold', color='#1C2833', pad=14)

# Leyenda combinada
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=10, framealpha=0.8,
           loc='upper right', edgecolor='#DDE3EA')

plt.tight_layout()
plt.savefig('figura_distribucion_clusters_v2.png', dpi=150,
            bbox_inches='tight', facecolor='#F7F9FC')
plt.show()
print(f"\n✅ Total usuarios: {total_u:,} | Total interacciones: {total_i:,}")

In [ ]:
# Verificar qué DataFrames tienen carrera
print("df_rfm columns:", df_rfm.columns.tolist())
print("df_fac columns:", df_fac.columns.tolist())
print("df_unificado_clustering columns:", df_unificado_clustering.columns.tolist())

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA — Top 12 carreras por perfil de usuario
# Celda autónoma
# ══════════════════════════════════════════════════════════════
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

PALETA = {'texto': '#333333', 'grid': '#E0E0E0'}
colores_perfil = {
    0: '#F39C12',   # Físico Perdido
    1: '#1A3A5C',   # Digital Consolidado
    2: '#2E86AB',   # Digital Perdido
    3: '#C0392B',   # Físico Activo
}

nombres_carrera = {
    'ADMINISTRACION AMBIENTAL':    'Admón. Ambiental',
    'ADMINISTRACION INDUSTRIAL':   'Admón. Industrial',
    'ADMINISTRACION TURISMO':      'Admón. Turismo',
    'INGENIERIA CIVIL':            'Ing. Civil',
    'INGENIERIA DE SISTEMAS':      'Ing. de Sistemas',
    'INGENIERIA ELECTRICA':        'Ing. Eléctrica',
    'INGENIERIA ELECTRONICA':      'Ing. Electrónica',
    'INGENIERIA FISICA':           'Ing. Física',
    'INGENIERIA INDUSTRIAL':       'Ing. Industrial',
    'INGENIERIA MECANICA':         'Ing. Mecánica',
    'INGENIERIA MECATRONICA':      'Ing. Mecatrónica',
    'LICENCIATURA EN ARTES':       'Lic. en Artes',
    'LICENCIATURA EN BILINGUISMO': 'Lic. Bilingüismo',
    'LICENCIATURA EN CIENCIAS':    'Lic. en Ciencias',
    'LICENCIATURA EN EDUCACION':   'Lic. en Educación',
    'LICENCIATURA EN LENGUAS':     'Lic. en Lenguas',
    'LICENCIATURA EN MUSICA':      'Lic. en Música',
    'MEDICINA':                    'Medicina',
    'MEDICINA VETERINARIA':        'Med. Veterinaria',
    'TECNOLOGIA ELECTRICA':        'Tec. Eléctrica',
    'TECNOLOGIA EN SALUD':         'Tec. en Salud',
    'TECNOLOGIA EN SOFTWARE':      'Tec. en Software',
    'TECNOLOGIA EN TURISMO':       'Tec. en Turismo',
    'TECNOLOGIA INDUSTRIAL':       'Tec. Industrial',
    'TECNOLOGIA MECANICA':         'Tec. Mecánica',
    'TECNOLOGIA QUIMICA':          'Tec. Química',
    'DOCTORADO EN EDUCACION':      'Doc. en Educación',
    'DOCTORADO EN INGENIERIA':     'Doc. en Ingeniería',
    'ESPECIALIZACION EN SALUD':    'Esp. en Salud',
    'ESPECIALIZACION - OTRAS':     'Esp. Otras',
    'OTRA INGENIERIA':             'Otra Ingeniería',
    'OTRA LICENCIATURA':           'Otra Licenciatura',
    'OTRA TECNOLOGIA':             'Otra Tecnología',
}

# --- DATOS ---
carrera_por_usuario = (
    df_unificado_clustering[['codigo', 'carrera_final']]
    .drop_duplicates(subset='codigo')
)

df_rfm_limpio_reset = df_rfm_limpio.reset_index()

df_cluster_carrera = df_rfm_limpio_reset[['codigo', 'cluster']].merge(
    carrera_por_usuario, on='codigo', how='left'
)

excluir = ['DESCONOCIDO', 'OTRO / POR REVISAR', 'PERSONAL ADM / OTROS',
           'DOCTORADO - OTROS', 'OTRO / NO APLICA']
df_cluster_carrera = df_cluster_carrera[
    df_cluster_carrera['carrera_final'].notna() &
    ~df_cluster_carrera['carrera_final'].isin(excluir)
]

# --- CALCULAR PORCENTAJES ---
carrera_cluster = (
    df_cluster_carrera
    .groupby(['cluster', 'carrera_final'])
    .size()
    .reset_index(name='usuarios')
)

total_cluster = df_cluster_carrera.groupby('cluster').size().reset_index(name='total')
carrera_cluster = carrera_cluster.merge(total_cluster, on='cluster')
carrera_cluster['pct'] = (carrera_cluster['usuarios'] / carrera_cluster['total'] * 100).round(1)

# Top 12 carreras por volumen global
top12 = (
    df_cluster_carrera['carrera_final']
    .value_counts()
    .head(12)
    .index.tolist()
)
print("Top 12 carreras:")
print(df_cluster_carrera['carrera_final'].value_counts().head(12))

carrera_cluster_top = carrera_cluster[carrera_cluster['carrera_final'].isin(top12)]

# --- PIVOT ---
pivot_carrera = carrera_cluster_top.pivot(
    index='carrera_final', columns='cluster', values='pct'
).fillna(0)

pivot_carrera.index = pivot_carrera.index.map(
    lambda x: nombres_carrera.get(x, x)
)
pivot_carrera.columns = [
    'P0 · Físico Perdido',
    'P1 · Digital Consolidado',
    'P2 · Digital Perdido',
    'P3 · Físico Activo',
]

pivot_carrera = pivot_carrera[pivot_carrera.sum(axis=1) > 0]
pivot_carrera = pivot_carrera.loc[
    pivot_carrera.mean(axis=1).sort_values(ascending=False).index
]

# --- FIGURA ---
cmap_tesis = LinearSegmentedColormap.from_list(
    'TesisBlue', ['#F8F9FA', '#2E86AB', '#1A3A5C']
)

fig, ax = plt.subplots(figsize=(11, 9), facecolor='white')
fig.patch.set_facecolor('white')

sns.heatmap(
    pivot_carrera,
    annot=True,
    fmt=".1f",
    cmap=cmap_tesis,
    linewidths=1,
    linecolor='white',
    cbar_kws={'label': '% de usuarios dentro del perfil', 'shrink': 0.8},
    ax=ax
)

ax.set_title('Concentración de las Top 12 Carreras por Perfil de Usuario',
             fontsize=14, fontweight='bold', color=PALETA['texto'], pad=20)
ax.set_ylabel('Programa Académico', fontsize=11, color=PALETA['texto'], labelpad=10)
ax.set_xlabel('Perfil de Usuario', fontsize=11, color=PALETA['texto'], labelpad=10)
ax.tick_params(axis='y', colors=PALETA['texto'], labelsize=10)
ax.tick_params(axis='x', labelsize=10, rotation=15)

for i, tick_label in enumerate(ax.get_xticklabels()):
    tick_label.set_color(colores_perfil[i])
    tick_label.set_fontweight('bold')

plt.tight_layout()
plt.savefig('figura_carreras_heatmap.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Guardada como figura_carreras_heatmap.png")

In [ ]:
print(df_rfm_limpio['cluster'].value_counts().sort_index())
print(df_rfm_limpio.groupby('cluster')[['recency_dias','intensidad_digital']].median())

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIGURA — Concentración de Facultades por Perfil de Usuario
# Celda autónoma
# ══════════════════════════════════════════════════════════════
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

PALETA = {'texto': '#333333', 'grid': '#E0E0E0'}
colores_perfil = {
    0: '#F39C12',   # Físico Perdido
    1: '#1A3A5C',   # Digital Consolidado
    2: '#2E86AB',   # Digital Perdido
    3: '#C0392B',   # Físico Activo
}

nombres_fac_bonitos = {
    'apoyo_administrativo':   'Apoyo Administrativo',
    'bellas_artes':           'Bellas Artes',
    'biblioteca':             'Biblioteca',
    'ciencias_agrarias':      'Cs. Agrarias',
    'ciencias_ambientales':   'Cs. Ambientales',
    'ciencias_basicas':       'Cs. Básicas',
    'ciencias_educacion':     'Cs. de la Educación',
    'ciencias_empresariales': 'Cs. Empresariales',
    'externo':                'Externo',
    'fac_desconocida':        'Fac. Desconocida',
    'ilex':                   'ILEX',
    'ingenierias':            'Ingenierías',
    'mecanica':               'Mecánica',
    'rectoria':               'Rectoría',
    'salud':                  'Salud',
    'tecnologia':             'Tecnología',
    'univirtual':             'Univirtual',
}

# --- DATOS ---
facultad_por_usuario = (
    df_unificado_clustering[['codigo', 'facultad']]
    .drop_duplicates(subset='codigo')
)

df_rfm_limpio_reset = df_rfm_limpio.reset_index()

df_cluster_facultad = df_rfm_limpio_reset[['codigo', 'cluster']].merge(
    facultad_por_usuario, on='codigo', how='left'
)

df_cluster_facultad = df_cluster_facultad[
    df_cluster_facultad['facultad'].notna() &
    (df_cluster_facultad['facultad'] != 'DESCONOCIDO') &
    (df_cluster_facultad['facultad'] != 'fac_desconocida')
]

# --- CALCULAR PORCENTAJES ---
facultad_cluster = (
    df_cluster_facultad
    .groupby(['cluster', 'facultad'])
    .size()
    .reset_index(name='usuarios')
)

total_cluster = df_cluster_facultad.groupby('cluster').size().reset_index(name='total')
facultad_cluster = facultad_cluster.merge(total_cluster, on='cluster')
facultad_cluster['pct'] = (facultad_cluster['usuarios'] / facultad_cluster['total'] * 100).round(1)

# --- PIVOT ---
pivot_facultad = facultad_cluster.pivot(
    index='facultad', columns='cluster', values='pct'
).fillna(0)

pivot_facultad.index = pivot_facultad.index.map(
    lambda x: nombres_fac_bonitos.get(x, x)
)
pivot_facultad.columns = [
    'P0 · Físico Perdido',
    'P1 · Digital Consolidado',
    'P2 · Digital Perdido',
    'P3 · Físico Activo',
]

pivot_facultad = pivot_facultad[pivot_facultad.sum(axis=1) > 0]
pivot_facultad = pivot_facultad.loc[
    pivot_facultad.mean(axis=1).sort_values(ascending=False).index
]

# --- FIGURA ---
cmap_tesis = LinearSegmentedColormap.from_list(
    'TesisBlue', ['#F8F9FA', '#2E86AB', '#1A3A5C']
)

fig, ax = plt.subplots(figsize=(11, 8), facecolor='white')
fig.patch.set_facecolor('white')

sns.heatmap(
    pivot_facultad,
    annot=True,
    fmt=".1f",
    cmap=cmap_tesis,
    linewidths=1,
    linecolor='white',
    cbar_kws={'label': '% de usuarios dentro del perfil', 'shrink': 0.8},
    ax=ax
)

ax.set_title('Concentración de Facultades por Perfil de Usuario',
             fontsize=14, fontweight='bold', color=PALETA['texto'], pad=20)
ax.set_ylabel('Facultad', fontsize=11, color=PALETA['texto'], labelpad=10)
ax.set_xlabel('Perfil de Usuario', fontsize=11, color=PALETA['texto'], labelpad=10)
ax.tick_params(axis='y', colors=PALETA['texto'], labelsize=10)
ax.tick_params(axis='x', labelsize=10, rotation=15)

for i, tick_label in enumerate(ax.get_xticklabels()):
    tick_label.set_color(colores_perfil[i])
    tick_label.set_fontweight('bold')

plt.tight_layout()
plt.savefig('figura_facultades_heatmap.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Guardada como figura_facultades_heatmap.png")

In [ ]:
# Verificar medianas reales de cada cluster actual
cols_modelo = ['frequency_dias_activos', 'recency_dias',
               'intensidad_digital', 'consistencia_uso']
print(df_rfm_limpio.groupby('cluster')[cols_modelo].median().round(2))
print("\nConteo:")
print(df_rfm_limpio['cluster'].value_counts().sort_index())

In [ ]:
print(df_rfm_limpio['cluster'].value_counts().sort_index())
print(df_rfm_limpio.groupby('cluster')[['recency_dias','intensidad_digital']].median())

In [ ]:
#mapeo = {0: 3, 1: 1, 2: 2, 3: 0}
#df_rfm_limpio['cluster'] = df_rfm_limpio['cluster'].map(mapeo)

# Verificar
#print(df_rfm_limpio['cluster'].value_counts().sort_index())
#print(df_rfm_limpio.groupby('cluster')[['recency_dias','intensidad_digital']].median())

In [ ]:
cols_modelo = ['frequency_dias_activos', 'recency_dias',
               'intensidad_digital', 'consistencia_uso']

print("=== MEDIANAS POR CLUSTER ===")
print(df_rfm_limpio.groupby('cluster')[cols_modelo].median().round(2))

print("\n=== CONTEO ===")
print(df_rfm_limpio['cluster'].value_counts().sort_index())

print("\n=== % USO FÍSICO ===")
print(df_plot.groupby('cluster').apply(
    lambda x: (x['tipo_recurso'] == 'fisico').mean() * 100
).round(1))

In [ ]:
cols_modelo = ['frequency_dias_activos', 'recency_dias',
               'intensidad_digital', 'consistencia_uso']

print(df_rfm_limpio.groupby('cluster')[cols_modelo].median().round(2))
print("\n", df_rfm_limpio['cluster'].value_counts().sort_index())

# Datos para streamlit

In [ ]:
# Exportar los datos necesarios para el tablero
df_rfm_limpio.to_csv('df_rfm_limpio.csv', index=True)
df_fac.to_csv('df_fac.csv', index=False)

print("Archivos exportados:")
print(f"df_rfm_limpio: {df_rfm_limpio.shape}")
print(f"df_fac: {df_fac.shape}")
print(f"\nClusters disponibles: {sorted(df_rfm_limpio['cluster'].unique())}")
print(f"\nColumnas df_fac: {df_fac.columns.tolist()}")

In [ ]:
print(df_fac.columns.tolist())
print(df_fac.shape)
print(df_fac.head(2))

In [ ]:
import os
print(f"df_rfm_limpio.csv: {os.path.getsize('df_rfm_limpio.csv') / 1024 / 1024:.1f} MB")
print(f"df_fac.csv: {os.path.getsize('df_fac.csv') / 1024 / 1024:.1f} MB")